In [ ]:
import socket, psutil

hostname = socket.gethostname()
mem = psutil.virtual_memory().total / (1024**3)

print(f"当前运行节点: {hostname}")
print(f"当前节点内存: {mem:.2f} GB")

# 0. Import Package

In [ ]:
import math
import qiskit
import itertools
import numpy as np
from scipy.spatial import Voronoi
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from numpy.linalg import eigvalsh
from collections import defaultdict
from itertools import product

#from qiskit import QuantumCircuit, execute
# from qiskit_aer import Aer
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector
from qiskit.circuit.library import QFT
from qiskit.circuit.library import PauliEvolutionGate
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
# from qiskit_algorithms import VQE
# from qiskit_algorithms.optimizers import SLSQP
from qiskit.circuit.library import TwoLocal
# from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit.primitives import StatevectorEstimator as Estimator
#from qiskit.opflow import PauliTrotterEvolution

In [ ]:
print(qiskit.__version__)

# 1. 几何结构
## 1.1 becke_radial_grid

In [ ]:
# try becke_radial_grid_clamped
def becke_radial_grid(num_r, alpha, nu):
    # 计算 u 空间的步长
    du = 1.0 / (num_r + 1)
    
    # 这里的技巧是偏移 0.5 * du
    # 使得第一个采样点在 0.5*du，而原点 u=0 恰好被“夹”住
    u = np.linspace(0.5 * du, 1.0 - 0.5 * du, num_r)

    # 防止 1 - u**nu 出现 <= 0 导致 log 产生 NaN 或 -inf
    inner = np.clip(1.0 - u**nu, 1e-16, 1.0)
    
    # 映射回 r 空间
    r = -alpha * np.log(inner)
    return r

## Lebedev
### Lebedev_6

In [ ]:
def lebedev_6():
    # 6-point Lebedev grid
    # 精确到 l = 1
    w = 4*np.pi/6
    pts = np.array([
        [ 1, 0, 0],
        [-1, 0, 0],
        [ 0, 1, 0],
        [ 0,-1, 0],
        [ 0, 0, 1],
        [ 0, 0,-1],
    ], dtype=float)
    weights = np.full(6, w)
    return pts, weights


### Lebedev_14

In [ ]:
def lebedev_14():
    a = np.sqrt(1/3)
    pts = np.array([
        [ 1, 0, 0], [-1, 0, 0],
        [ 0, 1, 0], [ 0,-1, 0],
        [ 0, 0, 1], [ 0, 0,-1],
        [ a, a, a], [ a, a,-a]
        [ a,-a, a], [ a,-a,-a],
        [-a, a, a], [-a, a,-a],
        [-a,-a, a], [-a,-a,-a],
    ], dtype=float)

    w1 = 4*np.pi/21
    w2 = 4*np.pi/84
    weights = np.array([w1]*6 + [w2]*8)
    return pts, weights


### Lebedev_26

In [ ]:
def lebedev_26():
    a = np.sqrt(1/3)
    b = np.sqrt(3/5)
    pts = np.array([
        [ 1, 0, 0], [-1, 0, 0],
        [ 0, 1, 0], [ 0,-1, 0],
        [ 0, 0, 1], [ 0, 0,-1],
        [ a, a, a], [ a, a,-a], [ a,-a, a], [ a,-a,-a],
        [-a, a, a], [-a, a,-a], [-a,-a, a], [-a,-a,-a],
        [ b, 0, 0], [-b, 0, 0],
        [ 0, b, 0], [ 0,-b, 0],
        [ 0, 0, b], [ 0, 0,-b],
        [ 0, a, b], [ 0, a,-b], [ 0,-a, b], [ 0,-a,-b],
        [ a, 0, b], [ a, 0,-b]
    ], dtype=float)

    w1 = 4*np.pi/30
    w2 = 4*np.pi/120
    w3 = 4*np.pi/60
    weights = np.array([w1]*6 + [w2]*8 + [w3]*12)
    return pts, weights


### Lebedev_50

In [ ]:
def lebedev_50():
    """
    Full Lebedev 50-point spherical grid.
    Returns:
        pts: (50,3) array of unit vectors
        w:   (50,) array of weights
    """

    pts = np.array([
        [ 0.0,  0.0,  1.0],
        [ 0.0,  0.0, -1.0],
        [ 1.0,  0.0,  0.0],
        [-1.0,  0.0,  0.0],
        [ 0.0,  1.0,  0.0],
        [ 0.0, -1.0,  0.0],

        [ 0.5773502691896257,  0.5773502691896257,  0.5773502691896257],
        [ 0.5773502691896257,  0.5773502691896257, -0.5773502691896257],
        [ 0.5773502691896257, -0.5773502691896257,  0.5773502691896257],
        [ 0.5773502691896257, -0.5773502691896257, -0.5773502691896257],
        [-0.5773502691896257,  0.5773502691896257,  0.5773502691896257],
        [-0.5773502691896257,  0.5773502691896257, -0.5773502691896257],
        [-0.5773502691896257, -0.5773502691896257,  0.5773502691896257],
        [-0.5773502691896257, -0.5773502691896257, -0.5773502691896257],

        [ 0.0,  0.5257311121191336,  0.85065080835204],
        [ 0.0,  0.5257311121191336, -0.85065080835204],
        [ 0.0, -0.5257311121191336,  0.85065080835204],
        [ 0.0, -0.5257311121191336, -0.85065080835204],

        [ 0.5257311121191336,  0.85065080835204,  0.0],
        [ 0.5257311121191336, -0.85065080835204,  0.0],
        [-0.5257311121191336,  0.85065080835204,  0.0],
        [-0.5257311121191336, -0.85065080835204,  0.0],

        [ 0.85065080835204,  0.0,  0.5257311121191336],
        [ 0.85065080835204,  0.0, -0.5257311121191336],
        [-0.85065080835204,  0.0,  0.5257311121191336],
        [-0.85065080835204,  0.0, -0.5257311121191336],

        [ 0.30901699437494745,  0.5,  0.8090169943749475],
        [ 0.30901699437494745,  0.5, -0.8090169943749475],
        [ 0.30901699437494745, -0.5,  0.8090169943749475],
        [ 0.30901699437494745, -0.5, -0.8090169943749475],
        [-0.30901699437494745,  0.5,  0.8090169943749475],
        [-0.30901699437494745,  0.5, -0.8090169943749475],
        [-0.30901699437494745, -0.5,  0.8090169943749475],
        [-0.30901699437494745, -0.5, -0.8090169943749475],

        [ 0.5,  0.8090169943749475,  0.30901699437494745],
        [ 0.5,  0.8090169943749475, -0.30901699437494745],
        [ 0.5, -0.8090169943749475,  0.30901699437494745],
        [ 0.5, -0.8090169943749475, -0.30901699437494745],
        [-0.5,  0.8090169943749475,  0.30901699437494745],
        [-0.5,  0.8090169943749475, -0.30901699437494745],
        [-0.5, -0.8090169943749475,  0.30901699437494745],
        [-0.5, -0.8090169943749475, -0.30901699437494745],

        [ 0.8090169943749475,  0.30901699437494745,  0.5],
        [ 0.8090169943749475,  0.30901699437494745, -0.5],
        [ 0.8090169943749475, -0.30901699437494745,  0.5],
        [ 0.8090169943749475, -0.30901699437494745, -0.5],
        [-0.8090169943749475,  0.30901699437494745,  0.5],
        [-0.8090169943749475,  0.30901699437494745, -0.5],
        [-0.8090169943749475, -0.30901699437494745,  0.5],
        [-0.8090169943749475, -0.30901699437494745, -0.5],
    ])

    # 权重（全部 50 个）
    w = np.array([
        0.126984126984127, 0.126984126984127,
        0.126984126984127, 0.126984126984127,
        0.126984126984127, 0.126984126984127,

        0.047619047619048, 0.047619047619048,
        0.047619047619048, 0.047619047619048,
        0.047619047619048, 0.047619047619048,
        0.047619047619048, 0.047619047619048,

        0.075, 0.075, 0.075, 0.075,
        0.075, 0.075, 0.075, 0.075,
        0.075, 0.075, 0.075, 0.075,
        0.075, 0.075, 0.075, 0.075,
        0.075, 0.075, 0.075, 0.075,
        0.075, 0.075, 0.075, 0.075,
        0.075, 0.075, 0.075, 0.075,
        0.075, 0.075, 0.075, 0.075,
    ])

    return pts, w


### Lebedev_74

In [ ]:
def lebedev_74():
    """
    Full Lebedev 74-point spherical grid.
    Returns:
        pts: (74,3) array of unit vectors
        w:   (74,) array of weights
    """

    a = 0.45970084338098305
    b = 0.6285393610547089
    c = 0.322185354626569
    d = 0.8360955967490217

    pts = np.array([
        [ 0.0,  0.0,  1.0],
        [ 0.0,  0.0, -1.0],
        [ 1.0,  0.0,  0.0],
        [-1.0,  0.0,  0.0],
        [ 0.0,  1.0,  0.0],
        [ 0.0, -1.0,  0.0],

        [ a,  a,  a], [ a,  a,-a], [ a,-a,  a], [ a,-a,-a],
        [-a,  a,  a], [-a,  a,-a], [-a,-a,  a], [-a,-a,-a],

        [ b,  0,  c], [ b,  0,-c], [-b,  0,  c], [-b,  0,-c],
        [ 0,  b,  c], [ 0,  b,-c], [ 0,-b,  c], [ 0,-b,-c],
        [ c,  b,  0], [ c,-b,  0], [-c,  b,  0], [-c,-b,  0],

        [ d,  d,  0], [ d,-d,  0], [-d,  d,  0], [-d,-d,  0],
        [ d,  0,  d], [ d,  0,-d], [-d,  0,  d], [-d,  0,-d],
        [ 0,  d,  d], [ 0,  d,-d], [ 0,-d,  d], [ 0,-d,-d],

        [ 0.0,  a,  b], [ 0.0,  a,-b], [ 0.0,-a,  b], [ 0.0,-a,-b],
        [ a,  0.0,  b], [ a,  0.0,-b], [-a,  0.0,  b], [-a,  0.0,-b],
        [ b,  a,  0.0], [ b,-a,  0.0], [-b,  a,  0.0], [-b,-a,  0.0],

        [ c,  c,  d], [ c,  c,-d], [ c,-c,  d], [ c,-c,-d],
        [-c,  c,  d], [-c,  c,-d], [-c,-c,  d], [-c,-c,-d],

        [ d,  c,  c], [ d,  c,-c], [ d,-c,  c], [ d,-c,-c],
        [-d,  c,  c], [-d,  c,-c], [-d,-c,  c], [-d,-c,-c],

        [ c,  d,  c], [ c,  d,-c], [ c,-d,  c], [ c,-d,-c],
        [-c,  d,  c], [-c,  d,-c], [-c,-d,  c], [-c,-d,-c],
    ])

    w1 = 0.047619047619047616
    w2 = 0.0380952380952381
    w3 = 0.03214285714285714
    w4 = 0.02857142857142857

    w = np.array(
        [w1]*6 +
        [w2]*8 +
        [w3]*12 +
        [w4]*48
    )

    return pts, w


### Lebedev_86

In [ ]:
def lebedev_86():
    """
    Full Lebedev 86-point spherical grid.
    Returns:
        pts: (86,3) array of unit vectors
        w:   (86,) array of weights
    """

    a = 0.2666354015167047
    b = 0.681507726536546
    c = 0.4174961227965453
    d = 0.872473431981953

    pts = np.array([
        [ 0.0,  0.0,  1.0],
        [ 0.0,  0.0, -1.0],
        [ 1.0,  0.0,  0.0],
        [-1.0,  0.0,  0.0],
        [ 0.0,  1.0,  0.0],
        [ 0.0, -1.0,  0.0],

        [ a,  a,  a], [ a,  a,-a], [ a,-a,  a], [ a,-a,-a],
        [-a,  a,  a], [-a,  a,-a], [-a,-a,  a], [-a,-a,-a],

        [ b,  0,  c], [ b,  0,-c], [-b,  0,  c], [-b,  0,-c],
        [ 0,  b,  c], [ 0,  b,-c], [ 0,-b,  c], [ 0,-b,-c],
        [ c,  b,  0], [ c,-b,  0], [-c,  b,  0], [-c,-b,  0],

        [ d,  d,  0], [ d,-d,  0], [-d,  d,  0], [-d,-d,  0],
        [ d,  0,  d], [ d,  0,-d], [-d,  0,  d], [-d,  0,-d],
        [ 0,  d,  d], [ 0,  d,-d], [ 0,-d,  d], [ 0,-d,-d],

        [ 0.0,  a,  b], [ 0.0,  a,-b], [ 0.0,-a,  b], [ 0.0,-a,-b],
        [ a,  0.0,  b], [ a,  0.0,-b], [-a,  0.0,  b], [-a,  0.0,-b],
        [ b,  a,  0.0], [ b,-a,  0.0], [-b,  a,  0.0], [-b,-a,  0.0],

        [ c,  c,  d], [ c,  c,-d], [ c,-c,  d], [ c,-c,-d],
        [-c,  c,  d], [-c,  c,-d], [-c,-c,  d], [-c,-c,-d],

        [ d,  c,  c], [ d,  c,-c], [ d,-c,  c], [ d,-c,-c],
        [-d,  c,  c], [-d,  c,-c], [-d,-c,  c], [-d,-c,-c],

        [ c,  d,  c], [ c,  d,-c], [ c,-d,  c], [ c,-d,-c],
        [-c,  d,  c], [-c,  d,-c], [-c,-d,  c], [-c,-d,-c],
    ])

    w1 = 0.0380952380952381
    w2 = 0.03214285714285714
    w3 = 0.02857142857142857
    w4 = 0.025396825396825397

    w = np.array(
        [w1]*6 +
        [w2]*8 +
        [w3]*12 +
        [w4]*60
    )

    return pts, w


### Lebedev grid select

In [ ]:
def lebedev_grid(order):
    """
    Return Lebedev directions (unit vectors) and weights for a given order.
    order: one of {6, 14, 26, 38, 50, 74, 86, 110, 146, 170, 194, 230, 266, 302, 350}
    """
    if order == 6:
        return lebedev_6()
    elif order == 14:
        return lebedev_14()
    elif order == 26:
        return lebedev_26()
    elif order == 50:
        return lebedev_50()
    elif order == 74:
        return lebedev_74()
    elif order == 86:
        return lebedev_86()
    else:
        raise ValueError(f"Lebedev order {order} not implemented.")


## 3D adaptive grid for hydrogen using Lebedev

In [ ]:
# 3D adaptive grid for hydrogen using Lebedev 2025 12 21

def hydrogen_adaptive_grid_lebedev(num_r, order, alpha, nu):
    # For the time being, we test the radial part via exponential stretch like r^1.5
    #r = np.linspace(0.1, r_max, num_r)**1.5
    # alpha=1.0
    # nu=2.0
    r = becke_radial_grid(num_r, alpha, nu)

    # r = r[r > 1e-3]
    
    leb_pts, leb_w = lebedev_grid(order)
    points = []
    for ri in r:
        for n in leb_pts:
            points.append( ri * n)

    points = np.array(points)
    #vor = Voronoi(points, qhull_options='Qbb Qc Qx QJ')
    vor = Voronoi(points, qhull_options='QJ')
    
    neighbors = {i: set() for i in range(len(points))}            
    for p, q in vor.ridge_points:
        neighbors[p].add(q)
        neighbors[q].add(p)
    for m in range(len(points)):
        for n in neighbors[m]:
            if m not in neighbors[n]:
                print("Asym neighbor:", m, n)
    return np.array(points), neighbors, vor

## generate 3D adaptive grid for hydrogen 

In [ ]:
# 6, 14, 26, 50, 74, 86
num_r = 18
order = 50
alpha = 2
nu = 1.2
points, neighbors, vor = hydrogen_adaptive_grid_lebedev(num_r, order, alpha, nu)
length = len(points)
num_bits = math.ceil(math.log2(length))
num_qubits = num_bits
print(f"Number of points: {length}")
print(f"Number of bits: {num_bits}")
D_phys = length      # 252
M = num_bits
D = 2**M                  # 256

if D_phys < D:
    pad = D - D_phys
    # 用最后一个点复制填充（比全零更物理）
    pad_points = np.repeat(points[-1][None, :], pad, axis=0)
    points_ext = np.vstack([points, pad_points])
else:
    points_ext = points


## Calculate the geometrical properties of facet

In [ ]:
# calculate the geometrical properties of facet
def facet_geometry(vor, ridge_vertices):
    # 退化 facet：少于 3 个顶点
    if len(ridge_vertices) < 3:
        return None, None, 0.0

    verts = vor.vertices[ridge_vertices]

    # 计算法向量
    v1 = verts[1] - verts[0]
    v2 = verts[2] - verts[0]
    normal = np.cross(v1, v2)
    norm = np.linalg.norm(normal)
    if norm < 1e-12:
        return None, None, 0.0
    normal /= norm

    # 计算 centroid
    centroid = verts.mean(axis=0)

    # 计算面积（投影 + shoelace）
    ax = np.argmin(np.abs(normal))
    proj = np.delete(verts, ax, axis=1)
    c2 = proj.mean(axis=0)
    angles = np.arctan2(proj[:,1] - c2[1], proj[:,0] - c2[0])
    order = np.argsort(angles)
    proj = proj[order]
    x, y = proj[:,0], proj[:,1]
    area = 0.5 * np.abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

    if area < 1e-14:
        return None, None, 0.0

    return normal, centroid, area


## Calculate cell volume

In [ ]:
def cell_volume(points, neighbors, sigma):
    N = len(points)
    volumes = np.zeros(N)

    # 计算 cell centroid
    cell_centroid = np.zeros((N,3))
    for m in range(N):
        centroids = [sigma[m][n]["centroid"] for n in neighbors[m] if n in sigma[m]]
        if centroids:
            cell_centroid[m] = np.mean(centroids, axis=0)
        else:
            cell_centroid[m] = points[m]

    # 计算体积
    for m in range(N):
        for n in neighbors[m]:
            if n not in sigma[m]:
                continue

            area = sigma[m][n]["area"]
            normal = sigma[m][n]["normal"]
            centroid_mn = sigma[m][n]["centroid"]

            h = abs(np.dot(cell_centroid[m] - centroid_mn, normal))
            if h < 1e-12:
                continue

            volumes[m] += area * h / 3.0

    # 避免 volume=0
    volumes[volumes < 1e-14] = 1e-14
    return volumes


## Plot the 3d hydrogen points

In [ ]:
# Plot the 3d hydrogen points
import matplotlib.pyplot as plt


#points = hydrogen_adaptive_grid()

r = np.linalg.norm(points, axis = 1)

fig = plt.figure(figsize= (6, 6))
ax = fig.add_subplot(111, projection='3d')

p = ax.scatter(points[:,0], points[:,2], points[:, 2], c=r, cmap='viridis', s=12)
fig.colorbar(p, ax=ax, label='radius')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_title('Hydrogen Adaptive Grid (colored by radius)')

plt.show()

## 几何预处理接口

In [ ]:
from collections import defaultdict

def build_geometry(points, neighbors, vor):
    sigma = defaultdict(dict)

    for (p, q), rv in zip(vor.ridge_points, vor.ridge_vertices):
        if -1 in rv:
            continue

        normal, centroid, area = facet_geometry(vor, rv)
        if normal is None:
            continue

        sigma[p][q] = {
            "area": area,
            "normal": normal,
            "centroid": centroid,
            "ridge_vertices": rv
        }
        sigma[q][p] = {
            "area": area,
            "normal": -normal,
            "centroid": centroid,
            "ridge_vertices": rv
        }

    volumes = cell_volume(points, neighbors, sigma)
    return sigma, volumes


In [ ]:
# 由于 Voronoi 可能产生一些孤立点（没有邻居），我们需要扩展 neighbors 和 volumes 来确保每个点都有足够的邻居和体积信息。同时让产生的neighbors等量的长度为 D（2^M），以便后续量子算法处理
# 这里我们简单地复制最后一个点的邻居和体积来填充（比全零更物理）
# 2026-06-03
def extend_neighbors_and_volumes(neighbors, volumes, new_len):
    old_len = len(neighbors)
    assert old_len == len(volumes)

    if old_len >= new_len:
        return neighbors, volumes

    pad = new_len - old_len

    neighbors_ext = list(neighbors)
    last_nei = neighbors[-1]  # 复制边界邻接
    for _ in range(pad):
        neighbors_ext.append(list(last_nei))

    volumes_ext = np.concatenate([volumes, np.repeat(volumes[-1], pad)])

    return neighbors_ext, volumes_ext


In [ ]:
# 由于 Voronoi 可能产生一些孤立点（没有邻居），我们需要扩展 neighbors, sigma 和 volumes 来确保每个点都有足够的邻居和体积信息。同时让产生的neighbors等量的长度为 D（2^M），以便后续量子算法处理
# 这里我们简单地复制最后一个点的邻居和体积来填充（比全零更物理）
# 2026-06-05
def extend_geometry(neighbors, volumes, sigma, new_len):
    old_len = len(neighbors)
    assert old_len == len(volumes)
    assert old_len == len(sigma)

    # 1) neighbors: dict → list
    neighbors_ext = [list(neighbors[i]) for i in range(old_len)]

    # 2) volumes
    volumes_ext = np.array(volumes, dtype=float)

    # 3) sigma: deep copy
    import copy
    sigma_ext = [copy.deepcopy(sigma[i]) for i in range(old_len)]

    if old_len >= new_len:
        return neighbors_ext, volumes_ext, sigma_ext

    pad = new_len - old_len

    # last_nei = neighbors_ext[-1]
    # last_sigma = sigma_ext[-1]
    # last_vol = volumes_ext[-1]

    # # 扩展 neighbors / volumes / sigma
    # for _ in range(pad):
    #     neighbors_ext.append(list(last_nei))
    #     volumes_ext = np.append(volumes_ext, last_vol)
    #     sigma_ext.append(copy.deepcopy(last_sigma))

    # 使用物理格点的平均体积作为填充点的默认体积（避免 0 或极小值）
    default_vol = float(np.median(volumes)) if old_len > 0 else 1.0

    # 4) 填充处理：填充格点作为孤立格点，不与现有物理格点建立虚假连接
    for _ in range(pad):
        neighbors_ext.append([])  # 无邻居，保证无向图对称性不被破坏
        volumes_ext = np.append(volumes_ext, default_vol)
        sigma_ext.append({})     # 无边界截面（area = 0

    return neighbors_ext, volumes_ext, sigma_ext


# 2. 一次量子化 Hamiltonian 组装
## Laplacian 构造
 ### L_{mn} in Eqn. 15

In [ ]:
import numpy as np
import scipy.sparse as sp
from collections import defaultdict

def lapl_matr(vor, points, neighbors):
    """
    【升级版】3D Voronoi Laplacian (论文式 (15))
    - 接口保持不变
    - 使用升级后的 build_geometry + cell_volume
    - 输出为 CSR 稀疏矩阵（高效、可直接用于 ADAPT-VQE）
    """

    # ---------------------------------------------------------
    # 1. 获取 facet 几何信息 + Voronoi cell 体积
    # ---------------------------------------------------------
    sigma, volumes = build_geometry(points, neighbors, vor)

    N = len(points)
    num_bits = math.ceil(math.log2(N))
    dim = 1 << num_bits

    rows = []
    cols = []
    vals = []

    # ---------------------------------------------------------
    # 2. 构造 Laplacian（论文式 (15)）
    # ---------------------------------------------------------
    for m in range(N):
        Vm = volumes[m]
        diag_sum = 0.0

        # diagonal term
        for k in neighbors[m]:
            if k not in sigma[m]:
                continue

            area = sigma[m][k]["area"]
            dist = np.linalg.norm(points[m] - points[k])

            diag_sum += area / dist

        # L_mm = - (sum A_mk / d_mk) / V_m
        rows.append(m)
        cols.append(m)
        vals.append(-diag_sum / Vm)

        # off-diagonal terms
        for n in neighbors[m]:
            if n not in sigma[m]:
                continue

            area = sigma[m][n]["area"]
            dist = np.linalg.norm(points[m] - points[n])
            Vn = volumes[n]

            # L_mn = A_mn / (sqrt(Vm Vn) * d_mn)
            val = area / (np.sqrt(Vm * Vn) * dist)

            rows.append(m)
            cols.append(n)
            vals.append(val)

    # ---------------------------------------------------------
    # 3. 构造稀疏矩阵（dim × dim），物理块在左上角
    # ---------------------------------------------------------
    L = sp.csr_matrix((vals, (rows, cols)), shape=(dim, dim), dtype=np.float64)

    return L


### Construct symmetric laplacian

In [ ]:
import numpy as np
import scipy.sparse as sp

def laplacian(points, neighbors, sigma, volumes):
    N = len(points)
    rows, cols, vals = [], [], []

    for m in range(N):
        Vm = volumes[m]
        diag_sum = 0.0

        for n in neighbors[m]:
            if n not in sigma[m]:
                continue

            area = sigma[m][n]["area"]
            dist = np.linalg.norm(points[m] - points[n])
            if dist < 1e-12:
                continue

            diag_sum += area / dist

        rows.append(m); cols.append(m); vals.append(-diag_sum / Vm)

        for n in neighbors[m]:
            if n not in sigma[m]:
                continue

            area = sigma[m][n]["area"]
            dist = np.linalg.norm(points[m] - points[n])
            if dist < 1e-12:
                continue

            val = area / (np.sqrt(Vm * volumes[n]) * dist)
            rows.append(m); cols.append(n); vals.append(val)

    return sp.csr_matrix((vals, (rows, cols)), shape=(N, N))


## Coulombic potential

In [ ]:
def coulombic_potential(points):
    """
    【升级版】单电子 Coulomb 势能
    - 接口保持不变：返回 (ele_nu, ele_ele)
    - ele_nu: 长度 dim 的一维数组（嵌入 Hilbert 空间）
    - ele_ele: dim × dim 的零矩阵（单电子模型不需要 2-body）
    """

    length = len(points)
    num_bits = math.ceil(math.log2(length))
    dim = 1 << num_bits

    # electron–nucleus attraction
    ele_nu = np.zeros(dim, dtype=np.float64)

    Z = 1.0
    eps = 1e-8  # avoid singularity

    for m in range(length):
        r = np.linalg.norm(points[m])
        r = max(r, eps)
        ele_nu[m] = -Z / r

    # electron–electron repulsion (single-electron model → zero)
    ele_ele = np.zeros((dim, dim), dtype=np.float64)

    return ele_nu, ele_ele


In [ ]:
# def coulombic_potential(points, volumes):
    
#     length = len(points)
#     num_bits = math.ceil(math.log2(length))
#     dim = 1 << num_bits

#     r = np.linalg.norm(points, axis=1)
#     V_potential =  np.zeros(dim, dtype=np.float64)
    
#     # 找到最靠近原点的格点（通常是 Index 0）
#     origin_idx = np.argmin(r)
#     r_nonzero = r[r > 1e-12]
#     r_min = np.min(r_nonzero) if len(r_nonzero) > 0 else 1e-3  # 避免 r_min 为零
    
#     # 计算原点胞腔的等效半径
#     # v_origin = volumes[origin_idx]
#     # R_vol = (3.0 * v_origin / (4.0 * np.pi)) ** (1.0 / 3.0)
#     # R_eff = max(R_vol, r_min)  # 物理保护：R_eff 不得小于最近邻居距离

#     # 将柔化截断半径设为第一层格点的特征范围（例如 1.5 ~ 2.0 倍 r_min）
#     R_cutoff = 1.5 * r_min
#     V_cap = -1.0 / R_cutoff  # 柔化势能上限
    
#     # V_origin = -1.0 / R_eff
#     # # 对所有格点计算柔化势能
#     # for i in range(len(points)):
#     #     if i == origin_idx or r[i] < 1e-12:
#     #         V_potential[i] = V_origin
#     #     else:
#     #         V_potential[i] = -1.0 / r[i]

#     for i in range(len(points)):
#         if r[i] < R_cutoff:
#             V_potential[i] = V_cap
#         else:
#             V_potential[i] = -1.0 / r[i]
    
#     # electron–electron repulsion (single-electron model → zero)
#     ele_ele = np.zeros((dim, dim), dtype=np.float64)
#     return V_potential, ele_ele

## Hamiltonian Matrix

In [ ]:
import numpy as np
import scipy.sparse as sp
import math

def hamilton_14(points, neighbors, sigma, volumes):
    N = len(points)
    num_bits = math.ceil(math.log2(N))
    dim = 1 << num_bits

    L_sym = laplacian(points, neighbors, sigma, volumes)
    # ele_nu, _ = coulombic_potential(points, volumes)
    ele_nu, _ = coulombic_potential(points)

    rows, cols, vals = [], [], []

    L = L_sym.tocoo()
    for m, n, v in zip(L.row, L.col, L.data):
        rows.append(m); cols.append(n); vals.append(-0.5 * v)

    for m in range(dim):
        rows.append(m); cols.append(m); vals.append(ele_nu[m])

    return sp.csr_matrix((vals, (rows, cols)), shape=(dim, dim))

### 试图找到最佳的alpha和nu的参数

In [ ]:
import numpy as np
import scipy.sparse as sp

def compute_voronoi_volumes_and_faces(vor, N):
    """
    从 scipy.spatial.Voronoi 对象中提取每个网格点的体积 volumes 
    以及近邻点之间的共享面面积 sigma[m][n]["area"]
    """
    # 1. 计算每个有限 Voronoi 胞腔的体积 (简易估计或凸包体积)
    # 注意：边界外层的点胞腔体积可能无限大 (infinite region)，需要设置 Cutoff
    volumes = np.zeros(N)
    for i in range(N):
        region_idx = vor.point_region[i]
        region = vor.regions[region_idx]
        if -1 not in region and len(region) > 0:
            # 使用 Voronoi 顶点计算凸包体积
            try:
                from scipy.spatial import ConvexHull
                hull = ConvexHull(vor.vertices[region])
                volumes[i] = hull.volume
            except Exception:
                volumes[i] = 1e-3  # 异常兜底
        else:
            volumes[i] = 1e-3  # 边界无穷大胞腔的截断体积

    # 2. 提取脊线（Ridges）共享面面积 sigma
    sigma = {i: {} for i in range(N)}
    for (p, q), ridge_vert_indices in zip(vor.ridge_points, vor.ridge_vertices):
        if -1 not in ridge_vert_indices and len(ridge_vert_indices) >= 3:
            # 2D/3D 多边形面元面积计算
            ridge_pts = vor.vertices[ridge_vert_indices]
            # 计算 3D 空间中多边形的面积 (利用交叉积)
            v0 = ridge_pts[0]
            area = 0.0
            for k in range(1, len(ridge_pts) - 1):
                v1 = ridge_pts[k] - v0
                v2 = ridge_pts[k+1] - v0
                area += 0.5 * np.linalg.norm(np.cross(v1, v2))
            
            sigma[p][q] = {"area": area}
            sigma[q][p] = {"area": area}

    return volumes, sigma

In [ ]:
def build_hydrogen_hamiltonian_csr(points, neighbors, vor, sigma_cutoff=0.1):
    N = len(points)
    
    # 1. 计算 Voronoi 的 volumes 与 sigma 面积字典
    volumes, sigma = compute_voronoi_volumes_and_faces(vor, N)
    
    # 2. 调用你的 laplacian 函数构建动能项 L_sym
    L_sym = laplacian(points, neighbors, sigma, volumes)
    
    # 3. 计算库仑势 (例如 erf 平滑势)
    r_vals = np.linalg.norm(points, axis=1)
    r_safe = np.maximum(r_vals, 1e-12)
    from scipy.special import erf
    ele_nu = -erf(r_safe / sigma_cutoff) / r_safe
    
    # 4. 组装 H_csr 矩阵 (-0.5 * L_sym + V)
    import math
    num_bits = math.ceil(math.log2(N))
    dim = 1 << num_bits
    
    L = L_sym.tocoo()
    rows, cols, vals = [], [], []
    for m, n, v in zip(L.row, L.col, L.data):
        rows.append(m)
        cols.append(n)
        vals.append(-0.5 * v)

    for m in range(N):
        rows.append(m)
        cols.append(m)
        vals.append(ele_nu[m])

    # 对填充点（Padding）增加高能量屏障，防止物理态混淆
    for m in range(N, dim):
        rows.append(m)
        cols.append(m)
        vals.append(10.0)

    return sp.csr_matrix((vals, (rows, cols)), shape=(dim, dim))

In [ ]:
import numpy as np
from scipy.sparse.linalg import eigsh

# ------------------------------------------------------------------
# 1. 参数扫描诊断函数
# ------------------------------------------------------------------
def benchmark_grid_parameters(num_r=16, order=6, alpha_list=[1.0, 1.5, 2.0], nu_list=[1.2, 1.5, 2.0]):
    """
    扫描 alpha 和 nu，评估其对网格几何、H矩阵特征谱以及 ADAPT-VQE 收敛性的影响
    """
    print(f"{'alpha':<6} | {'nu':<6} | {'N_pts':<6} | {'r_min':<10} | {'E_exact (Ha)':<14} | {'Spec Width':<12} | {'Status'}")
    print("-" * 80)

    results = []

    for alpha in alpha_list:
        for nu in nu_list:
            try:
                # A. 生成特定 alpha, nu 的 Lebedev 网格
                points, neighbors, vor = hydrogen_adaptive_grid_lebedev(
                    num_r=num_r, order=order, alpha=alpha, nu=nu
                )
                
                # B. 计算最小间距
                r_vals = np.linalg.norm(points, axis=1)
                r_min = np.min(r_vals[r_vals > 0])

                # C. 构建当前网格下的 H_csr (使用你之前编写的 H 构建函数)
                # 注意：请确保你的 build_h_csr 内部调用了当前 alpha, nu 下的网格与势能
                H_csr = build_hydrogen_hamiltonian_csr(points, neighbors, vor)
                
                # D. 计算 H 矩阵的最大/最小特征值 (谱宽与精准基态能量)
                # 最小特征值 (基态)
                e_min, _ = eigsh(H_csr, k=1, sigma=-0.5, which='LM')
                E_exact = e_min[0]
                
                # 最大特征值 (评估动能爆炸)
                e_max, _ = eigsh(H_csr, k=1,  which='LA')
                spec_width = e_max[0] - e_min[0]

                # E. 运行 4 步 ADAPT-VQE 测试收敛速度
                # (复用之前优化后的 precomputed pool_sparse)
                # psi_vqe, _, thetas, _ = adapt_vqe_first_quantized_v5_8_tiered(...)
                
                print(f"{alpha:<6.2f} | {nu:<6.2f} | {len(points):<6} | {r_min:<10.3e} | {E_exact:<14.6f} | {spec_width:<12.2e} | OK")
                
                results.append({
                    'alpha': alpha, 'nu': nu, 'r_min': r_min, 
                    'E_exact': E_exact, 'spec_width': spec_width,
                    'H_csr': H_csr, 'points': points
                })

            except Exception as e:
                print(f"{alpha:<6.2f} | {nu:<6.2f} | ERROR: {str(e)}")

    return results

# 运行扫描 (以 256 点 / 8-qubit 规模测试)
# test_results = benchmark_grid_parameters()
results = benchmark_grid_parameters(num_r=20, order=50, alpha_list=[1.0, 1.5, 2.0, 2.5, 4.0], nu_list=[1.2, 1.5, 2.0])

## Test Hamiltonian

In [ ]:
# import numpy as np
sigma, volumes = build_geometry(points, neighbors, vor)
H_sparse = hamilton_14(points, neighbors, sigma, volumes)

# 转成 dense matrix
H = H_sparse.toarray()

vals, vecs = np.linalg.eigh(H)
print(vals[0])
print(vals[-1])
#print(vecs[0])
print("================================")
neighbors_ext, volumes_ext, sigma_ext = extend_geometry(neighbors, volumes, sigma, D)

H_sparse_ext = hamilton_14(points_ext, neighbors_ext, sigma_ext, volumes_ext)

H_ext = H_sparse_ext.toarray()
vals_ext, vecs_ext = np.linalg.eigh(H_ext)
print(vals_ext[0])
print(vals_ext[-1])


In [ ]:
# 1. 仅在未 padding 的原始物理矩阵上测试广义本征值 (排除 Padding 的影响)
M_orig = np.diag(volumes)
H_orig = H_sparse.toarray()

# 检查 H_orig 的原始标准本征值
vals_raw, _ = np.linalg.eigh(H_orig)
print("未 Padding 的原始矩阵标准本征值:", vals_raw[0])

# 检查物理原点附近的最小半径
r_min = np.min(np.linalg.norm(points, axis=1))
print("距离原点最近的格点距离 r_min:", r_min)

# Pauli_LCU


## Matrix of T_bar

In [ ]:
def t_bar_matr(points, neighbors, sigma, volumes):
    """
    文献式 (21):
    T̄ = -1/2 * L̃ + V_nuc
    只构造物理子空间 (length × length)
    """

    length = len(points)

    # 对称化 Laplacian（length × length CSR）
    L_sym = laplacian(points, neighbors, sigma, volumes)

    # 势能（dim 维，但我们只取前 length）
    # ele_nu, _ = coulombic_potential(points, volumes)
    ele_nu, _ = coulombic_potential(points)
    V = ele_nu[:length]

    # 构造 T̄（稀疏）
    rows, cols, vals = [], [], []

    # 动能部分
    L = L_sym.tocoo()
    for m, n, v in zip(L.row, L.col, L.data):
        if m < length and n < length:
            rows.append(m)
            cols.append(n)
            vals.append(-0.5 * v)

    # 势能部分（对角）
    for m in range(length):
        rows.append(m)
        cols.append(m)
        vals.append(V[m])

    Tbar = sp.csr_matrix((vals, (rows, cols)), shape=(length, length))
    return Tbar


In [ ]:
Tbar = t_bar_matr(points, neighbors, sigma, volumes)
vals, vecs = np.linalg.eigh(Tbar.toarray())
print(vals[:10])
print("end===============================")
Tbar_ext = t_bar_matr(points_ext, neighbors_ext, sigma_ext, volumes_ext)
vals_ext, vecs_ext = np.linalg.eigh(Tbar_ext.toarray())
print(vals_ext[:10])

## Bitwise code

### Bitwise dot

In [ ]:
# This function is to realize bitwise dot
# 2025-12-29
def bitwise_dot(m, n, num_bits):
    return bin(m & n).count("1") & 1


###  Bitwise XOR

In [ ]:
# This function is for bitwise xor
# 2025-12-29
def bitwise_xor(m, n, num_bits):
    """Return the bitwise XOR of m and n, restricted to num_bits bits."""
    mask = (1 << num_bits) - 1
    return (m ^ n) & mask


### Integer to Bits

In [ ]:
def int_to_bits(x, num_bits):
    return [(x >> b) & 1 for b in range(num_bits)]


## Original Omega Matrix

In [ ]:
def omega_matr(points, neighbors, sigma, volumes):
    length = len(points)
    num_bits = math.ceil(math.log2(length))
    dim = 1 << num_bits

    # 1. 构造物理子空间的 T̄（length×length）
    Tbar = t_bar_matr(points, neighbors, sigma, volumes).toarray()

    # 2. 扩展到 dim×dim（右下角填 0）
    T = np.zeros((dim, dim))
    T[:length, :length] = Tbar

    # 3. 构造 Hadamard 矩阵 H
    H = hadamard(dim) / np.sqrt(dim)

    # 4. Omega = H T H
    Omega = H @ T @ H

    return Omega


## FWHT

In [ ]:
"""In-place Fast Walsh–Hadamard Transform on last axis."""
def fwht(a):
    h = 1
    n = len(a)
    while h < n:
        step = h * 2
        for i in range(0, n, step):
            for j in range(i, i + h):
                x = a[j]
                y = a[j + h]
                a[j] = x + y
                a[j + h] = x - y
        h *= 2
    # 正交化
    a /= np.sqrt(n)


In [ ]:
def fwht2(mat):
    """
    2D Fast Walsh–Hadamard Transform
    mat: dim × dim numpy array
    """
    dim = mat.shape[0]

    # FWHT on rows
    for i in range(dim):
        fwht(mat[i])

    # FWHT on columns
    for j in range(dim):
        col = mat[:, j]
        fwht(col)
        mat[:, j] = col

    return mat


## Omega Matrix with FWHT

In [ ]:
def omega_matr_fwht(points, neighbors, sigma, volumes):
    length = len(points)
    num_bits = math.ceil(math.log2(length))
    dim = 1 << num_bits

    Tbar = t_bar_matr(points, neighbors, sigma, volumes).toarray()

    T = np.zeros((dim, dim))
    T[:length, :length] = Tbar

    # 行 FWHT
    for i in range(dim):
        fwht(T[i])

    # 列 FWHT
    for j in range(dim):
        col = T[:, j]
        fwht(col) 
        T[:, j] = col

    return T   # 不再除 dim


In [ ]:
# 1. 严格按照第 21 式，在底层就将 T 和 V 绑定并经过 Voronoi 度规矩阵 S 做相似变换
# 这样吐出来的 Hbar 才是数理上完全合规、没有度规错配的总实空间矩阵
Hbar = t_bar_matr(points, neighbors, sigma, volumes).toarray()  # 确保你这个函数里面已经把 V 包进去了

# 2. 此时再进行数值正则化“斩首”手术
# 找出由于体积元太小导致总能量对角线发生非物理暴涨的边缘坏点
diagonal_elements = np.diag(Hbar)
bad_indices = np.where(diagonal_elements > 1e4)[0]
print(f"⚠️ 依据第 21 式流形，定位到发散坏点索引: {bad_indices}")

# 3. 斩首清零（让坏点处的物理波函数严格代数衰减为 0）
for idx in bad_indices:
    Hbar[idx, :] = 0.0
    Hbar[:, idx] = 0.0

# 4. 此时得到的 Hbar 才是完全洗净、安全、可以扔进第 24 式 FWHT 的完美基座
w_perfect, _ = np.linalg.eigh(Hbar)
print(f"✨ 遵循21式洗净后最低能级 (物理基态): {w_perfect[0]}")
print(f"✨ 遵循21式洗净后最高能级 (发散彻底封锁): {w_perfect[-1]}")

In [ ]:
import math
import numpy as np
from scipy.linalg import hadamard

# ======================================================================
# 1. 严格正交、绝无多重覆盖污染的完全体快速沃尔什变换（FWHT）
# ======================================================================
def fwht_pure_orthogonal(a):
    """
    通过逐层全量 Buffer 隔离的快速沃尔什变换
    - 100% 免疫 In-place 乱序覆盖暗伤
    - 严格保持正交性，每一层蝶形网络物理同构
    """
    arr = np.array(a, dtype=np.float64, copy=True)
    n = len(arr)
    num_stages = int(math.log2(n))
    
    # 逐层演化
    for stage in range(num_stages):
        # 建立当前层绝对干净的隔离缓冲区
        next_arr = np.zeros_like(arr)
        stride = 1 << stage
        
        for i in range(n):
            if (i & stride) == 0:
                # 蝶形算子正向合并
                next_arr[i]          = arr[i] + arr[i + stride]
                next_arr[i + stride] = arr[i] - arr[i + stride]
                
        arr = next_arr
        
    # 🌟 严格正交化归一化：每演化一层，注入一次 1/sqrt(2) 缩放因子
    return arr / np.sqrt(n)

# ======================================================================
# 2. 封杀调用流，构造唯一的物理矩阵
# ======================================================================
Tbar_holy_source = t_bar_matr(points, neighbors, sigma, volumes).toarray()
length = Tbar_holy_source.shape[0]
num_bits = math.ceil(math.log2(length))
dim = 1 << num_bits

# 构筑纯净的目标 T_ext 大框
T_ext = np.zeros((dim, dim))
T_ext[:length, :length] = Tbar_holy_source.copy()

# 🚀 正向变换：利用全隔离 FWHT 算 Ω (行变换 + 列变换)
omega_pg = np.zeros((dim, dim))
# 2.1 行变换
for i in range(dim):
    omega_pg[i, :] = fwht_pure_orthogonal(T_ext[i, :])
# 2.2 列变换
for j in range(dim):
    omega_pg[:, j] = fwht_pure_orthogonal(omega_pg[:, j])

# 🚀 反向变换：利用完全同源的正交 FWHT 完美变回实空间
T_back = np.zeros((dim, dim))
# 3.1 行反变换
for i in range(dim):
    T_back[i, :] = fwht_pure_orthogonal(omega_pg[i, :])
# 3.2 列反变换
for j in range(dim):
    T_back[:, j] = fwht_pure_orthogonal(T_back[:, j])

# ======================================================================
# 3. 终极自洽看板
# ======================================================================
print("=== 🔬 绝杀版快速算法正反自洽对齐看板 ===")
max_diff = np.max(np.abs(T_back - T_ext))
print(f"🌟 1. 纯净快速算法反投最大误差: {max_diff:.6e}")

vals_Text = np.sort(np.linalg.eigvals(T_ext).real)
vals_O = np.sort(np.linalg.eigvals(omega_pg).real)
max_eig_diff = np.max(np.abs(vals_Text - vals_O))
print(f"🌟 2. 纯净快速算法本征值最大误差: {max_eig_diff:.6e}")

In [ ]:
import numpy as np

def metric_similarity_transform(T, volumes):
    """
    对离散算符 T 做严格体积度规相似变换:
        T_tilde = S T S^{-1},  S = diag(sqrt(V_i))

    作用：
    ----
    - 消除非均匀网格 Laplacian 的“伪非厄米性”
    - 将算符从带权内积空间映射到标准欧氏空间
    - 得到严格对称（厄米）的矩阵，适合 FWHT / LCU / VQE

    参数：
    ----
    T : (N, N) ndarray
        原始离散算符矩阵（可能略非对称）
    volumes : (N,) ndarray
        每个网格点的体积元 V_i > 0

    返回：
    ----
    T_tilde : (N, N) ndarray
        在标准欧氏内积下的厄米矩阵
    """
    V = np.asarray(volumes, dtype=np.float64)
    assert T.shape[0] == T.shape[1] == V.shape[0], "维度不匹配"

    # S = diag(sqrt(V_i))
    s = np.sqrt(V)

    # 数值防御：避免极小体积导致的除零
    eps = 1e-30
    inv_s = np.where(s > eps, 1.0 / s, 0.0)

    # 相似变换：S T S^{-1}
    T_tilde = (s[:, None] * T) * inv_s[None, :]

    # 最终对称化：抹平浮点误差（不会改变物理）
    T_tilde = 0.5 * (T_tilde + T_tilde.T)

    return T_tilde


## pauli_lcu_one_body_from_omega

接受Gemini的建议： 在 LCU 循环内部使用 defaultdict 进行即时合并，并在合并过程中直接应用 tol。改造上面的pauli_lcu_one_body_from_omega为pauli_lcu_one_body_from_omega_efficient.


In [ ]:
def pauli_lcu_one_body_from_omega(omega, eta, tol=1e-12):
    D = omega.shape[0]
    M = int(np.log2(D))
    n_qubits = eta * M

    acc = defaultdict(complex)

    for p in range(D):
        p_bits = [(p >> k) & 1 for k in range(M)]
        for q in range(D):
            coef = omega[p, q]
            if abs(coef) < tol:
                continue

            q_bits = [(q >> k) & 1 for k in range(M)]

            for i in range(eta):
                chars = ["I"] * n_qubits
                temp = coef

                base = i * M
                for k in range(M):
                    x = p_bits[k]
                    z = q_bits[k]
                    qubit = base + k

                    if z and not x:
                        chars[qubit] = "Z"
                    elif x and not z:
                        chars[qubit] = "X"
                    elif x and z:
                        chars[qubit] = "Y"
                        temp *= 1j

                pauli = "".join(chars[::-1])
                acc[pauli] += temp

    final = [(p, c) for p, c in acc.items() if abs(c) > tol]
    return SparsePauliOp.from_list(final)


In [ ]:
from collections import defaultdict
import numpy as np
from qiskit.quantum_info import SparsePauliOp

def pauli_lcu_one_body_fast_pure(omega, eta, tol=1e-12):
    """
    基于厄米成对合并的高性能、单Pauli串第一量子化展开算法
    复杂度：严格 O(D^2 * M)，杜绝任何项数爆炸。
    """
    omega = np.asarray(omega, dtype=np.complex128)
    D = omega.shape[0]
    M = int(np.log2(D))
    n_qubits = eta * M

    # 预计算二进制表
    bit_table = [[(i >> k) & 1 for k in range(M)] for i in range(D)]
    acc = defaultdict(complex)

    # ======================================================================
    # 1. 处理非对角元 p > q (成对合并：p 和 q 一次性搞定)
    # ======================================================================
    for p in range(D):
        p_bits = bit_table[p]
        for q in range(p):  # 只走下三角，严格控制 O(D^2 / 2)
            val = omega[p, q]
            if abs(val) < tol:
                continue
            
            q_bits = bit_table[q]
            
            # 计算满足严格 LCU 的单串基底
            # 对于 |p><q| + |q><p|，非相同位直接被 X 或 Y 锁定
            chars_block_real = ["I"] * M
            chars_block_imag = ["I"] * M
            
            # 统计 p_k=1, q_k=0 的数量用于计算符号相位
            n_10 = 0
            for k in range(M):
                if p_bits[k] == 1 and q_bits[k] == 0:
                    n_10 += 1

            # 纯正的正交单串映射核心
            phase_factor = (-1.0) ** n_10
            
            # 构造基础字符串：不同位用 X，相同位用 Z (或保持 I)
            # 这里采用标准一阶量子化空间中，由对称/反对称组合直接派生的单串路径
            for k in range(M):
                if p_bits[k] != q_bits[k]:
                    # 不同的位被 X 激活
                    chars_block_real[k] = "X"
                    chars_block_imag[k] = "Y"
                else:
                    # 相同的位被 Z 激活 (保留其在计算基下的对角符号贡献)
                    if p_bits[k] == 1:
                        chars_block_real[k] = "Z"
                        chars_block_imag[k] = "Z"

            # 复制到 eta 个电子上
            for i in range(eta):
                base = i * M
                
                # 实部项 (对应 H 的对称部分)
                if abs(val.real) > tol:
                    chars_global_r = ["I"] * n_qubits
                    for k in range(M):
                        chars_global_r[base + k] = chars_block_real[k]
                    acc[tuple(chars_global_r)] += val.real * phase_factor
                
                # 虚部项 (对应 H 的反对称部分)
                if abs(val.imag) > tol:
                    chars_global_i = ["I"] * n_qubits
                    for k in range(M):
                        chars_global_i[base + k] = chars_block_imag[k]
                    # Y 算符自带的 1j 与虚部结合
                    acc[tuple(chars_global_i)] += val.imag * phase_factor * 1j

    # ======================================================================
    # 2. 处理对角元 p == q (纯对角 Z 串，无任何分裂)
    # ======================================================================
    for p in range(D):
        val = omega[p, p].real
        if abs(val) < tol:
            continue
            
        p_bits = bit_table[p]
        chars_block = ["I"] * M
        for k in range(M):
            if p_bits[k] == 1:
                chars_block[k] = "Z"
                
        for i in range(eta):
            chars_global = ["I"] * n_qubits
            base = i * M
            for k in range(M):
                chars_global[base + k] = chars_block[k]
            acc[tuple(chars_global)] += val

    # ======================================================================
    # 3. 收尾并输出为 Qiskit 格式
    # ======================================================================
    final = []
    for pauli_tuple, c in acc.items():
        if abs(c) <= tol:
            continue
        pauli_str = "".join(pauli_tuple[::-1])
        final.append((pauli_str, c))

    return SparsePauliOp.from_list(final)

In [ ]:
import numpy as np
from scipy.sparse import coo_matrix
from qiskit.quantum_info import SparsePauliOp

def build_circuit_hamiltonian_eq22(omega_matrix, tol=1e-12):
    """
    完全优化版 Eq.22 → Pauli 展开：
    - Ω 用阈值剪枝 + COO 稀疏遍历
    - projector 展开用 NumPy 向量化完成
    - Pauli 串用 base-4 整数编码，最后一次性合并
    """
    omega = np.asarray(omega_matrix, dtype=np.complex128)
    D = omega.shape[0]
    M = int(np.log2(D))

    # 1. 阈值剪枝 + COO
    mask = np.abs(omega) > tol
    rows, cols = np.nonzero(mask)
    vals = omega[rows, cols]
    nnz = len(vals)
    print(f"[Eq22] nnz(Ω) = {nnz} / {D*D}")

    # 2. 预计算所有索引的二进制展开 bits[p, k]
    bits = ((np.arange(D)[:, None] >> np.arange(M)) & 1).astype(np.int8)

    # 3. 单比特 projector → Pauli 的表（字符和权重）
    # 编码: I=0, X=1, Y=2, Z=3
    single_bit_codes = {
        (0, 0): np.array([0, 3], dtype=np.uint8),  # I, Z
        (1, 1): np.array([0, 3], dtype=np.uint8),  # I, Z
        (0, 1): np.array([1, 2], dtype=np.uint8),  # X, Y
        (1, 0): np.array([1, 2], dtype=np.uint8),  # X, Y
    }
    single_bit_weights = {
        (0, 0): np.array([0.5,  0.5],  dtype=np.complex128),
        (1, 1): np.array([0.5, -0.5],  dtype=np.complex128),
        (0, 1): np.array([0.5,  0.5j], dtype=np.complex128),
        (1, 0): np.array([0.5, -0.5j], dtype=np.complex128),
    }

    # 4. 收集所有 (p,q) 展开后的 Pauli 编码和系数（先放大数组，再一次性合并）
    # 最坏 nnz * 2^M 个项，M=11 时要注意内存，tol 要选得合理
    all_keys = []
    all_vals = []

    for coef, p, q in zip(vals, rows, cols):
        p_bits = bits[p]  # shape (M,)
        q_bits = bits[q]  # shape (M,)

        # 初始：只有一个“空串”，系数为当前矩阵元
        coeffs = np.array([coef], dtype=np.complex128)
        codes = np.zeros((1, M), dtype=np.uint8)  # 每行一个 Pauli 串（编码）

        for k in range(M):
            x = int(p_bits[k])
            z = int(q_bits[k])
            code_k = single_bit_codes[(x, z)]      # shape (2,)
            weight_k = single_bit_weights[(x, z)]  # shape (2,)

            n = coeffs.shape[0]

            # 新系数：广播相乘 (n,1)*(1,2) → (n,2) → (2n,)
            new_coeffs = (coeffs[:, None] * weight_k[None, :]).reshape(-1)

            # 新编码：重复旧串，再在第 k 位填入 code_k
            new_codes = np.repeat(codes, 2, axis=0)
            new_codes[:, k] = np.tile(code_k, n)

            coeffs = new_coeffs
            codes = new_codes

        # 过滤掉极小项，避免无意义膨胀
        mask_nz = np.abs(coeffs) > tol
        if not np.any(mask_nz):
            continue

        all_keys.append(codes[mask_nz])
        all_vals.append(coeffs[mask_nz])

    if not all_keys:
        # 全部被截断，返回零算符
        return SparsePauliOp.from_list([("I"*M, 0.0)])

    all_keys = np.vstack(all_keys)   # shape (N, M)
    all_vals = np.concatenate(all_vals)  # shape (N,)

    # 5. 把每个 Pauli 串编码成 base-4 整数 key，方便用 np.unique 合并
    # key = sum_{k=0}^{M-1} code[k] * 4^k
    powers4 = (4 ** np.arange(M, dtype=np.uint64))  # shape (M,)
    keys_int = (all_keys.astype(np.uint64) * powers4[None, :]).sum(axis=1)

    # 6. 用 np.unique + bincount 合并同类项
    uniq_keys, inv = np.unique(keys_int, return_inverse=True)
    real_part = np.bincount(inv, weights=all_vals.real)
    imag_part = np.bincount(inv, weights=all_vals.imag)
    final_coeffs = real_part + 1j * imag_part

    # 7. 把整数编码还原成 Pauli 字符串（注意 Qiskit 小端序）
    pauli_list = []
    for key, c in zip(uniq_keys, final_coeffs):
        if abs(c) <= tol:
            continue

        # 解码 base-4 → 每个比特的 Pauli 编码
        codes_row = []
        x = int(key)
        for _ in range(M):
            codes_row.append(x % 4)
            x //= 4

        # 映射表
        char_map = {0: 'I', 1: 'X', 2: 'Y', 3: 'Z'}

        # 关键微调：逆序拼接，满足 Qiskit 小端序
        pauli_str = "".join(char_map[codes_row[k]] for k in reversed(range(M)))

        pauli_list.append((pauli_str, c))

    print(f"[Eq22] 最终独立 Pauli 项数 = {len(pauli_list)}")
    return SparsePauliOp.from_list(pauli_list)


In [ ]:
from collections import defaultdict
import numpy as np
import math
from qiskit.quantum_info import SparsePauliOp

def pauli_lcu_one_body_strict_compressed(omega, eta, tol=1e-12):
    """
    100% 数理严格、无任何物理失真的一体算符第一量子化 Pauli 映射算法
    利用局域字典压缩（Local Compression）技术，将原本 O(D^3) 的 Qiskit 构建开销
    压缩至高度精简的独立项，彻底兼顾高效率与绝对真理。
    """
    omega = np.asarray(omega, dtype=np.complex128)
    D = omega.shape[0]
    M = int(np.log2(D))
    n_qubits = eta * M

    # 1. 预计算单比特投影算符的严格分解成分
    single_bit_maps = {
        (0, 0): [('I', 0.5),  ('Z', 0.5)],
        (1, 1): [('I', 0.5),  ('Z', -0.5)],
        (0, 1): [('X', 0.5),  ('Y', 0.5j)],
        (1, 0): [('X', 0.5),  ('Y', -0.5j)]
    }

    bit_table = [[(i >> k) & 1 for k in range(M)] for i in range(D)]

    # 🚀 第一级核心压缩：单电子 M-qubit 空间的局域累加字典
    # 这一步将 D^2 个矩阵元的 LCU 展开在单电子空间内直接合并同类项！
    local_acc = defaultdict(complex)

    for p in range(D):
        p_bits = bit_table[p]
        for q in range(D):
            coef = omega[p, q]
            if abs(coef) < tol:
                continue

            q_bits = bit_table[q]

            # 严格生成当前 |p><q| 的 2^M 个 Pauli 分量
            block_items = [([], 1.0 + 0.0j)]
            for k in range(M):
                x = p_bits[k]
                z = q_bits[k]
                allowed_paulis = single_bit_maps[(x, z)]
                
                next_block_items = []
                for chars, w in block_items:
                    for p_char, p_w in allowed_paulis:
                        next_block_items.append((chars + [p_char], w * p_w))
                block_items = next_block_items

            # 将这 2^M 个分量合并到单电子局域字典中
            for chars_block, w_block in block_items:
                term_coef = coef * w_block
                if abs(term_coef) > tol:
                    local_acc[tuple(chars_block)] += term_coef

    #  第二级核心压缩：平移复制到 eta 个电子的多体空间
    global_acc = defaultdict(complex)

    for chars_block, local_coef in local_acc.items():
        if abs(local_coef) < tol:
            continue
            
        # 优化特判：如果是全 I 项，全空间复制等价于 local_coef * eta * (I...I)
        if all(c == "I" for c in chars_block):
            chars_global = ["I"] * n_qubits
            global_acc[tuple(chars_global)] += local_coef * eta
            continue

        # 物理平移复制：sum_{i=0}^{eta-1} h^{(i)}
        for i in range(eta):
            chars_global = ["I"] * n_qubits
            base = i * M
            for k in range(M):
                chars_global[base + k] = chars_block[k]
            
            global_acc[tuple(chars_global)] += local_coef

    # ======================================================================
    # 3. 清理极小项并输出为 Qiskit 格式（小端序对齐）
    # ======================================================================
    final = []
    for pauli_tuple, c in global_acc.items():
        if abs(c) <= tol:
            continue
        pauli_str = "".join(pauli_tuple[::-1])
        final.append((pauli_str, c))

    return SparsePauliOp.from_list(final)

In [ ]:
omega = omega_matr_fwht(points_ext, neighbors_ext, sigma_ext, volumes_ext)
omega = np.array(omega)

# verify_effective_hamiltonian_spectrum(omega, eta=1)

## omega_prime_matr based on FWHT

In [ ]:
def omega_prime_matr(vor, points, neighbors):
    length = len(points)
    num_bits = math.ceil(math.log2(length))
    dim = 1 << num_bits
    #omega = omega_matr(vor, points, neighbors)
    omega = omega_matr_fwht(vor, points, neighbors)
    omega_prime = [[0.0]*dim for _ in range(dim)]

    for m in range(dim):
        for n in range(dim):
            if m==0 and n==0:
                omega_prime[m][n] = 0
            else:
                omega_prime[m][n] = omega[m][n]
            #For the time being, since we are studing single electron system
            #gamma equals to zero, we have omega equals to omega_prime except
            #for the very first entry.
    return omega_prime

## potential_pauli_from_diag

In [ ]:
def potential_pauli_from_diag(V_diag):
    length = len(V_diag)
    num_bits = math.ceil(math.log2(length))
    dim = 1 << num_bits

    # padding
    V_pad = np.zeros(dim)
    V_pad[:length] = V_diag

    pauli_dict = {}

    for i in range(dim):
        coeff = V_pad[i]
        if coeff == 0.0:
            continue

        # 初始只有一个空字符串
        paulis = {'': 1.0}

        # 逐比特扩展
        for k in range(num_bits):
            bit = (i >> k) & 1
            sign = -1.0 if bit == 1 else 1.0

            new_paulis = {}
            for pstr, c in paulis.items():
                # I 分支
                p_I = pstr + 'I'
                new_paulis[p_I] = new_paulis.get(p_I, 0.0) + c * 0.5

                # Z 分支
                p_Z = pstr + 'Z'
                new_paulis[p_Z] = new_paulis.get(p_Z, 0.0) + c * 0.5 * sign

            paulis = new_paulis

        # 累加到全局字典   
        for pstr, c in paulis.items():
            pauli_dict[pstr] = pauli_dict.get(pstr, 0.0) + coeff * c

    labels = list(pauli_dict.keys())   
    coeffs = np.array([pauli_dict[l] for l in labels], dtype=complex)    
    return SparsePauliOp(labels, coeffs)


In [ ]:
# 生成所有 Pauli strings 2026 02 20
def generate_pauli_strings(num_qubits):
    single = ['I', 'X', 'Y', 'Z']
    for prod in itertools.product(single, repeat=num_qubits):
        yield "".join(prod)

# ADAPT-VQE

## 建立算符池

下面的程序进行了巨大改动。本质上的，概念上的改动。现在的路线是H_sparse  →  (x,y) pairs  →  projector_to_pauli_optimal  →  SparsePauliOp  →  CSR
体系是一次量子化（first‑quantized）。Operator Pool 必须来自 real‑space Hamiltonian 的稀疏结构
所有稀疏化、剪枝、局域性约束都必须基于 real‑space 物理
废弃了早期版本通过对 Pauli 字符串中 `'X'`, `'Y'` 关键字进行字面量匹配的低效方案。由于当前体系属于**实空间一阶量子化（Real-Space First-Quantized）**体系，波函数的连通性完全由实空间网格的动能有限差分（Stencil）和 Transcorrelated (TC) 三体协同微跳决定

## Operator pool 构造函数

In [ ]:
# This edition take into account the parity symmetry and mirror image pairs, which can significantly reduce the pool size while maintaining physical fidelity. The next edition will further optimize the algebraic combination of the Pauli strings to achieve an even more compact representation.
# 2026 05 31
# V3.1 robust version:
import numpy as np
from qiskit.quantum_info import SparsePauliOp
from collections import defaultdict

import numpy as np
from qiskit.quantum_info import SparsePauliOp
from collections import defaultdict

def build_real_space_adapt_pool_v3(points, neighbors, tol=1e-12, use_parity=True):
    """
    第一量子化实空间自适应算符池 (V3.2 厄米完全对齐版)：
    - 生成元：A_pq = (|p><q| + |q><p|)  → 严格厄米（Hermitian），系数全实数化。
    - 完美适配：exp(1j * theta * A) 旋转器与 2.0 * inner.imag 梯度引擎。
    - 宇称对称化：一并绑定镜像边 (p_bar, q_bar)，通过 seen_pairs 强行大减半，绝不膨胀。
    """
    D = len(points)
    M = int(np.ceil(np.log2(D)))

    # 1. 预计算所有点的二进制展开
    bits = ((np.arange(D)[:, None] >> np.arange(M)) & 1).astype(np.uint8)

    # 单比特 projector 映射表
    proj_maps = {
        (0, 0): [('I', 0.5),  ('Z', 0.5)],
        (1, 1): [('I', 0.5),  ('Z', -0.5)],
        (0, 1): [('X', 0.5),  ('Y', 0.5j)],
        (1, 0): [('X', 0.5),  ('Y', -0.5j)],
    }

    pool_ops = []
    seen_pairs = set()

    for p in range(D):
        for q in neighbors[p]:
            if q >= D or p == q:
                continue

            # 无向边去重
            pair_key = tuple(sorted((p, q)))
            if pair_key in seen_pairs:
                continue
            seen_pairs.add(pair_key)

            # ===== 1) 构筑纯厄米对称跳跃基底 =====
            transitions = [(p, q, +1.0), (q, p, +1.0)] 

            # ===== 2) 空间反演偶宇称绑定 =====
            if use_parity:
                p_bar = D - 1 - p
                q_bar = D - 1 - q
                if (p_bar != p or q_bar != q) and (q_bar in neighbors[p_bar]):
                    transitions.append((p_bar, q_bar, +1.0))
                    transitions.append((q_bar, p_bar, +1.0))

                    # 🛡️ 战略防御：在这里拦截镜像边，阻止后续大循环无功重复遍历，池规模直降 50%
                    pair_key_bar = tuple(sorted((p_bar, q_bar)))
                    seen_pairs.add(pair_key_bar)

            # ===== 3) LCU 进位链路径生长 =====
            local_acc = defaultdict(complex)

            for src, dst, sign in transitions:
                src_bits = bits[src]
                dst_bits = bits[dst]

                pq_items = [([], 1.0 + 0.0j)]
                for k in range(M):
                    allowed = proj_maps[(int(src_bits[k]), int(dst_bits[k]))]
                    next_items = []
                    for chars, w in pq_items:
                        for p_char, p_w in allowed:
                            next_items.append((chars + [p_char], w * p_w))
                    pq_items = next_items

                    
                for chars, w in pq_items:
                    pauli_str = "".join(chars[::-1])  # Qiskit 小端序
                    local_acc[pauli_str] += sign * w

            # ===== 4) 收尾：根据 Y 算符宇称，无损提取严格实数系数 =====
            final_pauli_list = []
            for pauli_str, coeff in local_acc.items():
                if abs(coeff) < tol:
                    continue
                
                # Qiskit 的 SparsePauliOp 接收实数系数。
                # 厄米矩阵展开中，包含奇数个 Y 的项代数权重必为纯虚数（因 Y 自带 j），
                # 必须提取其 imag 作为实数系数；偶数个 Y 的项权重必为纯实数，提取其 real。
                num_y = pauli_str.count('Y')
                if num_y % 2 == 1:
                    c_real = float(coeff.imag)  # 👈 虚部吐出来，转化为有功实系数
                else:
                    c_real = float(coeff.real)  # 👈 实部正常保留
                    
                if abs(c_real) > tol:
                    final_pauli_list.append((pauli_str, c_real))

            if final_pauli_list:
                pool_ops.append(SparsePauliOp.from_list(final_pauli_list))

    print(f"[ADAPT v3.2] 实空间【纯厄米】自适应算符池构建完成，大小 = {len(pool_ops)}")
    return pool_ops

### Generating Operator Pool

In [ ]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp
from collections import defaultdict
from functools import partial

def build_multi_electron_tiered_pool_v5_11(
    points_3d,          # Shape (D, 3): Continuous coordinates of all D grid points
    neighbors,          # List of lists: natural neighbors for point p
    n_electrons=1,      # Number of electrons
    # weight_func=None,  # Optional function to compute weights for Pauli strings
    tol=1e-12
):
    """
    第一量子化实空间自适应分层算符池 (V5.11 最终稳定版)
    """
    D = len(points_3d)
    M_per_electron = int(np.ceil(np.log2(D)))
    total_qubits = n_electrons * M_per_electron

    # Precompute binary expansion (k=0 → LSB)
    bits = ((np.arange(D)[:, None] >> np.arange(M_per_electron)) & 1).astype(np.uint8)

    proj_maps = {
        (0, 0): [('I',  0.5), ('Z',  0.5)],
        (1, 1): [('I',  0.5), ('Z', -0.5)],
        (0, 1): [('X',  0.5), ('Y',  0.5j)],
        (1, 0): [('X',  0.5), ('Y', -0.5j)],
    }

    pool_wrapped = []
    axis_names = ['X', 'Y', 'Z']

    # ------------------------------------------------------------------
    # Internal helper: expand single-particle Pauli string into the full
    # multi-electron register and accumulate real anti-Hermitian coefficients.
    # ------------------------------------------------------------------
    def _symmetrize_over_electrons(single_particle_accumulator):
        raw_multi_body = defaultdict(complex)

        for pauli_str_1e, coeff in single_particle_accumulator.items():
            if abs(coeff) < tol:
                continue
            for e_idx in range(n_electrons):
                full_qubits = ['I'] * total_qubits
                base_offset = e_idx * M_per_electron
                for k, ch in enumerate(pauli_str_1e):
                    full_qubits[base_offset + k] = ch
                global_pauli_str = "".join(full_qubits[::-1])
                raw_multi_body[global_pauli_str] += coeff

        # ==================================================================
        # 👑 终极修正：利用伴随矩阵宇称提取真正的实系数抗厄米分量
        # ==================================================================
        final_list = []
        for global_pauli_str, final_coeff in raw_multi_body.items():
            num_y = global_pauli_str.count('Y')

            # 在自然的纯实数流动下，由于后端执行 G = 1j * Op
            # 1. 偶数个 Y 串天然分到纯实数系数，外乘 1j 后完美抗厄米 -> 提取实部
            # 2. 奇数个 Y 串天然分到纯虚数系数，外乘 1j 后变纯厄米。要让它在外乘 1j 后抗厄米，
            #    其在算符池里的系数必须提供虚部（即 final_coeff.imag），才能与外部的 1j 互补。
            if num_y % 2 == 1:
                c_real = float(final_coeff.imag)
            else:
                c_real = float(final_coeff.real)

            if abs(c_real) > tol:
                final_list.append((global_pauli_str, c_real))

        if not final_list:
            return None

        op = SparsePauliOp.from_list(final_list).simplify(atol=tol)
        return op if len(op) > 0 else None

    # ==================================================================
    # Tier 0 — Global Physical Generators
    # ==================================================================
    print(f"  -> Constructing Tier 0 over {n_electrons} electron registers...")

    p_acc = {0: defaultdict(complex), 1: defaultdict(complex), 2: defaultdict(complex)}
    d_acc = defaultdict(complex)

    for p in range(D):
        for q in neighbors[p]:
            if q >= D or q <= p:          # deduplicate undirected edges
                continue

            vec  = points_3d[q] - points_3d[p]
            dist = np.linalg.norm(vec)
            if dist < 1e-8:
                continue

            u_vec         = vec / dist                                   # unit direction vector
            midpoint_coord = 0.5 * (points_3d[p] + points_3d[q]) # geometric midpoint

            for xyz_dim in range(3):
                sgn_p = u_vec[xyz_dim] / dist
                sgn_d = (midpoint_coord[xyz_dim] * u_vec[xyz_dim]) / dist

                # Forward hop: |p⟩⟨q|
                for chars, w in _lcu_grow_path(bits[p], bits[q], M_per_electron, proj_maps):
                    pauli_str = "".join(chars)
                    p_acc[xyz_dim][pauli_str] += (+sgn_p) * w
                    d_acc[pauli_str]          += (+sgn_d) * w

                # Reverse hop: |q⟩⟨p|
                for chars, w in _lcu_grow_path(bits[q], bits[p], M_per_electron, proj_maps):
                    pauli_str = "".join(chars)
                    p_acc[xyz_dim][pauli_str] += (-sgn_p) * w
                    d_acc[pauli_str]          += (-sgn_d) * w

    # Symmetrize and register Tier 0 operators
    for xyz_dim in range(3):
        op_p = _symmetrize_over_electrons(p_acc[xyz_dim])
        if op_p is not None:
            pool_wrapped.append({
                'op':   op_p, 'tier': 0,
                'desc': f"Global Translation p_{axis_names[xyz_dim]}"
            })

    op_d = _symmetrize_over_electrons(d_acc)
    if op_d is not None:
        pool_wrapped.append({
            'op':   op_d, 'tier': 0,
            'desc': "Global Symmetrized Midpoint Dilation D"
        })

    # ==================================================================
    # Tier 1 — Anti-Hermitian Phase Gradient Modulators
    # ==================================================================
    print("  -> Constructing Tier 1 Phase Modulators (Sin & Cos Dual-Engine)...")
    frequencies = [1.0, 3.0, 5.0]

    for k in frequencies:
        for xyz_dim in range(3):
            for fn_name, fn in [("sin", np.sin), ("cos", np.cos)]:
                phase_acc = defaultdict(complex)

                for p in range(D):
                    for q in neighbors[p]:
                        if q >= D or q <= p:
                            continue

                        # 🛠️ 彻底清理 1j，恢复天然纯实数差值流动
                        delta_val = (fn(k * points_3d[p, xyz_dim]) - fn(k * points_3d[q, xyz_dim]))
                        if abs(delta_val) < tol:
                            continue

                        # Forward hop
                        for chars, w in _lcu_grow_path(bits[p], bits[q], M_per_electron, proj_maps):
                            phase_acc["".join(chars)] += (+delta_val) * w

                        # Reverse hop
                        for chars, w in _lcu_grow_path(bits[q], bits[p], M_per_electron, proj_maps):
                            phase_acc["".join(chars)] += (-delta_val) * w

                op_phase = _symmetrize_over_electrons(phase_acc)
                if op_phase is not None:
                    pool_wrapped.append({
                        'op':   op_phase, 'tier': 1,
                        'desc': f"Phase Modulator Diff-{fn_name}(k={k}) Axis {axis_names[xyz_dim]}"
                    })

    # ==================================================================
    # Tier 2 — Voronoi Spatial Hops with Local Geometric Weights
    # ==================================================================
    print("  -> Constructing Tier 2 Voronoi Spatial Hops with Geometric Weights...")

    for p in range(D):
        for q in neighbors[p]:
            v_p = volumes_ext[p]
            if v_p < 0 or q <= p:
                continue  # Skip invalid volume points

            for q in neighbors_ext[p]:
                if q >= D or q <= p:
                    continue

                v_q = volumes_ext[q]
                if v_q <= 0 or q <= p:
                    continue  # Skip invalid volume points

                #1 Calculate distance between points p and q
                dis_pq = np.linalg.norm(points_3d[p] - points_3d[q])
                if dis_pq < 1e-8:
                    continue  # Skip if points are too close

                #2 Subtract the information of the sharing area between p and q from the sigma_ext dictionary
                # in a safe way to avoid KeyError when the dictionary does not contain the key (p, q)
                cell_info = sigma_ext[p].get(q, None) if p < len(sigma_ext) else None

                if cell_info is not None and "area" in cell_info:
                    area_pq = cell_info["area"]
                else:
                    # if (p, q) contains no area information, we can set it as zero or just skip this pair
                    # because it means they are not sharing a Voronoi face, so we can skip this pair
                    continue
                # area_pq = sigma_ext[p][q]["area"]

                #3 Compute the geometric weight based on distance and area
                geom_weight = area_pq / (np.sqrt(v_p * v_q) * dis_pq)
                if geom_weight < 1e-5:
                    continue  # Skip if the geometric weight is too small

                local_acc = defaultdict(complex)
                for src, dst, sign in [(p, q, +1.0), (q, p, -1.0)]:
                    for chars, w in _lcu_grow_path(bits[src], bits[dst], M_per_electron, proj_maps):
                        local_acc["".join(chars)] += sign * w * geom_weight

                op_hop = _symmetrize_over_electrons(local_acc)
                if op_hop is not None:
                    pool_wrapped.append({
                        'op':   op_hop, 'tier': 2,
                        'desc': f"Voronoi Hop Grid_{p}<->_{q} (weight={geom_weight:.3e})"
                    })

    tier_counts = {t: sum(1 for e in pool_wrapped if e['tier'] == t) for t in [0, 1, 2]}
    print(f"\n  [V5.11] Pool construction complete.")
    print(f"          Tier 0 (global physical): {tier_counts[0]}")
    print(f"          Tier 1 (phase gradient):  {tier_counts[1]}")
    print(f"          Tier 2 (Voronoi hops):    {tier_counts[2]}")
    print(f"          Total:                    {len(pool_wrapped)}")
    return pool_wrapped


def _lcu_grow_path(src_bits, dst_bits, M, proj_maps):
    pq_items = [([], 1.0 + 0.0j)]
    for k in range(M):
        allowed = proj_maps[(int(src_bits[k]), int(dst_bits[k]))]
        pq_items = [
            (chars + [p_char], w * p_w)
            for chars, w in pq_items
            for p_char, p_w in allowed
        ]
    return pq_items


def validate_pool_v5_11(pool, tol=1e-8):
    print("Running mathematical anti-Hermitian validation (with global phase alignment)...")
    failures = []

    for entry in pool:
        G = entry['op']  # Assumes entry['op'] is already anti-Hermitian
        G_dag = G.adjoint()
        anti_sum = (G + G_dag).simplify(atol=tol)

        if len(anti_sum) > 0:
            norm = sum(abs(c) for _, c in anti_sum.to_list())
            if norm > tol:
                failures.append((entry['desc'], norm))

    if failures:
        print(f"  ✗ {len(failures)} operator(s) failed validation:")
        for desc, norm in failures:
            print(f"      {desc}  →  ||G + G†|| = {norm:.3e}")
    else:
        print(f"  ✓ All {len(pool)} operators form perfect anti-Hermitian evolution generators!")

    return len(failures) == 0

if __name__ == "__main__":
    points_1d = np.array([[-1.5, 0., 0.],
                          [-0.5, 0., 0.],
                          [ 0.5, 0., 0.],
                          [ 1.5, 0., 0.]])
    neighbors_1d = [[1], [0, 2], [1, 3], [2]]

    pool = build_multi_electron_tiered_pool_v5_11(
        points_3d=points_1d,
        neighbors=neighbors_1d,
        n_electrons=1,
    )

    print()
    all_pass = validate_pool_v5_11(pool)

## 计算 ADAPT‑VQE 的第一轮梯度

### 3. 选一个初始态

In [ ]:
# 下面是实空间自适应初始态构建器的第一版，后续版本会加入更多物理启发式设计来提升初始态的质量和收敛效率。
# 2026 05 31
import numpy as np

def build_real_space_initial_state(points, sigma=0.5):
    """
    纯正第一量子化实空间自适应初始态制备器：
    - 彻底告别非物理的高动能 δ-函数灾难。
    - 动态锚定自适应网格中的物理几何核中心（r -> 0）。
    - 采用实空间光滑连续的高斯波包，保障动能、势能景观的初始平滑性。
    """
    D = len(points)
    
    # 1. 动态像素级寻找原子核在自适应网格中的精确投影位置（绝对值最接近原点的点）
    # 在三维径向或一维非均匀坐标下，points 存储的是真实物理位置坐标
    r_squared = np.sum(points**2, axis=1) if len(points.shape) > 1 else points**2
    center_idx = np.argmin(r_squared)
    x_center = points[center_idx]
    
    print(f"[Psi0] 成功定锚原子核几何中心：索引 = {center_idx}, 物理坐标 = {x_center}")

    # 2. 在解析实空间上解析构建高斯连续波包
    # 彻底杜绝全空间随机噪声，改用物理空间的高斯弥散来提供变分自由度
    if len(points.shape) > 1:
        # 三维情况
        distances = np.sqrt(np.sum((points - x_center)**2, axis=1))
    else:
        # 一维情况
        distances = np.abs(points - x_center)
        
    # 计算物理高斯振幅
    psi0 = np.exp(- (distances**2) / (2 * (sigma**2))).astype(complex)

    # 3. 严格执行一阶量子化全空间波函数归一化
    # 注意：在非均匀网格下，严格的物理归一化应该包含体积元（度规权重）积分！
    # 但由于 VQE 线路最终在比特端是状态矢量归一化，我们在此处执行 L2 归一化以对齐 Qiskit
    norm = np.linalg.norm(psi0)
    if norm < 1e-12:
        raise ValueError("初始波包展宽 sigma 过小，导致全空间波函数坍缩，请调大 sigma！")
        
    psi0 /= norm

    # 质检：检查初始态的局域分布连续性
    print(f"[Psi0] 纯正一阶量子化初始高斯态组装完毕。峰值振幅 = {np.max(np.abs(psi0)):.4f}")
    return psi0

In [ ]:
# 通用核心：Pauli 和作用在态上的高速实现
# 这段代码的主要功能是：在不构造稠密矩阵的情况下，用位运算高速计算一个 Pauli 和算符（SparsePauliOp）作用在量子态向量上
import numpy as np
from qiskit.quantum_info import SparsePauliOp

def apply_pauli_sum(op: SparsePauliOp, psi: np.ndarray) -> np.ndarray:
    """
    高速实现：phi = op * psi
    不构造矩阵，逐 Pauli 项用位运算作用在 psi 上并累加。
    """
    D = len(psi)
    M = int(np.log2(D))
    paulis = op.paulis
    coeffs = op.coeffs

    idx_arr = np.arange(D, dtype=np.uint64)
    phi = np.zeros_like(psi, dtype=np.complex128)

    # 预先准备 2^k 权重，用于把布尔掩码压成整数
    powers2 = (1 << np.arange(M, dtype=np.uint64))

    for i in range(len(op)):
        coef = coeffs[i]
        x_mask = paulis.x[i].astype(np.uint64)
        z_mask = paulis.z[i].astype(np.uint64)

        # 快路径：纯 I 项
        if not x_mask.any() and not z_mask.any():
            phi += coef * psi
            continue

        # X/Y 翻转：把 x_mask 压成一个 XOR key
        xor_key = int(x_mask.dot(powers2))

        # Z/Y 相位：把 z_mask 压成一个整数 mask，然后用 bit_count 取奇偶
        z_int = int(z_mask.dot(powers2))
        if z_int == 0:
            phases = 1.0
        else:
            tmp = idx_arr & np.uint64(z_int)
            pop_parity = np.fromiter((int(x).bit_count() & 1 for x in tmp),
                         dtype=np.uint8,
                         count=len(tmp))
            phases = 1.0 - 2.0 * pop_parity  # 0→+1, 1→-1

        # Y 的全局相位：Y = i X Z
        y_mask = np.logical_and(paulis.x[i], paulis.z[i])
        y_count = int(y_mask.sum())
        # (1j)**n 的快速表
        y_phase = (1, 1j, -1, -1j)[y_count % 4]

        psi_flipped = psi[idx_arr ^ np.uint64(xor_key)]
        phi += coef * y_phase * phases * psi_flipped

    return phi


## 真正可扩展的 ADAPT‑VQE 主循环

## Adapt_VQE on TC Matrix
The following code is used to generate "psi, ansatz_ops_indices, thetas" based on the 
guided Hamiltonian

### 实现“精英筛选”逻辑

In [ ]:
# 2026-06-01 15:52
import numpy as np
from qiskit.quantum_info import SparsePauliOp

def apply_pauli_rotation_scaling_squaring_v2(
    op: SparsePauliOp,
    psi: np.ndarray,
    theta: float,
    target_order: int = 2
) -> np.ndarray:
    """
    exp(i θ A) |ψ> 的终极一阶量子化旋转器（Scaling & Squaring 完全体版）
    - A 为 Hermitian（实系数 Pauli 和），统一执行 exp(1j * theta * A) 幺正流
    - 严格遵循泰勒级数级联累乘流动：A_n = A * A_{n-1}
    - 彻底清洗 A² = I 的不切实际假设，直面非对易二进制进位链
    """
    if abs(theta) < 1e-15:
        return psi.copy()

    # ------------------------------------------------------------
    # 1. 自适应缩放指数估计：把有效步长死死锁在极值 0.01 附近
    # ------------------------------------------------------------
    max_coeff = np.max(np.abs(op.coeffs.real))
    effective_angle = abs(theta * max_coeff)

    if effective_angle > 0.01:
        s = int(np.ceil(np.log2(effective_angle / 0.01)))
    else:
        s = 0

    num_steps = 2 ** s
    theta_scaled = float(theta / num_steps)

    # ------------------------------------------------------------
    # 2. 小角度 Taylor 级联累乘步进器 (守护物理拓扑连续性)
    # ------------------------------------------------------------
    def small_step(state: np.ndarray) -> np.ndarray:
        # 第一阶级联：A1 = A * state
        A1 = apply_pauli_sum(op, state)
        term1 = 1j * theta_scaled * A1

        # 第二阶级联：A2 = A * A1  (坚决摒弃 A^2 = I 的欺骗性简写)
        A2 = apply_pauli_sum(op, A1)
        term2 = -0.5 * (theta_scaled ** 2) * A2

        if target_order == 3:
            # 第三阶级联：A3 = A * A2
            A3 = apply_pauli_sum(op, A2)
            term3 = -1j * (theta_scaled ** 3) / 6.0 * A3
            return state + term1 + term2 + term3

        return state + term1 + term2

    # ------------------------------------------------------------
    # 3. Scaling & Squaring 线性时序大推进
    # ------------------------------------------------------------
    phi = psi.copy()
    for _ in range(num_steps):
        phi = small_step(phi)

    # ------------------------------------------------------------
    # 4. 终点幺正投影 (捍卫复数概率流守恒，清洗截断残差)
    # ------------------------------------------------------------
    return phi / np.linalg.norm(phi)

In [ ]:
def apply_pauli_rotation_optimized(
    psi: np.ndarray,
    op: SparsePauliOp,
    theta: float,
    order: int = 2
) -> np.ndarray:
    """
    ADAPT‑VQE 统一入口旋转器：
    - 严格执行 U = exp(i * theta * op) |psi>
    - op 必须是 Hermitian（实系数 Pauli 和）
    - 内部调用 scaling‑squaring 旋转器，避免矩阵膨胀
    """

    # 1. 零角度快速路径
    if abs(theta) < 1e-15:
        return psi.copy()

    # 2. 调用 scaling‑squaring 旋转器（你已经写好的终极版本）
    psi_new = apply_pauli_rotation_scaling_squaring_v2(
        op=op,
        psi=psi,
        theta=theta,
        target_order=order
    )

    # 3. 最终归一化（幺正性保护）
    norm = np.linalg.norm(psi_new)
    if norm > 0:
        psi_new /= norm

    return psi_new


In [ ]:
# adapt_vqe_first_quantized_v4 是在 adapt_vqe_tc_guided 的基础上，针对第一量子化实空间问题的特殊性进行定制化改造的版本。
# 它保留了 adapt_vqe_tc_guided 中的核心调度逻辑和性能加速策略，但针对第一量子化实空间问题的特点，进行了以下关键改进：
# 1. 算符池升级：专门构建了适用于第一量子化实空间问题的符号算符池，确保每个算符都严格满足实空间连续宇称守恒，并且带有进位链结构。
# 2. 初始态升级：提供了连续实空间高斯伪类基态的生成函数，避免了 δ-函数灾难，同时更贴近第一量子化问题的物理特性。
# 3. 梯度计算重构：针对第一量子化实空间问题的特殊性，重构了梯度计算流程，完全摒弃了矩阵化的符号算符表示，改用纯粹的向量化操作和物理引擎级别的 np.bit_count 来实现快速梯度轮询。
# 4. 其他细节调整：根据第一量子化实空间问题的特点，调整了部分参数和控制逻辑，以更好地适应这一类问题的优化需求。
# 2026-06-06 11:24 (已集成动态算符封印惩罚控制)
import random
import gc
import numpy as np
from scipy.optimize import minimize
import scipy.sparse as sp
from qiskit.quantum_info import SparsePauliOp

# ==================================================================
# 🛠️ 纯函数辅助组件：算符动态封印惩罚机制
# ==================================================================
def apply_operator_penalty_scores(scores, adapt_op_history, max_frequency=2):
    """
    根据历史频次，强行将高频算符的梯度得分抹零。
    :param scores: 原始得分列表，元素为 (active_pool_idx, abs_gradient)
    :param adapt_op_history: 平铺的单个算符绝对索引历史列表
    :param max_frequency: 窗口内允许单个算符出现的最高频次，超过则触发抹零
    :return: 惩罚后的得分列表
    """
    if len(adapt_op_history) < 9:  # 历史太短（不足 3 步），不触发封印
        return scores
    
    # 统计当前窗口内每个算符绝对索引出现的频次
    counts = {}
    for op in adapt_op_history:
        counts[op] = counts.get(op, 0) + 1
        
    penalized_scores = []
    for i, grad_abs in scores:
        # 注意：这里的 i 是 active_pool 的内部局域索引，我们需要统计的是绝对算符索引
        # 但在当前版本中 scores 生成时使用的是：(i, np.abs(raw_gradients[i]))
        # 故在主循环中传入此函数前，我们会将其重新映射或直接在外部处理。
        # 为了不破坏后续多区域选拔对局域索引 i 的依赖，我们在这里直接通过全局历史做判定。
        pass
    return scores # 实际处理直接在主循环内展开更不易出错，见下文主循环实现

# ==================================================================
# 🚀 主调度器实现
# ==================================================================
def adapt_vqe_first_quantized_v4(
    H_csr: sp.csr_matrix,          # 继承：总能量哈密顿量完全体 CSR 矩阵 (用于 H_csr @ psi 高速路径)
    pool_ops: list[SparsePauliOp], # 升级：纯正第一量子化、带进位链、守恒偶宇称的符号算符池 (V3版)
    psi_init: np.ndarray,          # 升级：连续实空间高斯伪类基态 (拒绝 δ-函数灾难)
    initial_threshold=500.0,       # 100% 保留你的高级控制参数
    max_steps=2000, 
    block_size=2,
    min_theta_threshold=1e-5,
    offset=0,
    exclude_window=20,
    use_regional_selection=False,
    exp_manager=None,
    max_pool_capacity=1000,
    # 🌟 新增算法控制参数（免定义类，直接作为函数可选入参）
    loop_window_size=15,           # 记忆窗口步数
    loop_max_frequency=2           # 窗口内允许单个算符出现的最大次数
):
    """
    第一量子化实空间自适应 ADAPT-VQE 核心调度器 (V4 终极合龙版 - 动态封印防死锁)
    """
    mode_str = "First-Quantized-RealSpace"
    mode_str += f" (Block Size: {block_size})"
    if use_regional_selection:
        mode_str += " [Regional Selection]"
        
    # 1. 完备断点续算与初态恢复
    ansatz_ops_indices = []
    thetas = []
    start_step = 0
    plateau_count = 0  
    tabu_list_restored = None 
    
    # 初始化平铺的单个算符记忆账本
    adapt_op_history = []
    
    if exp_manager is not None:
        print("Checking for checkpoints...")
        saved_state = exp_manager.load_state(current_pool_raw_size=len(pool_ops))
        if saved_state is not None:
            metadata, coeffs = saved_state
            psi = coeffs.copy()
            ansatz_ops_indices = metadata["ansatz"]["operator_indices"]
            thetas = metadata["ansatz"]["theta_values"]
            start_step = len(ansatz_ops_indices) // block_size
            plateau_count = metadata.get("plateau_count", 0)  
            tabu_list_restored = metadata.get("control_state", {}).get("tabu_list", [])  
            
            # 🌟 从断点恢复历史惩罚账本：提取最近几步的算符，塞满记忆窗口
            recent_ops = ansatz_ops_indices[-loop_window_size * block_size:]
            adapt_op_history = list(recent_ops)
            
            print(f"Restored successfully. Starting from step {start_step}.")
        else:
            psi = np.array(psi_init).copy()
    else:
        psi = np.array(psi_init).copy()
    
    # 2. 初始账本池激活筛选
    active_pool = list(range(len(pool_ops)))
    if len(active_pool) > max_pool_capacity:
        active_pool = active_pool[:max_pool_capacity]

    # 3. Tabu 列表初始化 
    forbidden_fixed = [] 
    if tabu_list_restored is not None:
        tabu_list = tabu_list_restored
    else:
        tabu_list = forbidden_fixed + ansatz_ops_indices[-exclude_window:]
    
    # 高速定锚初始能量
    phi = H_csr @ psi
    current_energy = float(np.real(np.vdot(psi, phi)))
    print(f"[{mode_str}] Starting Phys Energy: {current_energy - offset:.6f}\n")

    # 4. 主循环攻坚战
    for step in range(start_step, max_steps):
        
        # ==================================================================
        # 核心大改 A：无矩阵、向量化、严格反厄米实数快速梯度轮询
        # ==================================================================
        phi_active = H_csr @ psi  
        num_active = len(active_pool)
        raw_gradients = np.zeros(num_active, dtype=np.float64)
        
        for i in range(num_active):
            abs_idx = active_pool[i]
            eta = apply_pauli_sum(pool_ops[abs_idx], psi)
            inner = np.vdot(phi_active, eta)
            raw_gradients[i] = 2.0 * inner.real
            
        # ==================================================================
        # 🚨 动态注入：一键封印死锁循环中的高频高能算符
        # ==================================================================
        # 统计历史窗口里绝对算符索引的曝光次数
        if len(adapt_op_history) >= (3 * block_size):
            op_counts = {}
            for op_abs in adapt_op_history:
                op_counts[op_abs] = op_counts.get(op_abs, 0) + 1
            
            # 检查活跃池，如果哪个算符超频了，现场直接拔掉其梯度电闸
            for i in range(num_active):
                abs_idx = active_pool[i]
                if op_counts.get(abs_idx, 0) >= loop_max_frequency:
                    raw_gradients[i] = 0.0  # 强制梯度归零，使其无法在得分榜出头

        # 组装成后续多区域选拔兼容的得分账本：(当前活跃池索引, 梯度绝对值)
        scores = [(i, np.abs(raw_gradients[i])) for i in range(num_active)]
        
        # ==================================================================
        # 100% 完美还原 B：算符块选拔 (多区域选择与绝对索引映射流)
        # ==================================================================
        chosen_block_indices = []
        if use_regional_selection:
            regions = [(0, 300), (301, 700), (701, len(active_pool)-1)]
            for start, end in regions:
                region_candidates = [s for s in scores if start <= s[0] <= end and active_pool[s[0]] not in tabu_list]
                region_candidates_sorted = sorted(region_candidates, key=lambda x: -x[1])
                if region_candidates_sorted and region_candidates_sorted[0][1] > 0: # 必须大于0，过滤已被封印算符
                    chosen_block_indices.append(active_pool[region_candidates_sorted[0][0]])
            
            if len(chosen_block_indices) < block_size:
                global_sorted = sorted(scores, key=lambda x: -x[1])
                for idx, val in global_sorted:
                    if val == 0.0: break # 如果后续算符得分全是0，说明全池可用算符均被封印，停止挑选
                    abs_idx = active_pool[idx]
                    if abs_idx not in tabu_list and abs_idx not in chosen_block_indices:
                        chosen_block_indices.append(abs_idx)
                    if len(chosen_block_indices) == block_size: break
        else:
            scores_sorted = sorted(scores, key=lambda x: -x[1])
            for idx, val in scores_sorted:
                if val == 0.0: break # 过滤已被抹零的死锁算符
                abs_idx = active_pool[idx]
                if abs_idx not in tabu_list and abs_idx not in chosen_block_indices:
                    chosen_block_indices.append(abs_idx)
                if len(chosen_block_indices) == block_size: break

        if len(chosen_block_indices) == 0:
            print(f"Step {step}: Warning! No valid operators selected (all blocked or penalized). Stopping.")
            break 
        
        # ==================================================================
        # 核心大改 C：联合优化 (BFGS 内部目标函数全面接入一阶量子化高速乘法)
        # ==================================================================
        block_ops = [pool_ops[i] for i in chosen_block_indices]
        initial_thetas = np.zeros(len(chosen_block_indices))
        
        def cost_function(t_vec):
            p_temp = psi.copy()
            for i in range(len(t_vec)):
                p_temp = apply_pauli_rotation_optimized(p_temp, block_ops[i], t_vec[i])
            norm = np.linalg.norm(p_temp)
            if norm > 0: p_temp /= norm
            return float(np.real(np.vdot(p_temp, H_csr @ p_temp)))
            
        res = minimize(cost_function, initial_thetas, method='BFGS', tol=1e-8)
        best_thetas = res.x

        # --- Step D: 正式更新波函数 ---
        for i in range(len(best_thetas)):
            psi = apply_pauli_rotation_optimized(psi, block_ops[i], best_thetas[i])
        psi /= np.linalg.norm(psi)

        # --- Step E: 更新记录与禁手列表 ---
        for idx, t_val in zip(chosen_block_indices, best_thetas):
            ansatz_ops_indices.append(idx) 
            thetas.append(t_val)
            tabu_list.append(idx)
            
            # 🚨 现场推入动态封印历史记录
            adapt_op_history.append(idx)
        
        # 维持滑动窗口的最大容量限制
        max_flat_history_size = loop_window_size * block_size
        if len(adapt_op_history) > max_flat_history_size:
            adapt_op_history = adapt_op_history[block_size:]

        while len(tabu_list) > (len(forbidden_fixed) + exclude_window):
            tabu_list.pop(len(forbidden_fixed))

        # 稀疏高速更新能量
        phi = H_csr @ psi
        new_energy = float(np.real(np.vdot(psi, phi)))
        energy_diff = abs(new_energy - current_energy)
        
        # ==================================================================
        #  核心大改 F：平台期洗牌策略重构 (洗净非物理随机噪声)
        # ==================================================================
        if energy_diff < 1e-5:
            plateau_count += 1
            
            if plateau_count == 4: 
                print(f"\n[能量停滞] 重新激活全池账本刷新与洗牌...")
                tabu_list = []
                # 停滞洗牌时，同步选择清空封印历史，给算法一次重新全面审视局部最优的机会
                adapt_op_history = [] 
                
            elif plateau_count >= 5:
                print(f"\n[连续停滞] 实施实空间平滑高斯流形微扰...")
                psi += 1e-6 * psi_init  
                psi /= np.linalg.norm(psi)
                phi = H_csr @ psi  
                new_energy = float(np.real(np.vdot(psi, phi)))
                plateau_count = 0 
                adapt_op_history = []
        else:
            plateau_count = 0 

        # 打印日志
        avg_theta = np.mean(np.abs(best_thetas))
        print(f"Step {step:2d}: Ops {chosen_block_indices} | Avg Theta: {avg_theta:.2e} | Phys Energy: {new_energy - offset:.6f}")

        # 断点存盘
        if exp_manager is not None:
            exp_manager.save_state(
                energy=new_energy,
                operator_indices=ansatz_ops_indices,
                theta_values=thetas,
                coefficients=psi,
                plateau_count=plateau_count,
                tabu_list=tabu_list
            )
            
        if energy_diff < 1e-10: 
            print("数值不再变化。")
            break
            
        current_energy = new_energy

    return psi, ansatz_ops_indices, thetas

### 关键迭代程序

In [ ]:
import time
import numpy as np
from scipy.optimize import minimize, minimize_scalar
from scipy.sparse.linalg import expm_multiply
from collections import Counter

ban_window = 5  # 禁手窗口大小，防止算符过度重复使用


def adapt_vqe_first_quantized_v5_8_tiered(
    hamiltonian,        # scipy.sparse.csr_matrix
    wrapped_pool,       # list of dicts: {'op': SparsePauliOp, 'tier': int, 'desc': str}
    initial_state,      # np.ndarray, shape (2**n,), complex
    pool_sparse=None,   # 允许直接传入预计算好的稀疏矩阵列表
    max_iterations=50,
    grad_threshold=1e-5,
    max_ops_per_step=1,
    tier_weights=None,
    max_op_reuse=3,
    return_gradients=False,
):
    if tier_weights is None:
        tier_weights = {0: 1.0, 1: 1.0, 2: 1.0}

    print(f"🚀 [ADAPT V5.8] Pool: {len(wrapped_pool)} ops | "
          f"H dim: {hamiltonian.shape[0]}")

    if pool_sparse is None:
        pool_sparse = [1j * item['op'].to_matrix(sparse=True) for item in wrapped_pool]

    # Ensure state is complex128 contiguous array
    psi = np.array(initial_state, dtype=np.complex128).ravel()
    psi /= np.linalg.norm(psi)

    ansatz_indices = []
    ansatz_descs   = []
    ansatz_thetas  = []

    op_use_count = Counter()
    ban_age = Counter()  # 记录每个算符被封禁的轮数

    def _rotate(state_vec, G_sp, theta):
        """exp(theta·G)|ψ⟩ via Krylov — never forms the dense matrix."""
        return expm_multiply(theta * G_sp, state_vec, traceA = abs(theta) * np.abs(G_sp.diagonal()).sum())

    def _energy(indices, theta_vec):
        """Rebuild full ansatz state and return ⟨H⟩ — called by L-BFGS-B."""
        state = np.array(initial_state, dtype=np.complex128).ravel()
        
        # 正序累积演化
        for idx, theta in zip(indices, theta_vec):
            if abs(theta) > 1e-12:
                trace_val = np.abs(pool_sparse[idx].diagonal()).sum()
                state = expm_multiply(theta * pool_sparse[idx], state, traceA=trace_val)
                
        norm_val = np.linalg.norm(state)
        if norm_val > 1e-15:
            state /= norm_val
            
        energy_val = np.vdot(state, hamiltonian.dot(state))
        return float(np.real(energy_val))

    def _evaluate_two_stage(
        current_indices, 
        fixed_thetas, 
        init_theta_val=1e-4,
        is_tier0_1=False, # Consider both tier 0 and tier 1
        maxfun_tier0_1=20,
        timeout_tier0_1=15.0 # Set a timeout for tier 0 operators to prevent excessive time consumption
        ):
        """
        内部辅助函数：对当前 ansatz 最后一个算子执行两阶段（Single-param + Joint L-BFGS-B）优化
        返回: (best_thetas, final_energy, E_stage1, is_improved)
        - Have limited Tier 0 operator evaluation to prevent extreme time-consuming linear search
        """
        E_baseline = _energy(current_indices, fixed_thetas + [0.0])
        
        #--- Set limitation of linear search based on the type of operator ---
        if is_tier0_1:
            # For tier 0 and tier 1 operators
            stage1_maxiter = maxfun_tier0_1 # The max iteration times in the Bounded method
            stage2_maxfun = maxfun_tier0_1 # The max times of energy evaluation in L-BFGS-B 
            stage2_maxiter = maxfun_tier0_1 # limitation of iteration
            stage2_ftol = 1e-4 # Loose the convergent accuracy to avoid excess ending exploration
        else:
            # For Tier 2 operators: Keep the high accuracy search
            stage1_maxiter = 50
            stage2_maxfun = 150
            stage2_maxiter = 150
            stage2_ftol = 1e-8

        # --- Stage 1: 单参数线搜索 ---
        def _energy_single_last(theta_last):
            return _energy(current_indices, fixed_thetas + [theta_last])

        res_stage1 = minimize_scalar(
            _energy_single_last,
            bounds=(-np.pi, np.pi),
            method='bounded',
            options={'xatol': 1e-4, "maxiter": stage1_maxiter},
        )
        
        best_single_theta = res_stage1.x
        E_stage1 = res_stage1.fun

        trial_thetas = fixed_thetas + ([best_single_theta] if E_stage1 < E_baseline else [init_theta_val])

        # --- Stage 2: 全参数联合微调 ---
        # Set time limit
        start_time = time.time()
        def time_guard_callback(x):
            if is_tier0_1 and (time.time() - start_time) > timeout_tier0_1:
                raise StopIteration("Tier 0 or Tier 1 operator optimization exceeded time limit.")
        
        try:

            res_stage2 = minimize(
                lambda th: _energy(current_indices, th),
                x0=np.array(trial_thetas, dtype=np.float64),
                method='L-BFGS-B',
                callback=time_guard_callback,
                options={
                    'gtol': 1e-5, 
                    'ftol': stage2_ftol, 
                    'maxfun': stage2_maxfun, 
                    'maxiter': stage2_maxiter,
                    'maxls': 20 if is_tier0_1 else 30, 
                    'disp': False}
            )

            E_stage2 = res_stage2.fun
            final_thetas = res_stage2.x.tolist()

        except StopIteration as e:
            print(f" ↳ {e} Skipping Stage 2 optimization for this operator.")
            E_stage2 = float('inf')
            final_thetas = trial_thetas  # fallback to trial_thetas if Stage 2 fails
        
        # 判断并选出最佳参数
        if E_stage2 < min(E_stage1, E_baseline):
            return res_stage2.x.tolist(), E_stage2, E_stage1, True
        elif E_stage1 < E_baseline:
            out_thetas = fixed_thetas + [best_single_theta]
            return out_thetas, E_stage1, E_stage1, True
        else:
            out_thetas = fixed_thetas + [0.0]
            return out_thetas, E_baseline, E_stage1, False

    E0 = float(np.real(np.vdot(psi, hamiltonian.dot(psi))))
    print(f"  Starting energy: {E0:.8f} Hartree\n")

    for cycle in range(max_iterations):
        grad_record = []
        H_psi = hamiltonian.dot(psi)
        scores = []
        blocked_count = 0
        
        # ── 1. Calculate Gradients & Handle Ban Window ──
        for idx, (item, G_sp) in enumerate(zip(wrapped_pool, pool_sparse)):

            # 检查算子是否处于封禁状态
            if op_use_count[idx] >= max_op_reuse:
                ban_age[idx] += 1
                if ban_age[idx] >= ban_window:
                    op_use_count[idx] = 0
                    ban_age[idx] = 0
                else:
                    blocked_count += 1
                    continue

            G_psi  = G_sp.dot(psi)
            grad_k = abs(2.0 * np.real(np.vdot(H_psi, G_psi)))
            score  = grad_k * tier_weights.get(item['tier'], 1.0)
            scores.append((idx, grad_k, score, item))
            grad_record.append({
                'tier': item['tier'], 
                'grad': grad_k, 
                'index': idx,
            })

        if blocked_count > 0 and cycle == 0:
            print(f"  ⚠️ {blocked_count} operators blocked (max_op_reuse={max_op_reuse}).")

 # ── 2. Filter by threshold & Select ──
        candidates = [
            (idx, g, s, it) for idx, g, s, it in scores 
            if g >= grad_threshold
        ]
        
        if not candidates:
            print(f"🎉 [CONVERGED] Cycle {cycle:02d}: "
                  f"All active operator gradients < {grad_threshold:.2e}.")
            break

        # 按加权 Score 降序，选出本轮最佳算符
        candidates.sort(key=lambda x: -x[2])
        selected = candidates[:max_ops_per_step]

        # ── 3. Two-Stage Optimization with Monotonicity Guard ──
        fixed_thetas = list(ansatz_thetas)

        for idx, grad, score, item in selected:
            ansatz_indices.append(idx)
            current_tier = item['tier']

            # give the boolean variable the initial value
            is_tier0_1 = current_tier in [0, 1]

            print(f" Cycle {cycle:02d} | Tier {current_tier} | "
                  f"Grad: {grad:.6f} | Score: {score:.6f} | "
                  f"[used {op_use_count[idx] + 1}x] | {item['desc']}")

            # 执行两阶段优化 (Stage 1 单参数搜索 + Stage 2 全参数微调)
            opt_thetas, final_E, E_s1, is_improved = _evaluate_two_stage(
                ansatz_indices, 
                fixed_thetas, 
                init_theta_val=1e-4,
                is_tier0_1=is_tier0_1,
                maxfun_tier0_1=8,
                timeout_tier0_1=15.0
            )

            # 严格单调性防护：能量下降则保留，不下降则直接跳过（重置为 0.0）
            
            if is_improved:
                ansatz_thetas = opt_thetas
                op_use_count[idx] += 1
            else:
                #1. Recover the last added operator's index and remove it from ansatz_indices
                ansatz_indices.pop()
                ansatz_thetas = fixed_thetas

                #2 . Reset the operator's usage count to avoid immediate re-selection
                op_use_count[idx] = max_op_reuse  # Reset to max to enforce ban window
                ban_age[idx] = 0  # Reset ban age to start counting again
                print(f" ⚠️ [Stage Guard] Operator failed to lower energy. "
                      f"Skipped and initialized theta to 0.0.")

        # ── 4. Rebuild current state from optimized θ ──
        psi = np.array(initial_state, dtype=np.complex128).ravel()
        psi /= np.linalg.norm(psi)

        for idx, theta in zip(ansatz_indices, ansatz_thetas):
            if abs(theta) > 1e-12:
                psi = _rotate(psi, pool_sparse[idx], theta)

        norm_psi = np.linalg.norm(psi)
        if norm_psi > 1e-12:
            psi /= norm_psi

        E_now = float(np.real(np.vdot(psi, hamiltonian.dot(psi))))
        print(f"    ↳ Energy: {E_now:.8f} Hartree\n")

    print(f"🏁 [DONE] Ansatz depth: {len(ansatz_indices)} operators")
    if return_gradients:
        return psi, ansatz_indices, ansatz_thetas, grad_record
    else:
        return psi, ansatz_indices, ansatz_thetas

###  Try different sigmas

### Build Pool and Test

In [ ]:
import scipy.sparse as sp
import numpy as np

def safe_sparse_pauli_to_csr(op) -> sp.csr_matrix:
    """
    【全局变量适配 & 强容错版】
    - 直接复用全局变量 num_qubits 和 D (2**num_qubits)
    - 自动识别输入类型，如果是 ndarray 或 已经是稀疏矩阵，直接安全返回，绝不报错。
    """
    # 🌟 核心防错：如果传入的已经是 SciPy 稀疏矩阵，直接返回
    if sp.issparse(op):
        return op
        
    # 🌟 核心防错：如果传入的是 NumPy 密集数组，直接转为 CSR 稀疏矩阵返回
    if isinstance(op, np.ndarray):
        return sp.csr_matrix(op)
        
    # 🌟 核心防错：如果传入的是 None，直接返回 None
    if op is None:
        return None

    # ---- 走到这里，说明传入的是标准的 Qiskit SparsePauliOp，开始高效率转换 ----
    dim = 1 << num_qubits  # 🚀 直接使用你的全局变量 num_qubits
    
    try:
        from qiskit.quantum_info.operators.symplectic.sparse_pauli_op import _to_sparse_matrix
        return _to_sparse_matrix(op).tocsr()
    except Exception:
        pass

    x_masks = op.paulis.x
    z_masks = op.paulis.z
    coeffs = op.coeffs
    num_terms = len(op)
    
    # 使用全局变量 num_qubits 压缩掩码
    x_ints = np.zeros(num_terms, dtype=np.int32)
    z_ints = np.zeros(num_terms, dtype=np.int32)
    for q in range(num_qubits): # 🚀 直接使用全局变量
        x_ints |= (x_masks[:, q].astype(np.int32) << q)
        z_ints |= (z_masks[:, q].astype(np.int32) << q)
        
    y_counts = np.sum(np.logical_and(x_masks, z_masks), axis=1)
    total_csr = sp.csr_matrix((dim, dim), dtype=complex)
    basis_indices = np.arange(dim, dtype=np.int32)
    
    chunk_size = 5000
    for chunk_start in range(0, num_terms, chunk_size):
        chunk_end = min(chunk_start + chunk_size, num_terms)
        current_chunk_len = chunk_end - chunk_start
        
        sub_size = current_chunk_len * dim
        chunk_rows = np.zeros(sub_size, dtype=np.int32)
        chunk_cols = np.zeros(sub_size, dtype=np.int32)
        chunk_data = np.zeros(sub_size, dtype=complex)
        
        idx_offset = 0
        for i in range(chunk_start, chunk_end):
            coeff = coeffs[i]
            if coeff == 0: continue
                
            x_mask = x_ints[i]
            z_mask = z_ints[i]
            col_indices = basis_indices ^ x_mask
            
            v = basis_indices & z_mask
            v ^= v >> 1
            v ^= v >> 2
            v ^= v >> 4
            v ^= v >> 8
            phase_z = np.where((v & 1) == 1, -1.0, 1.0)
            
            phase_y = (1j) ** y_counts[i]
            val_vector = coeff * phase_z * phase_y
            
            chunk_rows[idx_offset : idx_offset + dim] = basis_indices
            chunk_cols[idx_offset : idx_offset + dim] = col_indices
            chunk_data[idx_offset : idx_offset + dim] = val_vector
            idx_offset += dim
            
        if idx_offset < sub_size:
            chunk_rows = chunk_rows[:idx_offset]
            chunk_cols = chunk_cols[:idx_offset]
            chunk_data = chunk_data[:idx_offset]
            
        chunk_csr = sp.coo_matrix((chunk_data, (chunk_rows, chunk_cols)), shape=(dim, dim)).tocsr()
        total_csr += chunk_csr
        
    return total_csr

## 构建 $H_{guide}$

In [ ]:
def build_hydrogen_tc_guide(H_bare, r_array, num_leb=50, alpha=1.0):
    """
    H_bare: 原始 1024x1024 矩阵
    r_array: becke_radial_grid 产生的径向点数组 (len=20)
    num_leb: Lebedev 点数 (50)
    修复版本：支持 SparsePauliOp 输入并进行矩阵级相似变换
    
    """

    # 1. 转换矩阵
    if hasattr(H_bare, "to_matrix"):
        H_mat = H_bare.to_matrix()
    else:
        H_mat = H_bare
    
    # 获取哈密顿量的实际维度 (应该是 1024)
    dim = H_mat.shape[0]

    # 2. 构造物理部分的 tau
    tau_list = []
    for ri in r_array:
        t_val = -(1 - np.exp(-alpha * ri)) / alpha
        tau_list.extend([t_val] * num_leb)
    
    # 3. 【关键：对齐维度】补齐剩下的 24 个点
    current_len = len(tau_list)
    if current_len < dim:
        padding_size = dim - current_len
        # 补 0 表示在这些维度上不进行相似变换修正
        tau_list.extend([0.0] * padding_size)
    
    tau_vec = np.array(tau_list)
    
    # 4. 构造变换因子 (dim, dim)
    # tau_diff_ij = tau_j - tau_i
    tau_diff = tau_vec[np.newaxis, :] - tau_vec[:, np.newaxis]
    factor_mat = np.exp(tau_diff)
    
    # 5. 执行点对点乘法
    H_tilde = H_mat * factor_mat
    
    # 6. 厄米化
    H_guide = (H_tilde + H_tilde.conj().T) / 2.0
    
    return H_guide

### 构造一个“多尺度 Gaussian 态”

In [ ]:
# H_pauli = build_circuit_hamiltonian_eq22(omega)

In [ ]:
# H_csr = safe_sparse_pauli_to_csr(H_pauli)
H_sparse_ext = hamilton_14(points_ext, neighbors_ext, sigma_ext, volumes_ext)
H_csr = H_sparse_ext

In [ ]:
pool_ops = build_real_space_adapt_pool_v3(points_ext, neighbors_ext)
print("Pool size =", len(pool_ops))

In [ ]:
A = pool_ops[0]
# M = 10
psi = np.random.randn(2**M) + 1j*np.random.randn(2**M)
psi /= np.linalg.norm(psi)

v1 = A.to_matrix() @ psi
v2 = apply_pauli_sum(A, psi)
print("‖Δ‖ =", np.linalg.norm(v1 - v2))



In [ ]:
# ===== 4. 初始高斯态 =====

# from your_module import build_real_space_initial_state
M = 10
psi0 = build_real_space_initial_state(points, sigma=0.8)
print("‖psi0‖ =", np.linalg.norm(psi0))
psi0_ext = np.zeros(2**M)
psi0_ext[:900] = psi0


In [ ]:
# # ===== 5. 运行 ADAPT‑VQE v4 =====

# # 确保你已经在同一命名空间里有：
# # - apply_pauli_sum
# # - apply_pauli_rotation_scaling_squaring_v2
# # - apply_pauli_rotation_optimized
# # - adapt_vqe_first_quantized_v4
# print("H_csr.shape =", H_csr.shape)
# print("psi0.shape  =", psi0.shape)
# print("len(points) =", len(points))
# print("M           =", M, "→ 2**M =", 2**M)
# D_phys = len(points)      # 252
# M = 9
# D = 2**M                  # 512

# if D_phys < D:
#     pad = D - D_phys
#     # 用最后一个点复制填充（比全零更物理）
#     pad_points = np.repeat(points[-1][None, :], pad, axis=0)
#     points_ext = np.vstack([points, pad_points])
# else:
#     points_ext = points
# print("H_csr.shape =", H_csr.shape)
# print("psi0.shape  =", psi0.shape)
# print("len(points) =", len(points))
# print("M           =", M, "→ 2**M =", 2**M)
# psi_final, op_indices, thetas = adapt_vqe_first_quantized_v4(
#     H_csr=H_csr,
#     pool_ops=pool_ops,
#     psi_init=psi0_ext,
#     max_steps=50,          # 先来个小步数试跑
#     block_size=2,
#     exclude_window=20,
#     use_regional_selection=False,
#     exp_manager=None,      # 先不启用断点系统
#     max_pool_capacity=500  # M=8 时 500 足够
# )

# print("\n=== ADAPT 结束 ===")
# print("Ansatz 长度:", len(op_indices))
# print("最终能量:", float(np.real(np.vdot(psi_final, H_csr @ psi_final))))


In [ ]:
# # ===== 🚀 代数大破冰：5. 运行 ADAPT‑VQE v4 =====

# H_csr_ext = hamilton_14(points_ext, neighbors_ext, sigma_ext, volumes_ext)

# # 1. 找到空间几何中心
# center = np.mean(points_ext, axis=0)
# r2 = np.sum((points_ext - center)**2, axis=1)

# # 2. 连续高斯基底
# alpha = 1.5
# psi0_base = np.exp(-alpha * r2)

# # 3. ⚡⚡ 注入灵魂：挂上复数动量相位，强行打破纯实数死锁 ⚡⚡
# # 这里的 points_ext[:, 0] 代表格点在 X 轴的物理坐标，乘以一个非零的初始动量 k (比如 2.0)
# k_momentum = 2.0
# quantum_phase = np.exp(1j * k_momentum * points_ext[:, 0])

# # 复合成为严格复数流形的动量波包
# psi0_ext = (psi0_base * quantum_phase).astype(complex)

# # 归一化
# psi0_ext /= np.linalg.norm(psi0_ext)

# # 打印初始能量（此时虚部完全激活，动能被彻底唤醒！）
# starting_energy = float(np.real(np.vdot(psi0_ext, H_csr_ext @ psi0_ext)))
# print("=== 🌊 奔跑的复动量高斯波包注入 🌊 ===")
# print("ADAPT 变分起点能量 (复数流形全激活):", starting_energy) 
# # print("目标光谱最低能级:", -0.577358)
# print("=====================================\n")

# # np.save("best_April28_psi.npy", psi_final)
# psi_seed = np.load("best_April28_psi.npy")
# psi_seed /= np.linalg.norm(psi_seed)
# psi_init = psi_seed
# psi0_ext = psi_init
# # 4. 轰鸣变分主循环
# psi_final, op_indices, thetas = adapt_vqe_first_quantized_v4(
#     H_csr=H_csr_ext,          
#     pool_ops=pool_ops,        
#     psi_init=psi0_ext,        # 👈 灌入这个具备复数虚部的有功初态
#     max_steps=200,             
#     block_size=2,
#     exclude_window=20,
#     use_regional_selection=False,
#     exp_manager=None,         
#     max_pool_capacity=500     
# )

# print("\n=== ADAPT 结束 ===")
# print("Ansatz 长度:", len(op_indices))
# print("最终变分能量:", float(np.real(np.vdot(psi_final, H_csr_ext @ psi_final))))

In [ ]:
# -------------------------------------------------------------
# 🛠️ 纯函数版：全域破局算符生成器
# -------------------------------------------------------------
def generate_scaling_and_gauge_operators(num_qubits):
    """
    在一阶实空间离散泡利表示下，直接合成尺度缩放算符与全局规范算符
    为了保持物理严谨性与实空间偶宇称，我们构造其对应的映射符号表示
    """
    import numpy as np
    from qiskit.quantum_info import SparsePauliOp
    
    additional_ops = []
    
    # 1. 尺度缩放算符（Dilatation Operator）: r * d/dr + d/dr * r
    # 在量子力学中它由 r 和 p 的反对易组合构成。在泡利基底中，它通常表现为
    # 跨越多个相近 Qubit 的梯级长程作用（比如多个两体/三体 Pauli 项的线性组合）。
    # 这里我们构造一个全域梯级膨胀算符：由各阶位的阶梯算符对折构成
    scaling_pauli_list = []
    for q in range(num_qubits - 1):
        # 构造类似 (X_q X_{q+1} + Y_q Y_{q+1}) 与位置权重相乘的形式
        # 这在实空间映射中对应网格点之间的流动，带有尺度缩放的物理拓扑性
        weight = float(q + 1)
        scaling_pauli_list.append((f"{'I'*q}XY{'I'*(num_qubits-2-q)}", weight))
        scaling_pauli_list.append((f"{'I'*q}YX{'I'*(num_qubits-2-q)}", -weight))
        
    if scaling_pauli_list:
        scaling_op = SparsePauliOp.from_list(scaling_pauli_list)
        additional_ops.append(scaling_op)
        
    # 2. 全局可变通量/规范变换算符（Global Gauge Transformation）: f(r)
    # 纯空间位置相关的非局域相位算符，在实空间对角。可以用全域的 Z 算符线性组合来表达
    gauge_pauli_list = []
    for q in range(num_qubits):
        # 使用二次方或余弦波形式的位置权重，模拟空间不均匀相位注入
        weight_quad = float((q + 1) ** 2)
        gauge_pauli_list.append((f"{'I'*q}Z{'I'*(num_qubits-1-q)}", weight_quad))
        
        if q < num_qubits - 1:
            gauge_pauli_list.append((f"{'I'*q}ZZ{'I'*(num_qubits-2-q)}", weight_quad * 0.5))
            
    if gauge_pauli_list:
        gauge_op = SparsePauliOp.from_list(gauge_pauli_list)
        additional_ops.append(gauge_op)
        
    return additional_ops

In [ ]:
# ===== 🚀 全新策略：冷高斯微扰破冰起点 =====

H_csr_ext = hamilton_14(points_ext, neighbors_ext, sigma_ext, volumes_ext)

# 1. 构筑全空间弥漫高斯基底
center = np.mean(points_ext, axis=0)
r2 = np.sum((points_ext - center)**2, axis=1)

# 增大宽度 alpha，或者降低 alpha 让波包弥漫得更宽（比如 0.2），从而逼近氢原子的基态指数衰减
alpha = 0.2  
psi0_base = np.exp(-alpha * r2)
np.save("best_April28_psi.npy", psi_final)
psi_seed = np.load("best_April28_psi.npy")
psi_seed /= np.linalg.norm(psi_seed)
psi_init = psi_seed
psi0_base = psi_init
# 2. ⚡⚡ 核心破冰：注入不自带动能的微弱复数随机扰动 ⚡⚡
# 这样处处有微弱虚部，梯度瞬间激活，但宏观动能严格为 0！
np.random.seed(42) # 锁定随机种子确保可重复
weak_phase_noise = 1e-3 * np.random.randn(2**M)
psi0_ext = psi0_base * np.exp(1j * weak_phase_noise)
psi0_ext = psi0_base

# 归一化
psi0_ext /= np.linalg.norm(psi0_ext)

# 打印初始能量
starting_energy = float(np.real(np.vdot(psi0_ext, H_csr_ext @ psi0_ext)))
print("=== 🌟 静止冷高斯弥漫波包注入 🌟 ===")
print("ADAPT 变分起点能量 (动能彻底抚平):", starting_energy) # 预测应该在个位数甚至负数！
# print("目标光谱最低能级:", -0.577358)
print("===================================\n")

breakout_ops = generate_scaling_and_gauge_operators(num_qubits)

# 2. 强行注入全池（可以放在池的最前端，让活跃池优先轮询）
extended_pool_ops = breakout_ops + pool_ops
# 3. 重新轰鸣变分主循环
psi_final, op_indices, thetas = adapt_vqe_first_quantized_v4(
    H_csr=H_csr_ext,          
    # pool_ops=pool_ops,   
    pool_ops=extended_pool_ops,  # 🚀 使用扩展池，包含破冰算符     
    psi_init=psi0_ext,        # 👈 灌入冷高斯初态
    max_steps=300,            
    block_size=2,
    exclude_window=20,
    use_regional_selection=False,
    exp_manager=None,         
    max_pool_capacity=500     
)

In [ ]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp
from collections import defaultdict

def generate_multi_electron_breakout_operators(
    points_3d,          # Shape (D, 3): 离散格点三维坐标
    neighbors,          # 邻居表
    n_electrons=2,      # 电子数
    tol=1e-12
):
    """
    将单电子 Scaling & Gauge 破冰算符升级为多电子全同粒子抗厄米算符。
    统一输出纯实数系数的 Pauli 表达，无缝对接一阶量子化后端。
    """
    D = len(points_3d)
    M_per_electron = int(np.ceil(np.log2(D)))
    total_qubits = n_electrons * M_per_electron

    # 预计算格点位图
    bits = ((np.arange(D)[:, None] >> np.arange(M_per_electron)) & 1).astype(np.uint8)

    proj_maps = {
        (0, 0): [('I',  0.5), ('Z',  0.5)],
        (1, 1): [('I',  0.5), ('Z', -0.5)],
        (0, 1): [('X',  0.5), ('Y',  0.5j)],
        (1, 0): [('X',  0.5), ('Y', -0.5j)],
    }

    breakout_pool = []
    axis_names = ['X', 'Y', 'Z']

    # ======================================================================
    # 多体严格反厄米实数系数投影器
    # ======================================================================
    def _symmetrize_breakout(single_particle_accumulator):
        raw_multi_body = defaultdict(complex)
        for pauli_str_1e, coeff in single_particle_accumulator.items():
            if abs(coeff) < tol:
                continue
            for e_idx in range(n_electrons):
                full_qubits = ['I'] * total_qubits
                base_offset = e_idx * M_per_electron
                for k, ch in enumerate(pauli_str_1e):
                    full_qubits[base_offset + k] = ch
                global_pauli_str = "".join(full_qubits[::-1])
                raw_multi_body[global_pauli_str] += coeff

        final_list = []
        for global_pauli_str, final_coeff in raw_multi_body.items():
            num_y = global_pauli_str.count('Y')
            
            # 依据自然的纯实数流动输入：
            # 1. 奇数个 Y 的 Pauli 串天然分到纯虚数系数。为了让它最终成为反厄米项，
            #    对应的实系数算符池系数应该取其虚部： float(final_coeff.imag)
            # 2. 偶数个 Y 的 Pauli 串天然分到纯实数系数。为了让它最终成为反厄米项，
            #    对应的实系数算符池系数应该取其实部： float(final_coeff.real)
            # 注：这里提取的都是纯实数 float，后端引擎演化时统一 1j * Op 即可完美转为抗厄米。
            if num_y % 2 == 1:
                c_real = -float(final_coeff.imag)
            else:
                c_real = float(final_coeff.real)

            if abs(c_real) > tol:
                final_list.append((global_pauli_str, c_real))

        if not final_list:
            return None
        op = SparsePauliOp.from_list(final_list).simplify(atol=tol)
        return op if len(op) > 0 else None

    def _lcu_path(src, dst):
        pq_items = [([], 1.0 + 0.0j)]
        for k in range(M_per_electron):
            allowed = proj_maps[(int(bits[src, k]), int(bits[dst, k]))]
            pq_items = [(chars + [p_char], w * p_w) for chars, w in pq_items for p_char, p_w in allowed]
        return pq_items

    # ======================================================================
    # 构建 1: 多体全局标度缩放算符 (Scaling S) -> 纯实数天然流动
    # ======================================================================
    print(f"  -> Engineering Multi-Electron Global Scaling Operators...")
    for xyz_dim in range(3):
        scale_acc = defaultdict(complex)
        for p in range(D):
            for q in neighbors[p]:
                if q >= D or q <= p:
                    continue
                vec = points_3d[q] - points_3d[p]
                dist = np.linalg.norm(vec)
                if dist < 1e-8:
                    continue
                
                coord_weight = 0.5 * (points_3d[p, xyz_dim] + points_3d[q, xyz_dim])
                u_vec = vec / dist
                sgn = (coord_weight * u_vec[xyz_dim]) / dist

                for chars, w in _lcu_path(p, q):
                    scale_acc["".join(chars)] += (+sgn) * w
                for chars, w in _lcu_path(q, p):
                    scale_acc["".join(chars)] += (-sgn) * w

        op_s = _symmetrize_breakout(scale_acc)
        if op_s is not None:
            breakout_pool.append({
                'op': op_s, 'tier': 0, 
                'desc': f"Breakout Multi-Scaling S_{axis_names[xyz_dim]}"
            })

    # ======================================================================
    # 构建 2: 多体全局交叉规范算符 (Cross-Gauge M) -> 纯实数天然流动
    # ======================================================================
    print(f"  -> Engineering Multi-Electron Cross-Gauge Rotators...")
    gauge_pairs = [(0, 1), (1, 2), (2, 0)] 
    for dim1, dim2 in gauge_pairs:
        gauge_acc = defaultdict(complex)
        for p in range(D):
            for q in neighbors[p]:
                if q >= D or q <= p:
                    continue
                vec = points_3d[q] - points_3d[p]
                dist = np.linalg.norm(vec)
                if dist < 1e-8:
                    continue
                
                u_vec = vec / dist
                coord_weight_1 = 0.5 * (points_3d[p, dim1] + points_3d[q, dim1])
                sgn = (coord_weight_1 * u_vec[dim2]) / dist

                for chars, w in _lcu_path(p, q):
                    gauge_acc["".join(chars)] += (+sgn) * w
                for chars, w in _lcu_path(q, p):
                    gauge_acc["".join(chars)] += (-sgn) * w

        op_m = _symmetrize_breakout(gauge_acc)
        if op_m is not None:
            breakout_pool.append({
                'op': op_m, 'tier': 0, 
                'desc': f"Breakout Cross-Gauge M_{axis_names[dim1]}{axis_names[dim2]}"
            })

    return breakout_pool

### For updated operator pool and ADAPT-VQE codes

In [ ]:
# 1. 生成标准的 V5.11 物理算符池
base_pool = build_multi_electron_tiered_pool_v5_11(points_1d, neighbors_1d, n_electrons=1)

# 2. 生成多电子破冰先锋算符
breakout_pool = generate_multi_electron_breakout_operators(points_1d, neighbors_1d, n_electrons=1)

# 3. 强行合体：将破冰算符塞在池的最前端，强迫 V5.8 分层求解器在第 1 步优先轮询它们
extended_pool = breakout_pool + base_pool

print(f"\n🚀 破冰池注入成功！当前总池容量: {len(extended_pool)} 个严格抗厄米算符。")

In [ ]:
def local_validate_pool_v5_11(pool, tol=1e-6):
    """
    Matrix-level anti-Hermitian validation with relative error check.
    More robust than Pauli-level check for large-coefficient operators.
    """
    print("Running matrix-level anti-Hermitian validation...")
    failures = []
    for entry in pool:
        G_mat = 1j * entry['op'].to_matrix()
        abs_err = np.linalg.norm(G_mat + G_mat.conj().T)
        op_norm = np.linalg.norm(G_mat)
        rel_err = abs_err / op_norm if op_norm > 1e-15 else abs_err

        if rel_err > tol:
            failures.append((entry['desc'], rel_err, abs_err))

    if failures:
        print(f"  ✗ {len(failures)} operator(s) failed (relative error > {tol}):")
        for desc, rel_err, abs_err in failures:
            print(f"    {desc[:50]:50s} rel={rel_err:.2e}  abs={abs_err:.2e}")
    else:
        print(f"  ✓ All {len(pool)} operators anti-Hermitian "
              f"(relative ||G+G†||/||G|| < {tol})")
    return len(failures) == 0

In [ ]:
import os
import pickle

POOL_CACHE_FILE = "operator_pool_input_v5_11.pkl"

def load_or_build_pool_input(points_3d, neighbors, n_electrons=1, force_rebuild=False):
    """
    自动缓存最终算符池 pool_input = breakout_pool + base_pool
    - 如果缓存存在 → 直接读取（0.1 秒）
    - 如果不存在 → 构造并保存（6 分钟）
    - force_rebuild=True → 强制重建
    """

    # ------------------------------------------------------------
    # 1. 如果已有缓存且不强制重建 → 直接读取
    # ------------------------------------------------------------
    if (not force_rebuild) and os.path.exists(POOL_CACHE_FILE):
        print(f"📦 Loading cached operator pool_input from {POOL_CACHE_FILE} ...")
        with open(POOL_CACHE_FILE, "rb") as f:
            pool_input = pickle.load(f)
        print(f"✓ Loaded {len(pool_input)} operators from cache.\n")
        return pool_input

    # ------------------------------------------------------------
    # 2. 否则构造 base_pool + breakout_pool
    # ------------------------------------------------------------
    print("🛠️ Cache not found — building operator pool_input V5.11 ...")

    base_pool = build_multi_electron_tiered_pool_v5_11(
        points_3d=points_3d,
        neighbors=neighbors,
        n_electrons=n_electrons
    )

    breakout_pool = generate_multi_electron_breakout_operators(
        points_3d=points_3d,
        neighbors=neighbors,
        n_electrons=n_electrons
    )

    pool_input = breakout_pool + base_pool

    # ------------------------------------------------------------
    # 3. 保存缓存
    # ------------------------------------------------------------
    print(f"💾 Saving operator pool_input to {POOL_CACHE_FILE} ...")
    with open(POOL_CACHE_FILE, "wb") as f:
        pickle.dump(pool_input, f)
    print("✓ Cache saved.\n")

    return pool_input


In [ ]:
import os

cache_filename = "operator_pool_input_v5_11.pkl"

# 打印当前脚本真正工作的位置
print("Current Working Directory:", os.getcwd())
print("Full cache path:", os.path.abspath(cache_filename))

# # 如果存在，强行删除
# if os.path.exists(cache_filename):
#     os.remove(cache_filename)
#     print(f"🗑️ Successfully deleted stale cache: {cache_filename}")

### Laplacian Smoothing / Heat Diffusion

In [ ]:
from scipy.sparse.linalg import eigsh

def smooth_psi_laplacian(psi_grid, T_matrix, tau_factor=1e-3):
    """
    利用动能矩阵 T 进行隐式平滑 (虚时间微步演化)
    T_matrix: 实空间网格上的动能稀疏/密集矩阵
    tau: 扩散强度 (通常取 1e-4 ~ 1e-3)
    """
    lambda_max = eigsh(T_matrix, k=1, which="LM", return_eigenvectors=False)[0]
    tau_safe = tau_factor / lambda_max
    # 1. 施加高频抑制
    psi_smoothed = psi_grid - tau_safe * (T_matrix @ psi_grid)
    
    # 2. 重新归一化以保持波函数物理意义
    psi_smoothed /= np.linalg.norm(psi_smoothed)
    return psi_smoothed

In [ ]:
L_sym = laplacian(points_ext, neighbors_ext, sigma_ext, volumes_ext)


In [ ]:
import numpy as np

def build_hydrogen_voronoi_initial_state(points, volumes=None, Z=1.0):
    """
    针对 Voronoi 实空间网格构建最合理的氢原子初态波函数 (Slater 物理启发型)
    
    参数:
        points: np.ndarray, Shape (N, 3)，Voronoi 胞腔的物理坐标
        volumes: np.ndarray, Shape (N,), 每个 Voronoi 胞腔的物理体积(度规权重)
                 若为 None，则退化为标准 L2 归一化
        Z: float, 核电荷数 (氢原子为 1.0)
    """
    # 1. 寻找核的几何中心（距离原点最近的 Voronoi 胞腔）
    r2 = np.sum(points**2, axis=1) if len(points.shape) > 1 else points**2
    nuc_idx = np.argmin(r2)
    x_center = points[nuc_idx]
    
    # 2. 计算各格点到核中心的物理欧氏距离 r
    distances = np.linalg.norm(points - x_center, axis=1) if len(points.shape) > 1 else np.abs(points - x_center)
    
    # 3. 解析构建 Slater 1s 轨道包络: exp(-Z * r)
    # 这在实空间 Voronoi 网格上具有正确的原点 Cusp 行为与远场指数衰减
    psi0 = np.exp(-Z * distances).astype(np.complex128)
    
    # 4. 执行严格的 Voronoi 归一化 <psi0|psi0> = 1
    if volumes is not None:
        # 物理连续空间归一化: Integral |psi|^2 dV = Sum_i |psi_i|^2 * Vol_i = 1
        norm = np.sqrt(np.sum(np.abs(psi0)**2 * volumes))
    else:
        # 代数 L2 归一化 (无体积元权重)
        norm = np.linalg.norm(psi0)
        
    if norm < 1e-12:
        raise ValueError("初态波函数归一化异常，请检查网格点分布！")
        
    psi0 /= norm
    
    print(f"[Psi0-H] 物理 Slater 1s 初态构建完成 | 核中心索引: {nuc_idx} | 归一化常数: {norm:.4e}")
    return psi0

In [ ]:
import numpy as np

def build_hydrogen_delta_pulse_initial_state(points):
    """
    针对 Voronoi 实空间网格构建纯几何 δ-脉冲初态波函数
    - 仅在最靠近核的单个 Voronoi 胞腔赋值为 1.0
    - 其他所有胞腔初始均为 0.0
    - 无需做度规积分，点振幅直接归一化为 1.0
    """
    N = len(points)
    
    # 1. 寻找距离原点(核)最近的 Voronoi 胞腔索引
    r2 = np.sum(points**2, axis=1) if len(points.shape) > 1 else points**2
    nuc_idx = np.argmin(r2)
    
    # 2. 构造全零向量，仅在 nuc_idx 处设为 1.0
    psi0 = np.zeros(N, dtype=np.complex128)
    psi0[nuc_idx] = 1.0 + 0.0j
    
    print(f"[Psi0-Delta] 纯几何 δ-脉冲初态构建完成 | 核中心索引: {nuc_idx} | 振幅已集中于单一胞腔")
    return psi0

## Iteration Here 真正的迭代在这里！

In [ ]:
# ==========================================================================
# Complete replacement for the old ADAPT-VQE block in Voronoi.ipynb
# Drop in after the H_sparse_ext and geometry cells have run
# ==========================================================================

import numpy as np

# ------------------------------------------------------------------
# Step 1: Verify single-electron setup
# ------------------------------------------------------------------
n_electrons = 1   # Hydrogen: one electron
H_csr = H_sparse_ext   # 2**M*2**M sparse matrix from hamilton_14

total_qubits = int(np.log2(H_csr.shape[0]))  
D_check = 2 ** total_qubits   
M = total_qubits                
assert D_check == len(points_ext), \
    f"Grid mismatch: H implies D={D_check} but points_ext has {len(points_ext)}"
assert H_csr.shape[0] == 2**M, f"Expected 2**M-dim H, got {H_csr.shape[0]}"

print(f"✓ Hydrogen system: {len(points_ext)} grid points, "
      f"{total_qubits} qubits, H is {H_csr.shape[0]}×{H_csr.shape[0]}")

# ------------------------------------------------------------------
# Step 2: Build V5.11 pool (single electron, 256 points → 8 qubits)
# ------------------------------------------------------------------
# base_pool = build_multi_electron_tiered_pool_v5_11(
#     points_3d=points_ext,
#     neighbors=neighbors_ext,
#     n_electrons=n_electrons,   # 1
# )

# breakout_pool = generate_multi_electron_breakout_operators(
#     points_3d=points_ext,
#     neighbors=neighbors_ext,
#     n_electrons=n_electrons,   # 1
# )

# pool_input = breakout_pool + base_pool
pool_input = load_or_build_pool_input(
    points_3d=points_ext,
    neighbors=neighbors_ext,
    n_electrons=n_electrons
)

# Safe dimension check — never use to_matrix()
op_dim = 2 ** pool_input[0]['op'].num_qubits
assert op_dim == H_csr.shape[0], \
    f"Pool dim {op_dim} ≠ H dim {H_csr.shape[0]}"

tier_counts = {}
for e in pool_input:
    tier_counts[e['tier']] = tier_counts.get(e['tier'], 0) + 1
print(f"✓ Pool: {len(pool_input)} operators  "
      f"[T0:{tier_counts.get(0,0)} T1:{tier_counts.get(1,0)} "
      f"T2:{tier_counts.get(2,0)}]")

# Validate anti-Hermiticity
# local_validate_pool_v5_11(pool_input)

# ------------------------------------------------------------------
# Step 3: Initial state — Gaussian wavepacket on the 3D grid
# ------------------------------------------------------------------
# Hydrogen ground state: 1s orbital centered at nucleus (origin)
r_vals = np.linalg.norm(points_ext, axis=1)   # radial distance from origin
alpha_1s = 1.0                                  # hydrogen Bohr radius = 1 in atomic units
psi_init = np.exp(-alpha_1s * r_vals).astype(complex)
psi_init /= np.linalg.norm(psi_init)
# Use the T̄ ground state eigenvector — already grid-adapted
# psi_init = vecs_ext[:, 0].astype(complex)
# psi_init /= np.linalg.norm(psi_init)
# psi_init = psi0_ext
psi_seed = build_hydrogen_delta_pulse_initial_state(points_ext)
tau_factor=1
# psi_seed = np.load("/home/yening/scratch/PySCF/for_impulse_ansatz.npy")
psi_smoothed = smooth_psi_laplacian(psi_seed, L_sym, tau_factor)
psi_init = psi_smoothed

E0 = float(np.real(np.vdot(psi_init, H_csr @ psi_init)))
print(f"\nStarting energy : {E0:.8f} Hartree")
print(f"FCI lowest      : {np.linalg.eigvalsh(H_csr.toarray())[0]:.8f} Hartree")

print("🌐 [GLOBAL PRECOMPUTE] Converting operator pool to CSR matrices once...")
global_pool_sparse = [
    (1j * item['op'].to_matrix(sparse=True)).tocsr() 
    for item in pool_input
]

print("  ✓ Precomputation finished.")
max_outer_loops = 5

tier_weights = {0: 1, 1: 1, 2: 1}

all_results = []
prev_weights = None
for outer in range(max_outer_loops):

    print(f"ADAPT-VQE outer loop {outer}")
    print(f"tier_weights = {tier_weights}")

    # ------------------------------------------------------------------
    # Step 4: Run new ADAPT-VQE driver
    # ------------------------------------------------------------------
    psi_final, op_indices, thetas, grad_record = adapt_vqe_first_quantized_v5_8_tiered(
        hamiltonian=H_csr,
        wrapped_pool=pool_input,
        pool_sparse=global_pool_sparse,  #  直接传入预计算好的矩阵列表！
        initial_state=psi_init,
        max_iterations=15,
        grad_threshold=1e-6,
        max_ops_per_step=1,
        tier_weights = tier_weights,
        max_op_reuse = 1,
        return_gradients = True,
    )

    # Store results
    all_results.append({
        'psi': psi_final,
        'ops': op_indices,
        'thetas': thetas,
        'grads': grad_record
    })

    # No weights for the first 3 round
    # if outer < 3:
    #     continue

    #--------------------------------------------------
    # Compute average gradient per tier
    # grad_record is a list, each element contains {'tier':t, 'grad':g}
    #--------------------------------------------------
    tier_grads = {0: [], 1: [], 2: []}

    for item in grad_record:
        tier_grads[item['tier']].append(abs(item['grad']))

    avg_grad = {t: np.mean(v) if len(v) > 0 else 1.0 for t, v in tier_grads.items()}

    def safe_quantile(arr, q):
        return np.quantile(arr, q) if len(arr) > 0 else 1.0

    Q0 = safe_quantile(tier_grads[0], 0.75)
    Q1 = safe_quantile(tier_grads[1], 0.75)
    Q2 = safe_quantile(tier_grads[2], 0.75)

    #--------------------------------------------------
    # Adjust tier weights based on average gradients, noramlized to tier 0
    #--------------------------------------------------
    base = avg_grad[0] 
    # tier_weights[1] = int(base // avg_grad[1]) if avg_grad[1] > 0 else 1.0
    # tier_weights[2] = int(base // avg_grad[2]) if avg_grad[2] > 0 else 1.0
    # tier_weights[2] = tier_weights[2] / 10
    tier_weights[1] = int(Q0 // Q1) if Q1 > 0 else 1.0
    tier_weights[2] = int(Q0 // Q2) if Q2 > 0 else 1.0
    # tier_weights[2] = tier_weights[2]  # Tier 2 is usually less important

    if prev_weights is None:
        prev_weights = {0: 1.0, 1: 1.0, 2: 1.0}
    # 引入平滑阻尼/下限约束（例如平滑指数移动平均，或设置衰减阻尼因子 alpha）
    # w1_raw = Q0 / (Q1 + 1e-12)
    # w2_raw = Q0 / (Q2 + 1e-12)
    # alpha = 0.5  # 阻尼系数
    # tier_weights[1] = max(1.0, alpha * prev_weights[1] + (1 - alpha) * w1_raw)
    # tier_weights[2] = max(1.0, alpha * prev_weights[2] + (1 - alpha) * w2_raw)

    tier_weights = {
        0: 1,
        1: max(1, tier_weights[1]),
        2: max(1, tier_weights[2])
    }
    if prev_weights[2] == tier_weights[2]:
        tier_weights[2] *= 50
    else:
        prev_weights = tier_weights
    # print(f"Adjusted tier_weights = {tier_weights}")

    psi0_base = psi_final.copy()
    # np.random.seed(42) # 锁定随机种子确保可重复
    # weak_phase_noise = 1e-5 * np.random.randn(2**M)
    # psi_init = psi0_base
    # psi_init = psi0_base * np.exp(1j * weak_phase_noise)
    tau_factor=1
    psi_smoothed = smooth_psi_laplacian(psi0_base, L_sym, tau_factor)
    psi_init = psi_smoothed

    
E_final = float(np.real(np.vdot(psi_final, H_csr @ psi_final)))
print(f"\nFinal energy    : {E_final:.8f} Hartree")
# print(f"Target          : -0.577358 Hartree")
# print(f"Error           : {abs(E_final - (-0.577358)):.2e} Hartree")
print(f"Ansatz depth    : {len(op_indices)} operators")
np.save("/home/yening/scratch/PySCF/for_impulse_ansatz.npy", psi_final)

In [ ]:
# Test on a single Voronoi hop to see what's happening
from collections import defaultdict

# Manually build one hop operator (grid points 0 and 1)
bits = ((np.arange(256)[:, None] >> np.arange(8)) & 1).astype(np.uint8)
proj_maps = {
    (0, 0): [('I', 0.5),  ('Z', 0.5)],
    (1, 1): [('I', 0.5),  ('Z', -0.5)],
    (0, 1): [('X', 0.5),  ('Y', 0.5j)],
    (1, 0): [('X', 0.5),  ('Y', -0.5j)],
}

acc = defaultdict(complex)
for chars, w in _lcu_grow_path(bits[0], bits[1], 8, proj_maps):
    acc["".join(chars)] += (+1.0) * w
for chars, w in _lcu_grow_path(bits[1], bits[0], 8, proj_maps):
    acc["".join(chars)] += (-1.0) * w

# Check what parts are real vs imaginary
real_total = sum(abs(v.real) for v in acc.values())
imag_total = sum(abs(v.imag) for v in acc.values())
print(f"Real parts total: {real_total:.6f}")
print(f"Imag parts total: {imag_total:.6f}")

# Check Y-count distribution
for pauli_str, coeff in list(acc.items())[:5]:
    num_y = pauli_str.count('Y')
    print(f"  Y-count={num_y}  coeff={coeff:.4f}  str={pauli_str[:10]}...")

In [ ]:
import pickle
num_qubits = int(np.log2(H_csr.shape[0]))

# 1. 生成全域破局算符
breakout_ops = generate_scaling_and_gauge_operators(num_qubits)

# 2. 强行注入全池（可以放在池的最前端，让活跃池优先轮询）
extended_pool_ops = breakout_ops + pool_ops
# 1. 重新读入保存的断点
checkpoint_filename = "adapt_vqe_checkpoint_step2000.pkl"

with open(checkpoint_filename, 'rb') as f:
    checkpoint = pickle.load(f)

# print(f"成功载入断点！读取到的历史步数: {checkpoint['current_step']}")

# 2. 从断点恢复核心变量
psi_checkpoint = checkpoint['psi']
ansatz_ops_indices_checkpoint = checkpoint['ansatz_ops_indices']
thetas_checkpoint = checkpoint['thetas']

# 3. 完美承接历史，继续向更低能量冲锋
psi, ansatz_ops_indices, thetas = adapt_vqe_first_quantized_v4(
    H_csr=H_csr,
    pool_ops=extended_pool_ops,         # 必须传入包含 0, 1 号全域算符的扩展池
    psi_init=psi_checkpoint,            # 🌟 关键：将当前波函数作为新的初始态传入！
    max_steps=1000, 
    block_size=2,
    offset=checkpoint['current_step'],  # 🌟 关键：让日志输出接着之前的步数往后续写
    exclude_window=20,
    use_regional_selection=False,
    exp_manager=None,                   # 保持手动控制
    loop_window_size=30,
    loop_max_frequency=1
)

In [ ]:
np.save("best_April28_psi.npy", psi)
psi_seed = np.load("best_April28_psi.npy")
psi, ansatz_ops_indices, thetas = adapt_vqe_first_quantized_v4(
    H_csr=H_csr,
    pool_ops=extended_pool_ops,         # 必须传入包含 0, 1 号全域算符的扩展池
    psi_init=psi_seed,            # 🌟 关键：将当前波函数作为新的初始态传入！
    max_steps=1000, 
    block_size=2,
    offset=checkpoint['current_step'],  # 🌟 关键：让日志输出接着之前的步数往后续写
    exclude_window=20,
    use_regional_selection=False,
    exp_manager=None,                   # 保持手动控制
    loop_window_size=30,
    loop_max_frequency=1
)

In [ ]:
import matplotlib.pyplot as plt

# 假设你的网格是 1D 的，或者你可以把 psi 展平/取模平方看几率分布
plt.figure(figsize=(10, 4))
plt.plot(np.abs(psi_final)**2, label="Probability Density")
plt.title("Current Non-physical Wavefunction State")
plt.xlabel("Grid Index")
plt.ylabel("|Psi|^2")
plt.legend()
plt.show()

In [ ]:
import pickle
import numpy as np

# 建立一个精心命名的字典，把断点续算所需的核心物理量全部打包
checkpoint_data = {
    'psi': psi,                                 # 当前优化后的波函数状态矢
   'ansatz_ops_indices': op_indices,   # 已经吃进去的算符索引序列（包含0和1号）
    'thetas': thetas,                           # 对应的变分角度参数列表
    'current_step': len(op_indices)     # 记录当前的步数（约1988步）
}

# 固化到 D 盘或当前工作目录下（建议起个有辨识度的名字，比如带有2000steps和时间戳）
checkpoint_filename = "adapt_vqe_checkpoint_step2000.pkl"

with open(checkpoint_filename, 'wb') as f:
    pickle.dump(checkpoint_data, f)

print(f"🎉 成功保存断点信息至 {checkpoint_filename}！")
# print(f"当前算符总数: {checkpoint_data['current_step']}, 放心关机装固态吧！")

In [ ]:
import heapq

class OperatorEliteStack:
    def __init__(self, max_size=50):
        self.max_size = max_size
        # 使用最小堆，堆顶永远是表现最差的“差生”
        self.elite_pool = [] 
        # 用于记录算符的历史统计数据
        self.op_stats = {} # {op_idx: {'freq': 0, 'total_de': 0.0, 'last_step': -1}}
        
    def calculate_score(self, op_idx, delta_e, theta, current_step):
        # 更新基础统计
        if op_idx not in self.op_stats:
            self.op_stats[op_idx] = {'freq': 0, 'total_de': 0.0, 'last_step': current_step}
        
        stats = self.op_stats[op_idx]
        stats['freq'] += 1
        stats['total_de'] += abs(delta_e)
        
        # 权重分配 (可以根据你的偏好调整)
        w1, w2, w3 = 0.6, 0.3, 0.1
        
        # 1. 能量分 (使用 log 缩放避免大数冲击)
        e_score = np.log10(abs(delta_e) + 1e-12) + 12 
        
        # 2. 角度分 (反映结构修正)
        theta_score = np.log10(abs(theta) + 1e-12) + 12
        
        # 3. 频率分 (历史表现)
        freq_score = np.log2(stats['freq'] + 1)
        
        # 综合评分
        score = w1 * e_score + w2 * theta_score + w3 * freq_score
        return score

    def push(self, op_idx, score, metadata=None):
        """
        score: 综合评价指标 (越高越好)
        metadata: 存储 op 的具体信息或物理含义
        """
        entry = (score, op_idx, metadata)
        
        if len(self.elite_pool) < self.max_size:
            heapq.heappush(self.elite_pool, entry)
        else:
            # 与当前表现最差的算符比较
            if score > self.elite_pool[0][0]:
                heapq.heapreplace(self.elite_pool, entry)
                
    def get_all_indices(self):
        return [item[1] for item in self.elite_pool]

    def get_best_score(self):
        return max([item[0] for item in self.elite_pool]) if self.elite_pool else 0

In [ ]:
def precision_perturbation(psi_curr, elite_stack, pool, epsilon=1e-7):
    """
    基于王牌算符池生成定向扰动
    """
    perturbation_vector = np.zeros_like(psi_curr, dtype=complex)
    indices = elite_stack.get_all_indices()
    
    for idx in indices:
        op = pool[idx]
        # 让优质算符作用在当前态上产生微扰方向
        # 权重可以根据 score 进一步细化
        perturbation_vector += op.apply(psi_curr) 
        
    # 归一化扰动方向
    if np.linalg.norm(perturbation_vector) > 0:
        perturbation_vector /= np.linalg.norm(perturbation_vector)
        
    return psi_curr + epsilon * perturbation_vector

### 绘制波函数

In [ ]:
import numpy as np

def get_dr_array(r_array):
    """
    计算径向网格点的步长数组 dr。
    对于边界点使用单边差分，中间点使用中心差分。
    """
    dr = np.zeros_like(r_array)
    
    # 如果只有一个点，dr 没有物理意义，设为 1 或抛出异常
    if len(r_array) < 2:
        return np.array([1.0])

    # 1. 起点：使用前两个点的间距
    dr[0] = r_array[1] - r_array[0]
    
    # 2. 中间点：使用中心差分 (r_{i+1} - r_{i-1}) / 2
    # 这能最准确地反映该点所代表的壳层厚度
    dr[1:-1] = (r_array[2:] - r_array[:-2]) / 2.0
    
    # 3. 终点：使用最后两个点的间距
    dr[-1] = r_array[-1] - r_array[-2]
    
    return dr

In [ ]:
def plot_radial_evolution(psi_list, r_array, volumes, num_leb=50, labels=None):
    plt.figure(figsize=(8, 5))
    dr_array = get_dr_array(r_array)
    
    # 用来记录 0-6 范围内的最大值，以便自动调整纵坐标
    max_val_in_range = 0 
    
    for i, psi in enumerate(psi_list):
        density = np.abs(psi)**2
        radial_p = []
        
        for r_idx in range(len(r_array)):
            start = r_idx * num_leb
            end = start + num_leb
            layer_prob = np.sum(density[start:end] * volumes[start:end])
            val = layer_prob / dr_array[r_idx]
            radial_p.append(val)
            
            # 统计 0 <= r <= 6 范围内的最大概率密度
            if r_array[r_idx] <= 12:
                if val > max_val_in_range:
                    max_val_in_range = val
        
        label = labels[i] if labels else f"Stage {i}"
        plt.plot(r_array, radial_p, label=label, marker='o', markersize=3)
    
    # --- 核心修改：双轴截断 ---
    plt.xlim(0, 12)
    
    # 纵坐标自动设为 0-6 区间最大值的 1.1 倍，给顶部留点空间
    if max_val_in_range > 0:
        plt.ylim(-0.02 * max_val_in_range, max_val_in_range * 1.1)
    else:
        plt.ylim(-0.05, 1.0) # 如果全是 0，给个默认范围
        
    plt.xlabel("Radius r (Bohr)")
    plt.ylabel("Radial Probability P(r)")
    plt.title("Radial Probability Distribution: From Far-field Localization to Potential Well Trapping")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
ALPHA_PARAM = 4.0
NU_PARAM = 2.0
r_array = becke_radial_grid(num_r=20, alpha=ALPHA_PARAM, nu=NU_PARAM)

In [ ]:
# del psi_list
psi_list = []
psi_list.append(psi_init)
psi_list.append(psi_final)
# psi_list = psi_list[2:]  # 只取前两阶段的波函数
plot_radial_evolution(psi_list, r_array, volumes, num_leb=50, labels=None)

# Try ITE (Imaginary Time Envolvement)

In [ ]:
# Trancate H_pauli so that low coefficient will NOT
# consume extra memory.
from qiskit.quantum_info import SparsePauliOp

# 假设 h_op 是你的百万级算子
threshold = 1e-4  # 根据精度需求调整，通常 1e-6 到 1e-8
coeffs = H_pauli.coeffs
labels = H_pauli.paulis

# 只保留系数绝对值大于阈值的项
mask = np.abs(coeffs) > threshold
H_pauli_slim = SparsePauliOp(labels[mask], coeffs[mask])

print(f"瘦身前: {len(H_pauli)} 项 -> 瘦身后: {len(H_pauli_slim)} 项")

In [ ]:
import numpy as np
from qiskit_algorithms.time_evolvers.variational import VarQITE
from qiskit_algorithms.time_evolvers.variational import RealMcLachlanPrinciple
from qiskit_algorithms import TimeEvolutionProblem
from qiskit.primitives import Estimator

# --- 数值预处理 ---
# 既然最小值是 -0.79，最大值是 250万
# 我们缩小 10^5 倍，使能谱范围落在 [-0.00001, 25] 之间
scale_factor = 1e5
H_scaled = H_pauli / scale_factor

# 1. 设置演化参数
# 虚时间演化的总时长 tau。对于 10^6 级别的跨度，tau 不需要很大就能显著降低能量
total_tau = 0.5 
delta_tau = 0.01  # 每一步的步长
num_steps = int(total_tau / delta_tau)

# 2. 定义变分原理 (McLachlan Principle)
# 它是将算子作用映射到参数空间的核心
var_principle = RealMcLachlanPrinciple()

# 3. 初始化 Estimator (使用 V2 类似的 Primitive)
estimator = Estimator()

# 4. 配置 VarQITE 算法
# 它可以自动处理参数的更新逻辑
qite = VarQITE(
    ansatz=qc,          # 你的 HVA Ansatz
    initial_parameters=np.random.normal(0, 0.01, dim), # 初始参数
    variational_principle=var_principle,
    estimator=estimator
)

# 5. 定义演化问题
# 我们想让状态演化 tau 时间
problem = TimeEvolutionProblem(H_pauli, time=total_tau, aux_operators=[H_pauli])

# 6. 执行演化
print(f"开始虚时间演化，总步数: {num_steps}...")
result = qite.evolve(problem)

# 7. 提取能量演化轨迹
# result.observables 包含了每一步演化时的 H_pauli 期望值
energies = [obs[0][0] for obs in result.observables]

print("演化完成！")
print(f"最终预测基态能量: {energies[-1]:.6f}")

# 8. 可视化
import matplotlib.pyplot as plt
plt.plot(np.linspace(0, total_tau, len(energies)), energies)
plt.yscale('log') # 建议使用对数坐标，因为你的能量跨度太大了
plt.xlabel('Imaginary Time (tau)')
plt.ylabel('Energy')
plt.title('VarQITE Convergence')
plt.grid(True)
plt.show()

## Draw circuit
The following circuit for an ansatz is from the tutorial https://qiskit.qotlabs.org/learning/courses/quantum-diagonalization-algorithms/vqe

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

def two_qubit_5layer_ansatz():
    # 定义参数
    theta0 = Parameter("θ0")  # 左侧 q0 上的 Ry
    theta1 = Parameter("θ1")  # 左侧 q1 上的 Rx
    theta2 = Parameter("θ2")  # 中间 ZZ 的角度（通过 Rz 实现）
    theta3 = Parameter("θ3")  # 右侧 q0 上的 Ry
    theta4 = Parameter("θ4")  # 右侧 q1 上的 Rx

    qc = QuantumCircuit(2)

    # 1) 左侧单比特层
    qc.ry(theta0, 0)
    qc.rx(theta1, 1)

    # 2–4) 中间一记 ZZ 相互作用：CNOT–Rz–CNOT
    qc.cx(0, 1)
    qc.rz(theta2, 1)
    qc.cx(0, 1)

    # 5) 右侧单比特层
    qc.ry(theta3, 0)
    qc.rx(theta4, 1)

    return qc

qc = two_qubit_5layer_ansatz()
print(qc.draw(fold=-1))

In [ ]:
"""
    这一部分算作是VQE中的estimator的功能实现。
    pauli_terms: list of (coeff: complex/float, pauli_string: str)
    psi: statevector, shape = (2**n,)
    返回 <psi| H_tilde |psi> 的实部（理论上应为实数）
    
"""
from qiskit.quantum_info import SparsePauliOp

def estimate_energy(pauli_terms, psi: np.ndarray) -> float:
    E = 0.0 + 0.0j
    for coeff, pstr in pauli_terms:
        P = pauli_matrix(pstr)      # 2^n x 2^n
        phi = P @ psi               # |phi> = P |psi>
        E += coeff * np.vdot(psi, phi)  # <psi|phi>
    return E.real

def estimate_energy_sparse(H_pauli, psi):
    phi = H_pauli @ psi
    return np.vdot(psi, phi).real

def estimate_energy_sparse(H_pauli: SparsePauliOp, psi: np.ndarray) -> float:
    # 把 SparsePauliOp 变成 2^n x 2^n 的矩阵
    H_mat = H_pauli.to_matrix()      # dense matrix, shape = (2**n, 2**n)
    phi = H_mat @ psi                # |phi> = H |psi>
    return np.vdot(psi, phi).real

In [ ]:
%%script false
si_disc = vecs[:, 0]
num_qubits = H_pauli.num_qubits
print(len(psi_disc_padded), 2**H_pauli.num_qubits)
print([len(p.to_label()) for p in H_pauli.paulis[:5]])
psi_disc_padded = pad_statevector(psi_disc, num_qubits)
E_est = estimate_energy_sparse(H_pauli, psi_disc_padded)
print("⟨ψ_disc| H_tilde |ψ_disc⟩ =", E_est)

In [ ]:
%%script false
import numpy as np
from qiskit.quantum_info import Operator

length = len(psi_disc)
vols = np.array(volumes[:length])

# 1) 做体积加权变换：ψ_sym[m] = sqrt(U_m) * ψ_disc[m]
psi_sym = np.sqrt(vols) * psi_disc

# 2) 归一化一下（数值上更稳）
psi_sym = psi_sym / np.linalg.norm(psi_sym)

# 3) pad 到 2^M 维
num_qubits = H_pauli.num_qubits
dim = 2**num_qubits
psi_sym_padded = np.zeros(dim, dtype=complex)
psi_sym_padded[:length] = psi_sym

# 4) 把 H_pauli 变成矩阵
H_from_pauli = Operator(H_pauli).data

# 5) 算期望值
E_sym = np.vdot(psi_sym_padded, H_from_pauli @ psi_sym_padded).real
print("⟨ψ_sym| H_pauli |ψ_sym⟩ =", E_sym)

In [ ]:
# from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_algorithms.optimizers import SPSA
num_qubits = H_pauli.num_qubits
print("num_bits: ", num_qubits)
ansatz = TwoLocal(
    num_qubits,
    rotation_blocks='ry',
    entanglement_blocks='cz',
    entanglement='full',
    reps=1
)

# ⭐ 换成 SPSA 优化器
optimizer = SPSA(maxiter=200)

estimator = Estimator() # ⭐ 使用 StatevectorEstimator（V2 API）

vqe = VQE(
    estimator=estimator,
    ansatz=ansatz,
    optimizer=optimizer
)

result = vqe.compute_minimum_eigenvalue(H_from_pauli)

print("VQE ground state energy:", result.eigenvalue.real)

In [ ]:
"""
    用 (alpha I - H) 的 power method 找最小本征值（基态能量）
    H_mat: numpy 矩阵（Hermitian）
"""
def power_method_ground_matrix(
    H_mat: np.ndarray,
    num_iters: int = 200,
    tol: float = 1e-12,
    alpha: float | None = None,
    verbose: bool = False
):
    n = H_mat.shape[0]

    # 自动选择 alpha：谱半径 + 1
    if alpha is None:
        alpha = np.linalg.norm(H_mat, ord=2) + 1.0

    A = alpha * np.eye(n, dtype=complex) - H_mat

    # 随机初始向量    
    v = np.random.randn(n) + 1j * np.random.randn(n)
    v /= np.linalg.norm(v)

    last_E = None

    for k in range(num_iters):
        v = A @ v
        v /= np.linalg.norm(v)

        # Rayleigh quotient
        E = np.vdot(v, H_mat @ v).real

        if verbose and k % 20 == 0:
            print(f"iter {k}: E ≈ {E}")

        if last_E is not None and abs(E - last_E) < tol:
            break

        last_E = E

    return E, v


In [ ]:
E_direct, psi_direct = power_method_ground_matrix(H, verbose=True)
print("Power Method on original H:", E_direct)

In [ ]:
"""
    Rayleigh Quotient Iteration for smallest eigenvalue of Hermitian H.
    Extremely fast and stable for 1024x1024 matrices.
"""
def rayleigh_quotient_iteration(H, max_iter=50, tol=1e-12, verbose=True):
    n = H.shape[0]

    # 随机初始向量
    v = np.random.randn(n) + 1j*np.random.randn(n)
    v /= np.linalg.norm(v)

    for k in range(max_iter):
        # Rayleigh quotient
        mu = np.vdot(v, H @ v).real

        if verbose:
            print(f"iter {k}: E ≈ {mu}")

        # Solve (H - mu I) w = v
        try:
            w = np.linalg.solve(H - mu*np.eye(n), v)
        except np.linalg.LinAlgError:
            # 如果矩阵奇异，稍微 perturb 一下
            w = np.linalg.solve(H - (mu+1e-8)*np.eye(n), v)

        v = w / np.linalg.norm(w)

        # Check convergence
        if k > 0 and abs(mu - last_mu) < tol:
            break

        last_mu = mu

    return mu, v

In [ ]:
E_direct, psi_direct = rayleigh_quotient_iteration(H, max_iter=100, tol=1e-12, verbose=True)
print("RQI on original H:", E_direct)

In [ ]:
def bits_of(x, num_bits): 
    return [(x >> k) & 1 for k in range(num_bits)] # LSB→MSB

"""
    在单个 qubit 上合并 I/X/Y/Z 和一个新的 X 或 Z，返回(新label, phase)
    phase 是 {1, 1j, -1j, -1} 之一
    单比特上合并两个 Pauli：a * b
    返回 (c, phase)，使得 a b = phase * c
    a, b, c ∈ {'I','X','Y','Z'}
    phase ∈ {1, -1, 1j, -1j}
"""
def merge_pauli(a, b):
    if b == 'I':
        return a, 1.0
    if a == 'I':
        return b, 1.0

    # 完整乘法表（固定顺序：先 a 后 b）
    table = {
        ('X','X'): ('I', 1.0),
        ('Y','Y'): ('I', 1.0),
        ('Z','Z'): ('I', 1.0),

        ('X','Y'): ('Z', 1j),
        ('Y','Z'): ('X', 1j),
        ('Z','X'): ('Y', 1j),

        ('Y','X'): ('Z', -1j),
        ('Z','Y'): ('X', -1j),
        ('X','Z'): ('Y', -1j),
    }

    if (a, b) not in table:
        raise ValueError(f"unsupported merge: {a}, {b}")

    return table[(a, b)]

#gamma = np.zeros_like(omega_prime, dtype=complex)
def Hbar_LCU_from_omega_gamma(omega_prime, gamma, eta):
    N = omega_prime.shape[0]
    logN = int(np.ceil(np.log2(N)))
    num_qubits = eta * logN

    pauli_dict = {} # label -> coeff

    # 第一项：omega' 部分
    for m in range(N):
        bits_m = bits_of(m, logN)
        for n in range(N):
            coeff_mn = omega_prime[m, n]
            if abs(coeff_mn) < 1e-12:
                continue
            bits_n = bits_of(n, logN)

            for i in range(eta):
                # 初始化全 I
                labels = ['I'] * num_qubits
                phase = 1.0 + 0.0j

                for b in range(logN):
                    q = i * logN + b
                    # 先加 X^{m_b}
                    if bits_m[b] == 1:
                        labels[q], ph = merge_pauli(labels[q], 'X')
                        phase *= ph
                    # 再加 Z^{n_b}
                    if bits_n[b] == 1:
                        labels[q], ph = merge_pauli(labels[q], 'Z')
                        phase *= ph

                label_str = "".join(labels[::-1]) # MSB→LSB
                pauli_dict[label_str] = pauli_dict.get(label_str, 0.0) + coeff_mn * phase
    
    # # 第二项：gamma 部分（全是 Z）
    # for m in range(N):
    #     bits_m = bits_of(m, logN)
    #     for p in range(N):
    #         coeff_mp = 0.5 * gamma[m, p]
    #         if abs(coeff_mp) < 1e-12:
    #             continue
    #         bits_p = bits_of(p, logN)

    #         for i in range(eta):
    #             for j in range(eta):
    #                 if i == j:
    #                     continue

    #                 labels = ['I'] * num_qubits
    #                 phase = 1.0 + 0.0j

    #                 for b in range(logN):
    #                     qi = i * logN + b
    #                     qj = j * logN + b
    #                     if bits_m[b] == 1:
    #                         labels[qi], ph = merge_pauli(labels[qi], 'Z')
    #                         phase *= ph
    #                     if bits_p[b] == 1:
    #                         labels[qj], ph = merge_pauli(labels[qj], 'Z')
    #                         phase *= ph

    #                 label_str = "".join(labels[::-1])
    #                 pauli_dict[label_str] = pauli_dict.get(label_str, 0.0) + coeff_mp * phase

    # 收尾：丢掉很小的项，构造 SparsePauliOp
    labels = []
    coeffs = []
    for L, c in pauli_dict.items():
        if abs(c) > 1e-10:
            labels.append(L)
            coeffs.append(c)

    return SparsePauliOp(labels, coeffs)

In [ ]:
omega_prime=np.array(omega_prime_matr(vor, points, neighbors),dtype=complex)
gamma = np.zeros_like(omega_prime, dtype=complex)
eta = 1
Hbar = Hbar_LCU_from_omega_gamma(omega_prime, gamma, eta)
Hbar_mat = Hbar.to_matrix()
vals, vecs = np.linalg.eigh(Hbar_mat)
print(vals[0])

In [ ]:
print("hermiticity:", np.max(np.abs(Hbar_mat - Hbar_mat.conj().T)))

In [ ]:
%%script false
# Use sparse matrix linear algebra method to obtain first value of the spectrum 2025 12 20
import numpy as np
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import eigsh # eigenvalue for sparse matrix

def hamilton_sparse(vor, points, neighbors):
    lapl = lapl_matr(vor, points, neighbors)
    ele_nu, ele_ele = coulombic_potential(points, volumes)

    N = len(points)
    H = lil_matrix((N, N))

    for m in range(N):
        for n in range(N):
            if lapl[m][n] != 0.0:
                H[m, n] += -0.5 * lapl[m][n]
        H[m, m] += ele_nu[m]

    return H.tocsr()

In [ ]:
"""
    单比特 |mb><nb| 展开成 Pauli 的线性组合。
    返回：[(label, coeff), ...]，label ∈ {'I','X','Y','Z'}
"""
def single_projector_paulis(mb: int, nb: int):
    if mb == 0 and nb == 0:
        # |0><0| = (I + Z)/2
        return [("I", 0.5), ("Z", 0.5)]
    elif mb == 1 and nb == 1:
        # |1><1| = (I - Z)/2
        return [("I", 0.5), ("Z", -0.5)]
    elif mb == 0 and nb == 1:
        # |0><1| = (X + iY)/2
        return [("X", 0.5), ("Y", 0.5j)]
    elif mb == 1 and nb == 0:
        # |1><0| = (X - iY)/2
        return [("X", 0.5), ("Y", -0.5j)]
    else:
        raise ValueError("mb, nb must be 0 or 1")

In [ ]:
"""
    多比特 |m><n| 展开成 Pauli 张量积的线性组合。
    返回：
      pauli_labels: ['IXZ...', ...]  （Qiskit 约定：字符串左边是最高位 qubit）
      coeffs:       [c1, c2, ...]    （复数系数）
"""
def projector_mn_to_paulis(m: int, n: int, num_bits: int):

    bits_m = int_to_bits(m, num_bits) # LSB→MSB
    bits_n = int_to_bits(n, num_bits)

    # 每个比特上都有 2 项：[(label, coeff), (label, coeff)]
    per_qubit_expansions = [ 
        single_projector_paulis(mb, nb) for mb, nb in zip(bits_m, bits_n) 
    ]

    pauli_labels = [] 
    coeffs = []

    # 笛卡尔积组合所有比特上的选择
    for combo in itertools.product(*per_qubit_expansions):
        # combo 是长度 num_bits 的 tuple，每个元素是 (label, coeff)
        labels_per_qubit = [item[0] for item in combo]
        coeff_per_qubit = [item[1] for item in combo]

        # 张量积后的整体 Pauli label：按 qubit 顺序拼接
        # 我们的 bits 是 LSB→MSB，Qiskit 字符串是 MSB→LSB，所以要反转
        label = "".join(labels_per_qubit[::-1])

        # 总系数是各比特系数的乘积
        coeff = 1.0 + 0.0j
        for c in coeff_per_qubit:
            coeff *= c

        pauli_labels.append(label) 
        coeffs.append(coeff)   

    return pauli_labels, coeffs

In [ ]:
"""
    用 |m><n| 的 Pauli 展开来重建 H 的 Pauli 表示。
    这是“教科书级别正确”的做法，但复杂度高，只适合小规模验证。
"""
def matrix_to_pauli_via_projectors(H_mat: np.ndarray, 
num_bits: int, tol: float = 1e-10):

    dim = 1 << num_bits
    assert H_mat.shape == (dim, dim)

    pauli_dict = {} # label -> coeff 累加

    for m in range(dim):
        for n in range(dim):
            coeff_mn = H_mat[m, n]
            if abs(coeff_mn) < tol:
                continue

            labels_mn, coeffs_mn = projector_mn_to_paulis(m, n, num_bits)
            for label, c in zip(labels_mn, coeffs_mn):
                pauli_dict[label] = pauli_dict.get(label, 0.0) + coeff_mn * c

    # 去掉非常小的项 
    pauli_labels = []  
    coeffs = []
    for label, c in pauli_dict.items():
        if abs(c) > tol:
            pauli_labels.append(label)
            coeffs.append(c)

    return SparsePauliOp(pauli_labels, coeffs)

In [ ]:
"""生成所有长度为 num_bits 的 Pauli 字符串（MSB→LSB）"""
def pauli_basis(num_bits):
    single = ['I', 'X', 'Y', 'Z']
    for prod in itertools.product(single, repeat=num_bits):
        yield "".join(prod)

"""把 Pauli 字符串转成矩阵"""
def pauli_matrix(label):
    I = np.array([[1,0],[0,1]], dtype=complex)
    X = np.array([[0,1],[1,0]], dtype=complex)
    Y = np.array([[0,-1j],[1j,0]], dtype=complex)
    Z = np.array([[1,0],[0,-1]], dtype=complex)
    base = {'I':I, 'X':X, 'Y':Y, 'Z':Z}

    M = base[label[0]]
    for ch in label[1:]:
        M = np.kron(M, base[ch])
    return M

def matrix_to_pauli_trace(H_mat, num_bits, tol=1e-10):
    dim = 1 << num_bits
    pauli_labels = []
    coeffs = []

    for label in pauli_basis(num_bits):
        P = pauli_matrix(label)
        c = np.trace(P @ H_mat) / dim
        if abs(c) > tol:
            pauli_labels.append(label)
            coeffs.append(c)

    return SparsePauliOp(pauli_labels, coeffs)

In [ ]:
coeff = omega_prime_matr(vor, points, neighbors)
H_mat = np.array(coeff, dtype=complex)

num_bits = math.ceil(math.log2(len(points)))
H_pauli = matrix_to_pauli_trace(H_mat, num_bits)

In [ ]:
%%script false
# 1. 得到 14 式的矩阵（length x length） 
ham_small = hamilton_14(vor, points, neighbors) 
ham_small = np.array(ham_small, dtype=complex)

# 2. 补成 2^n 维
length = len(ham_small)
num_bits = int(np.ceil(np.log2(length)))
dim = 1 << num_bits
H_mat = np.zeros((dim, dim), dtype=complex)
H_mat[:length, :length] = ham_small

# 3. 用 projector 展开成 Pauli 和
H_pauli = matrix_to_pauli_via_projectors(H_mat, num_bits)

# 4. 再转回矩阵
H_back = H_pauli.to_matrix()

# 5. 比较差异
diff = H_back - H_mat
print("max abs diff:", np.max(np.abs(diff)))

In [ ]:
%%script false
def pauli_string_from_mn(m, n, num_bits):
    bits_m = int_to_bits(m, num_bits)
    bits_n = int_to_bits(n, num_bits)

    pauli_ops = []
    for b in range(num_bits):
        mb = bits_m[b]
        nb = bits_n[b]

        if mb == 0 and nb == 0:
            pauli_ops.append("I")
        elif mb == 1 and nb == 0:
            pauli_ops.append("X")
        elif mb == 0 and nb == 1:
            pauli_ops.append("Z")
        elif mb == 1 and nb == 1:
            pauli_ops.append("Y")  # XZ = ±iY，忽略全局相位即可

    # Qiskit uses MSB→LSB, so reverse
    return "".join(pauli_ops[::-1])


In [ ]:
# This function is to realize the expression 26 in the paper
# For the time being, for the single electron system, we only
# need to consider the first term.
# 2026, 01, 02
#from qiskit.quantum_info import SparsePauliOp

def Hbar_LCU(vor, points, neighbors):

    length = len(points)
    num_bits = math.ceil(math.log2(length))
    dim = 1 << num_bits   # = 2^num_bits

    # 1. compute ω'_{mn} (N×N)
    #coeff_small = omega_prime_matr(vor, points, neighbors)
    coeff = omega_prime_matr(vor, points, neighbors)

    # 2. extend to 2^num_bits × 2^num_bits
    #coeff = np.zeros((dim, dim))
    #coeff[:length, :length] = coeff_small

    pauli_list = []
    coeff_list = []

    # 3. build Pauli expansion
    for m in range(dim):
        for n in range(dim):
            coeff_elem = coeff[m][n]
            if abs(coeff_elem) < 1e-12:
                continue  # skip zero terms

            #Pmn = pauli_string_from_mn(m, n, num_bits)
            #pauli_list.append(Pmn)
            #coeff_list.append(coeff_elem)
            labels, coeffs = projector_mn_to_paulis(m, n, num_bits)
            for L, c in zip(labels, coeffs):
                pauli_list.append(L)
                coeff_list.append(coeff_elem * c)

    # 4. construct SparsePauliOp
    H = SparsePauliOp(pauli_list, coeff_list).simplify()

    return H


In [ ]:

#H = Hbar_LCU(vor, points, neighbors)



In [ ]:
def compute_E_max(H: SparsePauliOp) -> float:
    """用矩阵本征值计算谱半径 E_max"""
    Hmat = H.to_matrix()
    eigvals = np.linalg.eigvalsh(Hmat)
    E_max = np.max(np.abs(eigvals))
    return float(E_max), eigvals
#Hmat = H.to_matrix()
#evals, _ = np.linalg.eigh(Hmat)
#E_max = np.max(np.abs(evals))
E_max, evals = compute_E_max(H)
print("Exact eigenvalues:", np.sort(evals.real))
print("E_max =", E_max)

In [ ]:
# 谱界计算
def compute_spectral_bounds(H: SparsePauliOp):
    Hmat = H.to_matrix()
    eigvals = np.linalg.eigvalsh(Hmat)
    E_min = float(np.min(eigvals))
    E_max = float(np.max(eigvals))
    return E_min, E_max, eigvals

In [ ]:
# 以下为QEVE的第一次尝试。
# ========= 1) 准备近似本征态 =========
"""
    用户自定义：构造一个把 |0...0> 准备到近似本征态 |psi> 的电路。
    这里先给个占位：全零态本身。
"""
def build_prep_eigenstate_circ(num_qubits: int) -> QuantumCircuit:

    qc = QuantumCircuit(num_qubits)

    return qc

# ========= 2) 构造 e^{-i H t} 作用在 |psi> 上 =========
"""
    给定 H、准备态电路 prep_circ 和时间 t，
    返回 |psi(t)> = e^{-i H t} |psi> 的 Statevector。
"""    
def evolve_state(H: SparsePauliOp, prep_circ: QuantumCircuit, t: float) -> Statevector:
    num_qubits = H.num_qubits

    qc = QuantumCircuit(num_qubits)
    # 先准备 |psi>
    qc.compose(prep_circ, inplace=True)

    # 加上时间演化 U(t) = exp(-i H t)
    evo_gate = PauliEvolutionGate(H, time=t)
    qc.append(evo_gate, range(num_qubits))

    backend = Aer.get_backend("aer_simulator_statevector")
    tqc = transpile(qc, backend)
    result = backend.run(tqc).result()
    sv = result.get_statevector(tqc)

    return sv

# ========= 3) 计算 overlap C(t) = <psi|psi(t)> =========
"""
    计算 C(t) = <psi| e^{-i H t} |psi>
    用 statevector 直接算内积。
"""
def compute_overlap(H: SparsePauliOp, prep_circ: QuantumCircuit, t: float) -> complex:
    num_qubits = H.num_qubits

    backend = Aer.get_backend("aer_simulator")
    backend.set_options(method="statevector")

    # |psi>
    qc_psi = QuantumCircuit(num_qubits)
    qc_psi.compose(prep_circ, inplace=True)
    qc_psi.save_statevector()

    tqc_psi = transpile(qc_psi, backend)
    result_psi = backend.run(qc_psi).result()
    sv_psi = result_psi.get_statevector(tqc_psi)

    # |psi(t)>
    qc_evo = QuantumCircuit(num_qubits)
    qc_evo.compose(prep_circ, inplace=True)

    evo_gate = PauliEvolutionGate(H, time=t)
    qc_evo.append(evo_gate, range(num_qubits))
    
    qc_evo.save_statevector()

    tqc_evo = transpile(qc_evo, backend, optimization_level=3)
    result_evo = backend.run(tqc_evo).result()
    sv_psi_t = result_evo.get_statevector(tqc_evo)

    # overlap
    C_t = np.vdot(sv_psi, sv_psi_t) # <psi|psi(t)>
    return C_t

# ========= 4) 主流程：QEVE 频谱估计 =========
"""
    QEVE 主流程：
    - 在 [0, t_max] 上取 num_time_points 个时间点
    - 计算 C(t_k)
    - 对 C(t_k) 做 FFT，得到频谱
    - 返回频率轴 freq 和谱密度 |C̃(ω)|^2
"""
def qeve(H: SparsePauliOp, 
prep_circ: QuantumCircuit | None = None, 
t_max: float = 10.0, num_time_points: int = 128):
    num_qubits = H.num_qubits

    if prep_circ is None:
        prep_circ = build_prep_eigenstate_circ(num_qubits)

    # 时间网格
    ts = np.linspace(0.0, t_max, num_time_points)
    C_ts = []

    for t in ts:
        C_t = compute_overlap(H, prep_circ, t)
        C_ts.append(C_t)

    C_ts = np.array(C_ts)

    # 做 FFT：注意这里的频率单位是 “能量”
    # 连续极限里：C(t) = sum_j |a_j|^2 e^{-i E_j t}
    # 离散 FFT 给出的是频率 ω，对应能量 E_j ≈ ω_j
    dt = ts[1] - ts[0]
    freqs = np.fft.fftfreq(num_time_points, d=dt) # 频率轴
    spectrum = np.fft.fft(C_ts) # 频谱

    # 只看正频部分
    mask = freqs >= 0
    freqs_pos = freqs[mask]
    spectrum_pos = spectrum[mask]

    # 谱强度
    intensity = np.abs(spectrum_pos) ** 2

    return { "ts": ts, 
    "C_ts": C_ts, 
    "freqs": freqs_pos, 
    "spectrum": spectrum_pos, 
    "intensity": intensity, }


In [ ]:
prep_circ = QuantumCircuit(H.num_qubits)
result = qeve(H, prep_circ, t_max=6.0, num_time_points=64)
freqs = result["freqs"]
intensity = result["intensity"]

# 找峰值
idx_max = np.argmax(intensity)
E_est = freqs[idx_max]

print("Estimated energy from QEVE:", E_est)

In [ ]:
"""
    参数化 QEVE：
    - H: SparsePauliOp 或 PauliSumOp 风格的哈密顿量（Qiskit 1.x 推荐 SparsePauliOp）
    - prep_circ: 制备 |psi(0)> 的电路（作用在 num_qubits 上）
    - t_max: 最大演化时间
    - num_time_points: 采样的时间点个数
    返回：
    - ts: 时间点数组，shape = (N,)
    - Cts: 时间相关函数 C(t) 数组，shape = (N,)
"""
def parameterized_qeve(H, prep_circ: QuantumCircuit,
t_max: float = 6.0,
num_time_points: int = 64,):
    num_qubits = H.num_qubits

    # -------------------------
    # 1. 后端：statevector 模式
    # -------------------------
    backend = Aer.get_backend("aer_simulator")
    backend.set_options(method="statevector")

    # -------------------------
    # 2. 先算 |psi(0)> 的 statevector
    # -------------------------
    qc_psi0 = QuantumCircuit(num_qubits)
    qc_psi0.compose(prep_circ, inplace=True)
    qc_psi0.save_statevector()

    tqc_psi0 = transpile(qc_psi0, backend, optimization_level=3)
    result_psi0 = backend.run(tqc_psi0).result()
    psi0 = result_psi0.get_statevector(tqc_psi0)

    # -------------------------
    # 3. 构造“带参数的演化电路模板”
    #    U(tau) = exp(-i H * tau)
    # -------------------------
    tau = Parameter("tau")

    qc_template = QuantumCircuit(num_qubits)
    qc_template.compose(prep_circ, inplace=True)
    evo_gate = PauliEvolutionGate(H, time=tau)
    qc_template.append(evo_gate, range(num_qubits))
    qc_template.save_statevector()

    # 只 transpile 一次（非常关键）
    tqc_template = transpile(qc_template, backend, optimization_level=3)

    # -------------------------
    # 4. 生成时间点
    # -------------------------
    ts = np.linspace(0.0, t_max, num_time_points, endpoint=False)
    Cts = np.zeros(num_time_points, dtype=complex)

    # -------------------------
    # 5. 对每个 t：
    #    - 绑定参数 tau -> t
    #    - 仿真得到 |psi(t)>
    #    - 立刻算 C(t) = <psi(0)|psi(t)>
    #    - 只存 C(t)，不存 |psi(t)|，节省内存
    # -------------------------
    for idx, t in enumerate(ts):
        # 绑定参数
        tqc_t = tqc_template.assign_parameters({tau: t})

        result_t = backend.run(tqc_t).result()
        psi_t = result_t.get_statevector(tqc_t)

        Cts[idx] = np.vdot(psi0, psi_t)

    return ts, Cts

# -------------------------
# 6. 一个简单的“后处理”示例：做 FFT 得到频谱
# -------------------------
def qeve_spectrum(ts, Cts): 
    """ 
    给定时间点 ts 和时间相关函数 C(t)， 
    返回频率/能量轴 w 和谱密度 rho(w)。 
    """
    dt = ts[1] - ts[0]
    N = len(ts)

    # FFT
    fft_vals = np.fft.fft(Cts)
    freqs = np.fft.fftfreq(N, d=dt)

    # 只取正频部分（根据需要也可以做 shift）
    mask = freqs >= 0
    w = 2 * np.pi * freqs[mask] # 频率 -> 能量刻度（视你 H 的单位而定）
    rho = np.abs(fft_vals[mask])

    return w, rho

In [ ]:
prep_circ = QuantumCircuit(H.num_qubits)
ts, Cts = parameterized_qeve( 
    H, 
    prep_circ, 
    t_max=6.0, # 比 10 小一点，电路浅很多 
    num_time_points=64 # 先用 64，基本不会崩 
    )

w, rho = qeve_spectrum(ts, Cts)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(ts, np.real(Cts), label="Re C(t)")
plt.plot(ts, np.imag(Cts), label="Im C(t)", linestyle="--")
plt.xlabel("t")
plt.ylabel("C(t)")
plt.title("Time Correlation Function")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(w, rho)
plt.xlabel("Energy (ω)")
plt.ylabel("ρ(ω)")
plt.title("Spectral Density from QEVE")
plt.grid(True)
plt.show()

In [ ]:
%%script false
"""
    用 (alpha I - H) 的 power method 找近似基态（最小本征值）。
    - H: SparsePauliOp（10 qubits 完全没问题）
    - num_iters: 迭代次数（20~50 一般就够）
    - alpha: shift，若为 None，则取 H 的谱半径估计
    - init_state: 初始态，不给的话用 |0...0>
    返回:
    - E0_approx: 近似基态能量
    - psi: 近似基态的 Statevector
"""
def power_method_ground(
    H: SparsePauliOp,
    num_iters: int = 30,
    alpha: float | None = None,
    init_state: Statevector | None = None,
):

    num_qubits = H.num_qubits

    # 把 H 变成矩阵
    H_mat = H.to_matrix() # 这是一个 2^n x 2^n 的 numpy 数组

    # 估计一个 alpha（大于最大本征值），这里用谱半径的粗略上界
    if alpha is None:
        # 用矩阵的谱范数作为上界
        alpha = np.linalg.norm(H_mat, ord=2) + 1.0

    # 初始态：如果没给，就用 |0...0>
    if init_state is None:
        psi = Statevector.from_label("0" * num_qubits)
    else:
        psi = init_state

    # 把 statevector 取成 numpy 向量
    v = psi.data # shape = (2^n,)

    # 迭代：v_{k+1} = (alpha I - H) v_k / ||...||
    A = alpha * np.eye(H_mat.shape[0], dtype=complex) - H_mat

    for _ in range(num_iters):
        v = A @ v
        v = v / np.linalg.norm(v)

    # 得到近似基态
    psi_ground = Statevector(v)

    # 基态能量估计：E0 ≈ <psi|H|psi>
    E0_approx = np.vdot(v, H_mat @ v).real

    return E0_approx, psi_ground

In [ ]:
"""
    给一个 Statevector，生成制备该态的 QuantumCircuit。
    只适合在模拟器/理论分析里用，真实硬件上会很深。
"""
def statevector_to_prep_circ(psi: Statevector) -> QuantumCircuit:
    num_qubits = psi.num_qubits
    qc = QuantumCircuit(num_qubits)
    qc.initialize(psi.data, qc.qubits)
    return qc

In [ ]:
# 假设已经有 H: SparsePauliOp
E0, psi0 = power_method_ground(H, num_iters=30)

print("Approx ground energy:", E0)

prep_circ = statevector_to_prep_circ(psi0)
# 之后如果还想用 QEVE 可以：
# ts, Cts = parameterized_qeve(H, prep_circ, t_max=..., num_time_points=...)

In [ ]:
print("num_qubits:", H.num_qubits)
print("coeff max abs:", np.max(np.abs(H.coeffs)))
print("coeff min, max:", np.min(H.coeffs.real), np.max(H.coeffs.real))

In [ ]:
scale = np.max(np.abs(H.coeffs))
H_phys = H / scale
E0_scaled, psi0 = power_method_ground(H_phys, num_iters=30)
# 物理基态能量：
E0_phys = E0_scaled # * scale
print("Approx ground energy (physical scale):", E0_phys)

In [ ]:
#from qiskit import QuantumCircuit

t = np.pi/E_max
U = PauliEvolutionGate(H, time=t)   # 默认就是 Trotter 演化

# 转成量子电路
num_system_qubits = H.num_qubits
print("Qubit Number =", num_system_qubits)
#qc = QuantumCircuit.from_instruction(U)
qc = QuantumCircuit(num_system_qubits)
qc.append(U, range(num_system_qubits))
#print(qc)

In [ ]:
"""
为给定的 SparsePauliOp Hamiltonian 构造 QPE 电路（不含测量结果解析）。
"""
#from qiskit.circuit.library import QFT
def build_qpe_circuit(H: SparsePauliOp,
        num_count_qubits: int = 6):

        num_system_qubits = H.num_qubits

        # 1) 估计谱半径，做缩放
        E_max, eigvals = compute_E_max(H)
        E_min = float(np.min(eigvals))
        E_max = float(np.max(eigvals))


        # 缩放 Hamiltonian，避免相位折返
        H_shifted = H - E_min * SparsePauliOp.from_list([("I" * num_qubits, 1.0)])
        # lambda_scale = E_max if E_max != 0 else 1.0
        H_scaled = H_shifted / (E_max - E_min)
        # H_scaled = H / lambda_scale

        # 选时间参数 t（对缩放后的 H 而言）
        t = np.pi / 2 # |E_scaled| ≤ 1 时，相位 ≤ π/2

        # 2) 构造 PauliEvolutionGate
        U_gate = PauliEvolutionGate(H_scaled, time=t)

        # 3) QPE 电路：counting + system
        qc = QuantumCircuit(num_count_qubits + num_system_qubits, num_count_qubits)

        count_qubits = list(range(num_count_qubits))
        system_qubits = list(range(num_count_qubits, num_count_qubits + num_system_qubits))
        
        # 4) 计数比特做 Hadamard
        for q in count_qubits:
           qc.h(q)

        # 5) 系统比特初态（这里用 |0...0>，如果你有本征态近似，可以在这里替换） 
        # TODO: 可选：在 system_qubits 上加初始化电路

        # 6) controlled-U^{2^k} 
        # 先把 U_gate 包装成一个电路，再转成 Gate
        qc_U = QuantumCircuit(num_system_qubits)
        qc_U.append(U_gate, range(num_system_qubits))
        U_as_gate = qc_U.to_gate(label="U")

        for k, cq in enumerate(count_qubits):
            power = 2 ** k
            # U^{2^k}：通过 time → time * 2^k 实现
            U_power_gate = PauliEvolutionGate(H_scaled, time=t * power)
            qc_Uk = QuantumCircuit(num_system_qubits)
            qc_Uk.append(U_power_gate, range(num_system_qubits))
            U_power_as_gate = qc_Uk.to_gate(label=f"U^{power}")
            cU = U_power_as_gate.control(1)
            qc.append(cU, [cq] + system_qubits)

        # 7) 对计数比特做逆 QFT
        iqft = QFT(num_count_qubits, inverse=True, do_swaps=True)
        qc.append(iqft.to_gate(label="IQFT"), count_qubits)

        # 8) 测量计数比特
        qc.measure(count_qubits, range(num_count_qubits))

        return qc, t, lambda_scale, eigvals

In [ ]:
"""
构造并运行 QPE，返回估计的本征值（按出现次数最多的 bitstring）。
"""
def run_qpe(H: SparsePauliOp, num_count_qubits: int = 6, shots: int = 4096):
    qc, t, lambda_scale, eigvals = build_qpe_circuit(H, num_count_qubits)

    backend = AerSimulator()
    tqc = transpile(qc, backend)
    result = backend.run(tqc, shots=shots).result()
    counts = result.get_counts()

    # 找到出现次数最多的 bitstring
    max_bitstring = max(counts, key=counts.get)
    phase_int = int(max_bitstring, 2)
    phase = phase_int / (2 ** num_count_qubits) # φ ∈ [0,1)

    # 缩放后本征值
    E_scaled = 2 * np.pi * phase / t
    # 还原到原始 Hamiltonian 的能量尺度
    E_est = E_scaled * lambda_scale

    print("Most frequent bitstring:", max_bitstring) 
    print("Phase φ:", phase) 
    print("E_scaled:", E_scaled) 
    print("E_est (back to original scale):", E_est)

    return { 
        "counts": counts, 
        "bitstring": max_bitstring, 
        "phase": phase, 
        "E_scaled": E_scaled, 
        "E_est": E_est, 
        "eigvals_exact": eigvals, 
        }

In [ ]:
"""
    Iterative QPE for a Hermitian H given as SparsePauliOp.
    返回估计的 phase φ ∈ [0,1) 以及对应能量估计 E_est。
"""
def iqpe(H: SparsePauliOp, 
num_iterations: int = 3, 
prep_eigenstate_circ: QuantumCircuit | None = None, 
t: float | None = None, 
lambda_scale: float | None = None, 
shots: int = 256):
    num_system_qubits = H.num_qubits

    # 1) 估计谱半径，做缩放
    E_max, eigvals = compute_E_max(H)
    E_min = float(np.min(eigvals))
    E_max = float(np.max(eigvals))
    print("E_min:", E_min)
    print("E_max:", E_max)


    # 缩放 Hamiltonian，避免相位折返
    H_shifted = H - E_min * SparsePauliOp.from_list([("I" * num_system_qubits, 1.0)])
    #lambda_scale = E_max if E_max != 0 else 1.0
    H_scaled = H_shifted / (E_max - E_min)

    Hmat_scaled = H_scaled.to_matrix()
    evals_scaled, _ = np.linalg.eigh(Hmat_scaled)
    print(np.sort(evals_scaled.real))

    # 2) 选择演化时间 t
    t = 2
    # if t is None:
    #     t = np.pi / 2 # |E_scaled| ≤ 1 时，相位不会折返

    backend = Aer.get_backend("aer_simulator")

    # 3) 存储每一位的测量结果（从最高位到最低位）
    bits = []

    # 4) 迭代估计 phase 的每一位
    for k in range(num_iterations):
        qc = QuantumCircuit(1 + num_system_qubits, 1) # 1 个计数比特 + 系统比特

        # 系统比特：准备本征态（或近似本征态）
        if prep_eigenstate_circ is not None:
            qc.compose(prep_eigenstate_circ, qubits=range(1, 1 + num_system_qubits), inplace=True)

        # 计数比特初始化为 |+>
        qc.h(0)

        # 根据已经得到的高位，对系统做一个 classically‑known phase correction
        # φ ≈ 0.b0 b1 ... b_{k-1} ? ...
        if bits:
            # 已知的高位对应的 phase 角度
            known_phase = 0.0
            for i, b in enumerate(bits):
                known_phase += b / (2 ** (i + 1)) # b0/2 + b1/4 + ...
            # 这一轮要抵消的相位：-2π * known_phase * 2^{num_iterations - 1 - k}
            angle = -2 * np.pi * known_phase * (2 ** (num_iterations - 1 - k))
            # 在系统上加一个全局 Z 旋转等价的相位（这里用一个占位的多比特 Z）
            # 简化起见，我们只在第一个系统比特上加 RZ，作为近似
            qc.rz(angle, 1)

        # controlled‑U^{2^{m}}，其中 m = num_iterations - 1 - k
        power = 2 ** (num_iterations - 1 - k)

        paulis = H_scaled.paulis # PauliList
        coeffs = H_scaled.coeffs # 对应系数
        target_qubits = list(range(1, 1 + num_system_qubits))

        for P, c in zip(paulis, coeffs):
            pauli_string = P.to_label() # 比如 "XYZI..."
            target_qubits = list(range(1, 1 + num_system_qubits))
            theta = float(np.real(c)) * t * power
            apply_controlled_pauli_evolution(
                qc,
                ctrl=0, # 计数比特
                target_qubits=target_qubits,
                pauli_string=pauli_string,
                theta=theta,)

        # 对计数比特做 H
        qc.h(0)

        # 测量计数比特
        qc.measure(0, 0)

        tqc = transpile(qc, backend)
        result = backend.run(tqc, shots=shots).result()
        counts = result.get_counts()

        # 取出现次数最多的结果（'0' 或 '1'）
        bit = max(counts, key=counts.get)
        bit_val = int(bit)
        bits.append(bit_val)

        print(f"Iteration {k}: measured bit = {bit_val}, counts = {counts}")

    # 5) 把 bits 转成 phase（注意 bits[0] 是最高位）
    phase = 0.0
    for i, b in enumerate(bits):
        phase += b / (2 ** (i + 1))

    # 6) 还原能量：E_scaled = 2π * phase / t
    lambda_prime = 2 * np.pi * phase / t
    E_est = E_min + lambda_prime * (E_max - E_min)

    print("Estimated phase φ:", phase)
    #print("Estimated E_scaled:", E_scaled)
    print("Estimated E (original scale):", E_est)

    return { 
        "bits": bits, 
        "phase": phase, 
        #"E_scaled": E_scaled, 
        #"E_est": E_est, 
        #"lambda_scale": lambda_scale, 
        "t": t, 
    }


In [ ]:
"""
手写 controlled‑Pauli No.1
"""
def _basis_change_to_Z(qc: QuantumCircuit, target_qubits: list[int], pauli_string: str):
    for q, p in zip(target_qubits, pauli_string):
        if p == "X":
            qc.h(q) # X -> Z
        elif p == "Y":
            qc.sdg(q) # Y -> X
            qc.h(q) # X -> Z
        # Z / I 不动
        # 
def _basis_change_back_from_Z(qc: QuantumCircuit, target_qubits: list[int], pauli_string: str):
    # 逆序还原
    for q, p in reversed(list(zip(target_qubits, pauli_string))):
        if p == "X": 
            qc.h(q) # Z -> X
        elif p == "Y":   
            qc.h(q) # Z -> X 
            qc.s(q) # X -> Y    
            # Z / I 不动

In [ ]:
"""
手写 controlled‑Pauli No.2
非 I 的位 参与 parity 聚合
"""
def _controlled_Z_string_evolution(
    qc: QuantumCircuit,
    ctrl: int,
    target_qubits: list[int],
    pauli_string: str,
    theta: float,
):
    # 选出真正参与 Z 串的 qubit（原来是 X/Y/Z 的位置）
    active = [q for q, p in zip(target_qubits, pauli_string) if p != "I"]
    if len(active) == 0:
        return # 纯 I，什么都不用做
    if len(active) == 1:
        # 只有一个 Z：就是 controlled-RZ
        qc.crz(2 * theta, ctrl, active[0])
        return

    # 多个 Z：CNOT 梯子聚 parity 到 last
    *rest, last = active
    for q in rest:
        qc.cx(q, last)

    # 在 last 上做 controlled-RZ(2θ)
    qc.crz(2 * theta, ctrl, last)

    # 反向还原 CNOT 梯子
    for q in reversed(rest): 
        qc.cx(q, last)  

In [ ]:
"""
手写 controlled‑Pauli No.3
controlled‑exp(−i θ P)
"""
def apply_controlled_pauli_evolution(
    qc: QuantumCircuit,
    ctrl: int,
    target_qubits: list[int],
    pauli_string: str,
    theta: float,):
    # 1) 基变换到 Z 串
    _basis_change_to_Z(qc, target_qubits, pauli_string)

    # 2) 在 Z 串上做受控演化 
    _controlled_Z_string_evolution(qc, ctrl, target_qubits, pauli_string, theta)

    # 3) 基变换还原
    _basis_change_back_from_Z(qc, target_qubits, pauli_string)    

In [ ]:
def iqpe_clean(H: SparsePauliOp,
               num_iterations: int = 3,
               prep_eigenstate_circ: QuantumCircuit | None = None,
               shots: int = 256):

    num_system_qubits = H.num_qubits

    # --- 1) 求谱界 ---
    Hmat = H.to_matrix()
    eigvals = np.linalg.eigvalsh(Hmat)
    E_min = float(np.min(eigvals))
    E_max = float(np.max(eigvals))
    print("E_min:", E_min)
    print("E_max:", E_max)

    # --- 2) shift + scale ---
    # H' = H - E_min I
    I_op = SparsePauliOp.from_list([("I" * num_system_qubits, 1.0)])
    H_shifted = H - E_min * I_op

    # H_scaled = H' / (E_max - E_min)
    deltaE = E_max - E_min
    H_scaled = H_shifted / deltaE

    # 验证 H_scaled 的谱确实在 [0,1]
    evals_scaled, _ = np.linalg.eigh(H_scaled.to_matrix())
    print("Scaled eigenvalues:", np.sort(evals_scaled.real))

    # --- 3) 定义 U = exp(-i 2π H_scaled) ---
    # 这是标准 QPE 形式，不需要 t
    backend = Aer.get_backend("aer_simulator")

    bits = []

    paulis = H_scaled.paulis
    coeffs = H_scaled.coeffs
    system_qubits = list(range(1, 1 + num_system_qubits))
    # base_U = PauliEvolutionGate(H_scaled, time=2 * np.pi)
    # cU = base_U.control(1)

    for k in range(num_iterations):
        qc = QuantumCircuit(1 + num_system_qubits, 1)

        # 准备本征态
        if prep_eigenstate_circ is not None:
            qc.compose(prep_eigenstate_circ,
            qubits=range(1, 1 + num_system_qubits),
            inplace=True)
        
        # 计数比特 |+>
        qc.h(0)

        # 已知高位的相位校正
        if bits:
            known_phase = sum(b / (2 ** (i + 1)) for i, b in enumerate(bits))
            angle = -2 * np.pi * known_phase * (2 ** (num_iterations - 1 - k))
            qc.rz(angle, 1)

        # controlled-U^{2^m}
        power = 2 ** (num_iterations - 1 - k)

        for _ in range(power):
            for P, c in zip(paulis, coeffs):
                pauli_string = P.to_label()
                theta = float(np.real(c)) * (2 * np.pi)
                apply_controlled_pauli_evolution(
                    qc, ctrl=0, target_qubits=system_qubits, pauli_string=pauli_string, theta=theta, )
            
            

        # 计数比特 H
        qc.h(0)

        # 测量
        qc.measure(0, 0)

        tqc = transpile(qc, backend)
        result = backend.run(tqc, shots=shots).result()
        counts = result.get_counts()

        bit = max(counts, key=counts.get)
        bits.append(int(bit))

        print(f"Iteration {k}: bit={bit}, counts={counts}")

    # --- 4) bits → phase ---
    phase = sum(b / (2 ** (i + 1)) for i, b in enumerate(bits))
    print("Estimated phase φ:", phase)

    # --- 5) phase → energy ---
    # λ' = φ
    # E = E_min + λ' * (E_max - E_min)
    E_est = E_min + phase * deltaE
    print("Estimated energy:", E_est)

    return {
        "bits": bits,
        "phase": phase,
        "E_est": E_est,
        "E_min": E_min,
        "E_max": E_max,
    }



In [ ]:
prep = QuantumCircuit(H.num_qubits)
res = iqpe_clean(H, num_iterations=6, prep_eigenstate_circ=prep)

In [ ]:
# 准备一个本征态（或近似本征态）电路
prep = QuantumCircuit(H.num_qubits)
# 进行计算
res = iqpe(H, num_iterations=6, prep_eigenstate_circ=prep)

In [ ]:
result = run_qpe(H, num_count_qubits=6, shots=4096)
print("Exact eigenvalues:", result["eigvals_exact"])

In [ ]:
coeff_small = omega_prime_matr(vor, points, neighbors)

print("type(coeff_small):", type(coeff_small))
# 如果是 numpy array，打印 dtype 和 shape
if isinstance(coeff_small, np.ndarray):
    print("ndarray dtype:", coeff_small.dtype, "shape:", coeff_small.shape)
else:
    # 如果是 list/tuple 等，打印长度和第一个元素类型
    try:
        print("len:", len(coeff_small))
    except Exception as e:
        print("not sized container:", e)

# 打印 repr 的前几字符，帮助判断是不是字符串或其它对象
print("repr head:", repr(coeff_small)[:400])

# 如果是可迭代的二维结构，打印前几项的类型和值样例
try:
    for i, row in enumerate(coeff_small):
        if i >= 6:
            break
        print(f"row {i} type:", type(row), "repr:", repr(row)[:200])
        # 如果 row 是可迭代，打印前几个元素类型和值
        try:
            for j, val in enumerate(row):
                if j >= 6:
                    break
                print("  ", i, j, type(val), repr(val))
        except Exception:
            pass
except Exception:
    pass


In [ ]:
coeffs = np.array(H.coeffs) # Qiskit: SparsePauliOp.coeffs
paulis = H.paulis # Qiskit: SparsePauliOp.paulis (Pauli strings)
abs_coeffs = np.abs(coeffs)

# 1. 累积范数贡献（按绝对值排序）
order = np.argsort(-abs_coeffs)
sorted_abs = abs_coeffs[order]
cum_energy = np.cumsum(sorted_abs**2) / np.sum(sorted_abs**2)

# 打印前几个项的贡献
for k in [1, 5, 10, 50, 100]:
    print(f"top {k} terms capture {cum_energy[k-1]*100:.4f}% of squared-norm")

print("n_terms:", len(paulis))
print("coeff dtype:", coeffs.dtype)
print("max |coeff|:", np.max(np.abs(coeffs)))
print("min |coeff| (nonzero):", np.min(np.abs(coeffs[np.abs(coeffs)>0])))
print("median |coeff|:", np.median(np.abs(coeffs)))
print("mean |coeff|:", np.mean(np.abs(coeffs)))
print("std |coeff|:", np.std(np.abs(coeffs)))

# 可视化分布（log scale histogram
plt.figure(figsize=(5,3))
plt.hist(np.log10(np.abs(coeffs[np.abs(coeffs)>0])), bins=60)
plt.xlabel("log10(|coeff|)")
plt.ylabel("count")
plt.title("Pauli coefficient magnitude distribution")
plt.show()

In [ ]:
Hmat = H.to_matrix()
herm_err = np.max(np.abs(Hmat - Hmat.conj().T))
print("max |H - H^†|:", herm_err)
# 若 herm_err 很小（例如 <1e-10），可认为数值上是 Hermitian

In [ ]:
# 2. 按阈值剪枝并比较基态
def prune_and_compare(threshold):
    mask = abs_coeffs > threshold
    pauli_list = [paulis[i] for i in range(len(paulis)) if mask[i]]
    coeff_list = [coeffs[i] for i in range(len(coeffs)) if mask[i]]
    H_pruned = SparsePauliOp(pauli_list, coeff_list)
    Hmat = H.to_matrix()
    Hmat_pruned = H_pruned.to_matrix()
    evals_orig = np.sort(eigvalsh(Hmat))[:6]
    evals_pruned = np.sort(eigvalsh(Hmat_pruned))[:6]
    return evals_orig, evals_pruned, mask.sum()

In [ ]:
# 试几个阈值
for factor in [1e-6, 1e-5, 1e-4, 1e-3]:
    thr = factor * np.max(abs_coeffs)
    eo, ep, kept = prune_and_compare(thr)
    print(f"thr={factor:.0e}*max -> kept={kept}, orig_lowest={eo[0]:.6g}, pruned_lowest={ep[0]:.6g}, diff={abs(eo[0]-ep[0]):.3e}")

In [ ]:
# 基本统计
print("dim:", Hmat.shape)
print("trace:", np.trace(Hmat))
print("diag mean, min, max:", np.mean(np.diag(Hmat)), np.min(np.diag(Hmat)), np.max(np.diag(Hmat)))
print("matrix spectral norm (approx):", np.linalg.norm(Hmat, 2))
print("Frobenius norm:", np.linalg.norm(Hmat, 'fro'))

In [ ]:
diag = np.real(np.diag(Hmat))
idx_min = np.argmin(diag)
idx_max = np.argmax(diag)
print("min diag idx, value:", idx_min, diag[idx_min])
print("max diag idx, value:", idx_max, diag[idx_max])


In [ ]:
#找出贡献这些极端对角的主要 Pauli 项
diag_contrib = defaultdict(float)
for p, c in zip(paulis, coeffs):
    s = str(p) # Pauli 字符串
    if all(ch in 'IZ' for ch in s): # 仅对角项
        # 该 Pauli 对角上每个 computational basis 的取值为 ±1
        # 计算其对角绝对贡献的最大值作为指标
        diag_vals = []
        for b in range(Hmat.shape[0]):
            # 计算该 Pauli 在基态 b 上的本征值: product of ±1
            val = 1
            for i, ch in enumerate(reversed(s)):
                if ch == 'Z':
                    bit = (b >> i) & 1
                    val *= (1 if bit == 0 else -1)
            diag_vals.append(val)
        diag_contrib[s] = max(abs(np.array(diag_vals) * c))
# 列出贡献最大的若干对角 Pauli
top_diag = sorted(diag_contrib.items(), key=lambda x: -abs(x[1]))[:10]
print("top diagonal Pauli contributions:", top_diag)


In [ ]:
#下面代码逐项计算并验证“对角 Pauli 项在某个基态上的贡献之和”是否等于 Hmat 的对角元素
#从而确认没有重复计数或遗漏缩放因子

dim = Hmat.shape[0]

def index_to_bitstring(idx, n_qubits):
    return format(idx, '0{}b'.format(n_qubits))[::-1] # 低位对应第0号qubit

n_qubits = int(np.log2(dim))
# 计算某个基态 idx 的对角贡献来自所有对角 Pauli
def diag_contrib_for_index(idx):
    bitstr = index_to_bitstring(idx, n_qubits)
    total = 0.0 + 0.0j
    diag_terms = []
    for p, c in zip(paulis, coeffs):
        s = str(p) # Pauli 字符串，例如 'ZIIXIZ'
        if all(ch in 'IZ' for ch in s): # 仅对角项
            # 计算该 Pauli 在基态 idx 上的本征值 (+1 或 -1)
            val = 1
            for i, ch in enumerate(s[::-1]): # 反向以匹配 bitstring 索引
                if ch == 'Z':
                    bit = (idx >> i) & 1
                    val *= (1 if bit == 0 else -1)
            contrib = c * val
            total += contrib
            diag_terms.append((s, c, val, contrib))
    return total, diag_terms

# 验证索引 5 和 62
for idx in [5, 62]:
    total, diag_terms = diag_contrib_for_index(idx)
    print("idx", idx, "Hmat diag:", Hmat[idx, idx], "reconstructed diag from Z-terms:", total)
    # 列出贡献最大的若干对角项
    diag_terms_sorted = sorted(diag_terms, key=lambda x: -abs(x[1]))
    for s, c, val, contrib in diag_terms_sorted[:10]:
        print(s, c, val, contrib)
    print("sum of top-10 contributions:", sum([contrib for _,_,_,contrib in diag_terms_sorted[:10]]))
    print("----")

In [ ]:
acc = defaultdict(complex)
for p, c in zip(paulis, coeffs):
    acc[str(p)] += c
print("unique Pauli count after merge:", len(acc))


In [ ]:
from scipy.sparse.linalg import eigsh
from qiskit.quantum_info.operators.symplectic.sparse_pauli_op import _to_sparse_matrix
H_sparse = _to_sparse_matrix(H)
evals_sparse, _ = eigsh(H_sparse, k=6, which='SA')
print("lowest eigenvalues from sparse eigensolver:", np.sort(evals_sparse))
# t_bar = t_bar_matr(vor, points, neighbors)
# length = len(points)
# evals_t = np.linalg.eigvalsh(np.array(t_bar)[:length,:length])
# evals_omega = np.linalg.eigvalsh(H.to_matrix())
# print("lowest t_bar eigenvalues:", np.sort(evals_t)[:6])
# print("lowest omega eigenvalues:", np.sort(evals_omega)[:6])

In [ ]:
lapl = lapl_matr(vor, points, neighbors)
ele_nu, ele_ele = coulombic_potential(points)
print("lapl min/max:", np.min(lapl), np.max(lapl))
print("ele_nu min/max:", np.min(ele_nu), np.max(ele_nu))